In [1]:
import scipy.stats as stats











import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rc
import matplotlib.ticker
import matplotlib.patches as patches

%matplotlib ipympl
plt.style.use('default')
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams.update({'font.size': 14})
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['mathtext.default'] = 'regular'
plt.rcParams['xtick.direction']= 'in'
plt.rcParams['ytick.direction']= 'in'
plt.rcParams['xtick.labelsize']= 14.0
plt.rcParams['ytick.labelsize']= 14.0
plt.rcParams['mathtext.default'] = 'rm' # All math text will be upright Roman


plt.rcParams['font.serif'] = ['Times New Roman']

In [2]:
import h5py
import numpy as np
import pickle

def load_variables_from_hdf5(filepath, auto_assign=True):
    """Load variables from HDF5 file with proper handling of pickled objects"""
    vars_dict = {}
    
    with h5py.File(filepath, 'r') as f:
        for var_name in f.keys():
            data = f[var_name][()]
            
            # Check if this was a pickled object
            if var_name in f and 'data_type' in f[var_name].attrs:
                data_type = f[var_name].attrs['data_type']
                
                if isinstance(data_type, bytes):
                    data_type = data_type.decode('utf-8')
                
                if data_type == 'pickle':
                    # Unpickle the data
                    if isinstance(data, np.void):
                        # Extract bytes from numpy.void
                        pickled_bytes = data.tobytes()
                    else:
                        pickled_bytes = data
                    
                    try:
                        vars_dict[var_name] = pickle.loads(pickled_bytes)
                    except Exception as e:
                        print(f"Error unpickling {var_name}: {e}")
                        vars_dict[var_name] = data  # Fallback to original data
                elif data_type == 'string':
                    # Handle string data
                    if isinstance(data, bytes):
                        vars_dict[var_name] = data.decode('utf-8')
                    elif isinstance(data, np.ndarray) and data.dtype.kind == 'S':
                        vars_dict[var_name] = data.astype(str)
                    else:
                        vars_dict[var_name] = data
                else:
                    # Handle other data types (scalar, array)
                    vars_dict[var_name] = data
            else:
                # Handle legacy data without data_type attribute
                if isinstance(data, bytes):
                    vars_dict[var_name] = data.decode('utf-8')
                elif isinstance(data, np.ndarray) and data.dtype.kind == 'S':
                    vars_dict[var_name] = data.astype(str)
                else:
                    vars_dict[var_name] = data
    
    if auto_assign:
        import inspect
        frame = inspect.currentframe()
        try:
            caller_globals = frame.f_back.f_globals
            for var_name, var_value in vars_dict.items():
                caller_globals[var_name] = var_value
            
            print(f"Successfully loaded and assigned {len(vars_dict)} variables")
            print("Sample variables:")
            for i, var_name in enumerate(vars_dict.keys()):
                if i < 3:
                    var_type = type(vars_dict[var_name]).__name__
                    print(f"  - {var_name} ({var_type})")
                elif i == 3:
                    print(f"  ... and {len(vars_dict) - 3} more")
                    break
        finally:
            del frame
    
    return vars_dict

def load_and_assign_variables_hdf5(filepath):
    """Load variables from HDF5 and explicitly assign them using globals()"""
    vars_dict = {}
    
    with h5py.File(filepath, 'r') as f:
        for var_name in f.keys():
            data = f[var_name][()]
            
            # Check if this was a pickled object
            if var_name in f and 'data_type' in f[var_name].attrs:
                data_type = f[var_name].attrs['data_type']
                
                if isinstance(data_type, bytes):
                    data_type = data_type.decode('utf-8')
                
                if data_type == 'pickle':
                    # Unpickle the data
                    if isinstance(data, np.void):
                        # Extract bytes from numpy.void
                        pickled_bytes = data.tobytes()
                    else:
                        pickled_bytes = data
                    
                    try:
                        vars_dict[var_name] = pickle.loads(pickled_bytes)
                    except Exception as e:
                        print(f"Error unpickling {var_name}: {e}")
                        vars_dict[var_name] = data  # Fallback to original data
                elif data_type == 'string':
                    # Handle string data
                    if isinstance(data, bytes):
                        vars_dict[var_name] = data.decode('utf-8')
                    elif isinstance(data, np.ndarray) and data.dtype.kind == 'S':
                        vars_dict[var_name] = data.astype(str)
                    else:
                        vars_dict[var_name] = data
                else:
                    # Handle other data types (scalar, array)
                    vars_dict[var_name] = data
            else:
                # Handle legacy data without data_type attribute
                if isinstance(data, bytes):
                    vars_dict[var_name] = data.decode('utf-8')
                elif isinstance(data, np.ndarray) and data.dtype.kind == 'S':
                    vars_dict[var_name] = data.astype(str)
                else:
                    vars_dict[var_name] = data
    
    assigned_count = 0
    for var_name, var_value in vars_dict.items():
        globals()[var_name] = var_value
        assigned_count += 1
    
    print(f"Successfully loaded and assigned {assigned_count} variables")
    print("Variables are now available in the global namespace")
    print("Sample variables:")
    for i, (var_name, var_value) in enumerate(vars_dict.items()):
        if i < 3:
            var_type = type(var_value).__name__
            print(f"  - {var_name} ({var_type})")
        elif i == 3:
            print(f"  ... and {len(vars_dict) - 3} more")
            break
    
    return vars_dict

# Test function to check what's in your HDF5 file
def inspect_hdf5_file(filepath):
    """Inspect the structure and data types in an HDF5 file"""
    with h5py.File(filepath, 'r') as f:
        print(f"HDF5 file: {filepath}")
        print(f"Number of variables: {len(f.keys())}")
        print("\nVariable details:")
        
        for var_name in f.keys():
            data = f[var_name]
            print(f"\n{var_name}:")
            print(f"  Shape: {data.shape}")
            print(f"  Dtype: {data.dtype}")
            
            if 'data_type' in data.attrs:
                data_type = data.attrs['data_type']
                if isinstance(data_type, bytes):
                    data_type = data_type.decode('utf-8')
                print(f"  Stored as: {data_type}")
            else:
                print(f"  Stored as: unknown (no data_type attribute)")
            
            # Show a preview of the data
            actual_data = data[()]
            print(f"  Preview: {str(actual_data)[:100]}...")


def luminosity_evolution_correction(luminosities, luminosities_SD, redshifts, file_name, ref_z=0.25, evolution_model='power_law', plot_results=True, n_bins=7):
    """
    Apply luminosity evolution correction to AGN luminosity data.
    Aka correct the Malmquist bias.
    
    Parameters:
    -----------
    luminosities : array-like
        Original luminosity values
    luminosities_SD : array-like
        Standard deviations of original luminosities
    redshifts : array-like
        Redshift values
    ref_z : float, default=0.25
        Reference redshift for correction
    evolution_model : str, default='power_law'
        Evolution model ('power_law', 'exponential', 'linear')
    plot_results : bool, default=True
        Whether to create plots
    n_bins : int, default=10
        Number of redshift bins for binned output
    
    Returns:
    --------
    corrected_luminosities : array
        Luminosity values corrected for evolution
    corrected_luminosities_SD : array
        Standard deviations of corrected luminosities (properly propagated)
    correction_params : dict
        Parameters of the fitted evolution model
    binned_results : dict
        Dictionary containing binned corrected luminosities and their SDs
    """

    # Set MNRAS-compliant figure parameters
    plt.rcParams.update({
        'font.size': 12,
        'font.family': 'serif',
        'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'axes.linewidth': 2.5,
        'axes.grid': True,
        'grid.alpha': 0.7,
        'grid.linestyle': '--',
        'grid.linewidth': 0.8,
        'xtick.major.size': 8,
        'xtick.minor.size': 4,
        'ytick.major.size': 8,
        'ytick.minor.size': 4,
        'xtick.major.width': 2.0,
        'xtick.minor.width': 1.5,
        'ytick.major.width': 2.0,
        'ytick.minor.width': 1.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.top': True,
        'ytick.right': True,
        'legend.frameon': True,
        'legend.fancybox': True,
        'legend.edgecolor': 'black',
        'legend.facecolor': 'white',
        'legend.framealpha': 1.0
    })


    luminosities = np.asarray(luminosities)
    redshifts = np.asarray(redshifts)
    luminosities_SD = np.asarray(luminosities_SD)

    # Define evolution models
    def power_law(z, alpha):
        return (1 + z)**alpha
    
    def exponential(z, k):
        return np.exp(k * z)
    
    def linear(z, m):
        return 1 + m * z
    
    # Take natural log of luminosities for fitting
    log_luminosities = np.log(luminosities)
    
    # Choose evolution model and fit
    if evolution_model == 'power_law':
        X = np.log(1 + redshifts).reshape(-1, 1)
        model = stats.linregress(X.flatten(), log_luminosities)
        alpha = model.slope
        intercept = model.intercept
        evolution_func = power_law
        params = [alpha]
        correction_params = {'model': 'power_law', 'alpha': alpha}
    
    elif evolution_model == 'exponential':
        X = redshifts.reshape(-1, 1)
        model = stats.linregress(X.flatten(), log_luminosities)
        k = model.slope
        intercept = model.intercept
        evolution_func = exponential
        params = [k]
        correction_params = {'model': 'exponential', 'k': k}
    
    elif evolution_model == 'linear':
        def linear_fit_func(z, log_L0, m):
            return log_L0 + np.log(1 + m * z)
        
        popt, _ = curve_fit(linear_fit_func, redshifts, log_luminosities)
        log_L0, m = popt
        intercept = log_L0
        evolution_func = linear
        params = [m]
        correction_params = {'model': 'linear', 'm': m}
    
    else:
        raise ValueError("Evolution model must be 'power_law', 'exponential', or 'linear'")
    
    # Calculate correction factors
    correction_factors = evolution_func(redshifts, *params) / evolution_func(ref_z, *params)
    
    # Apply correction to normalize to reference redshift
    corrected_luminosities = luminosities / correction_factors

    # Propagate uncertainty: corrected_SD = original_SD / correction_factor
    corrected_luminosities_SD = luminosities_SD / correction_factors

    # Create redshift bins for binned output
    z_bins = np.linspace(np.min(redshifts), np.max(redshifts), n_bins + 1)
    z_bin_centers = (z_bins[:-1] + z_bins[1:]) / 2
    digitized_bins = np.digitize(redshifts, z_bins)
    
    # Organize corrected data by redshift bins
    binned_corrected_luminosities = []
    binned_corrected_luminosities_SD = []
    binned_corrected_luminosities_scatter = []  # Added for actual scatter
    bin_info = []
    
    for i in range(1, len(z_bins)):
        bin_mask = digitized_bins == i
        if np.sum(bin_mask) > 0:
            bin_corrected_lum = corrected_luminosities[bin_mask]
            bin_corrected_lum_SD = corrected_luminosities_SD[bin_mask]
            
            # Calculate the actual scatter of luminosity values in this bin
            bin_scatter = np.std(bin_corrected_lum) if len(bin_corrected_lum) > 1 else 0.0
            
            binned_corrected_luminosities.append(bin_corrected_lum)
            binned_corrected_luminosities_SD.append(bin_corrected_lum_SD)
            binned_corrected_luminosities_scatter.append(bin_scatter)
            
            bin_info.append({
                'bin_index': i-1,
                'z_center': z_bin_centers[i-1],
                'z_range': (z_bins[i-1], z_bins[i]),
                'n_objects': len(bin_corrected_lum),
                'redshifts': redshifts[bin_mask],
                'mean_luminosity': np.mean(bin_corrected_lum),
                'luminosity_scatter': bin_scatter,
                'mean_measurement_error': np.mean(bin_corrected_lum_SD)
            })
    
    # Create binned results dictionary
    binned_results = {
        'corrected_luminosities_by_bin': binned_corrected_luminosities,
        'corrected_luminosities_SD_by_bin': binned_corrected_luminosities_SD,
        'corrected_luminosities_scatter_by_bin': binned_corrected_luminosities_scatter,  # Added
        'bin_info': bin_info,
        'z_bins': z_bins,
        'z_bin_centers': z_bin_centers
    }

    if plot_results:
        # Color scheme
        purple = "#a714ff"
        pink = "#ff14f5"
        teal = "#14D8FF"
        main_blue = "#60B5FF"
        green = "#00FF9C"
        orange = "#ffbb14"
        red = "#FF5757"
        
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))
        
        z_bins_for_errors = np.linspace(min(redshifts), max(redshifts), 10)
        digitized = np.digitize(redshifts, z_bins_for_errors)
        
        z_bin_centers = []
        orig_lum_means = []
        orig_lum_stds = []
        corr_lum_means = []
        corr_lum_stds = []
        
        for i in range(1, len(z_bins_for_errors)):
            bin_mask = digitized == i
            if np.sum(bin_mask) > 1:
                z_bin_centers.append((z_bins_for_errors[i-1] + z_bins_for_errors[i]) / 2)
                orig_lum_means.append(np.mean(luminosities[bin_mask]))
                # Fixed: Use actual scatter of luminosity values, not measurement errors
                orig_lum_stds.append(np.std(luminosities[bin_mask]))
                corr_lum_means.append(np.mean(corrected_luminosities[bin_mask]))
                corr_lum_stds.append(np.std(corrected_luminosities[bin_mask]))

        axs[0].scatter(redshifts, luminosities, marker='o', alpha=0.5, color=main_blue, s=30, zorder=3)
        axs[0].errorbar(redshifts, luminosities, yerr=luminosities_SD, linestyle='', ecolor="black", capsize=5, capthick=2, zorder=0)
        axs[0].set_xlabel('Redshift (z)', fontsize=12)
        axs[0].set_ylabel('Original Luminosity', fontsize=12)
        axs[0].set_title('Original Luminosity vs Redshift', fontsize=12)
        axs[0].set_yscale('log')
        axs[0].grid(True, alpha=0.3, zorder=0)
        axs[0].spines['top'].set_linewidth(1.5)
        axs[0].spines['right'].set_linewidth(1.5)
        axs[0].spines['left'].set_linewidth(1.5)
        axs[0].spines['bottom'].set_linewidth(1.5)
        axs[0].tick_params(axis='both', which='major', width=1.5, length=5, labelsize=10)
        axs[0].tick_params(axis='both', which='minor', width=1, length=3, labelsize=8)
        
        axs[1].scatter(redshifts, corrected_luminosities, alpha=0.5, color=green, s=30, zorder=3)
        axs[1].errorbar(redshifts, corrected_luminosities, yerr=corrected_luminosities_SD, linestyle='', ecolor="black", capsize=5, capthick=2, zorder=0)
        axs[1].set_xlabel('Redshift (z)', fontsize=12)
        axs[1].set_ylabel('Corrected Luminosity', fontsize=12)
        axs[1].set_title(f'Corrected Luminosity vs Redshift (ref z={ref_z})', fontsize=12)
        axs[1].set_yscale('log')
        axs[1].grid(True, alpha=0.3, zorder=0)
        axs[1].spines['top'].set_linewidth(1.5)
        axs[1].spines['right'].set_linewidth(1.5)
        axs[1].spines['left'].set_linewidth(1.5)
        axs[1].spines['bottom'].set_linewidth(1.5)
        axs[1].tick_params(axis='both', which='major', width=1.5, length=5, labelsize=10)
        axs[1].tick_params(axis='both', which='minor', width=1, length=3, labelsize=8)
        
        axs[2].hist(np.log10(luminosities), bins=15, alpha=0.5, label='Original', color=purple)
        axs[2].hist(np.log10(corrected_luminosities), bins=15, alpha=0.5, label='Corrected', color=teal)
        axs[2].set_xlabel('Log Luminosity', fontsize=12)
        axs[2].set_ylabel('Number', fontsize=12)
        axs[2].set_title('Luminosity Distribution', fontsize=12)
        axs[2].legend(
            loc='upper right', 
            fontsize=9, 
            frameon=True, 
            fancybox=True, 
            shadow=True, 
            borderpad=0.8, 
            edgecolor='black', 
            facecolor='white', 
            handlelength=2.5,
            columnspacing=1.5,
            labelspacing=1.5
        )
        axs[2].grid(True, alpha=0.3, zorder=0)
        axs[2].spines['top'].set_linewidth(1.5)
        axs[2].spines['right'].set_linewidth(1.5)
        axs[2].spines['left'].set_linewidth(1.5)
        axs[2].spines['bottom'].set_linewidth(1.5)
        axs[2].tick_params(axis='both', which='major', width=1.5, length=5, labelsize=10)
        axs[2].tick_params(axis='both', which='minor', width=1, length=3, labelsize=8)
        
        plt.tight_layout()
        
        fig.savefig(file_name, dpi=300, 
             bbox_inches='tight', facecolor='white', edgecolor='none')
        
        plt.show()
    
    return corrected_luminosities, corrected_luminosities_SD, correction_params, binned_results



import numpy as np
from scipy.interpolate import interp1d

def calculate_agn_and_stellar_l3000_with_errors(obs_l3000, obs_l3000_err, stellar_frac, stellar_frac_err):
    """
    Calculate both AGN-only and stellar-only L3000 with error propagation across wavelengths.
    
    Parameters:
    -----------
    obs_l3000 : array-like, shape (n_objects, n_wavelengths) or (n_wavelengths,)
        Observed luminosity at each wavelength for each AGN (watts)
    obs_l3000_err : array-like, same shape as obs_l3000
        Error on observed luminosity (watts)
    stellar_frac : array-like, shape (n_objects,) or scalar
        Stellar fraction for each AGN (dimensionless, 0-1)
    stellar_frac_err : array-like, shape (n_objects,) or scalar
        Error on stellar fraction (dimensionless)
        
    Returns:
    --------
    agn_l3000 : array
        AGN-only luminosity (watts), same shape as obs_l3000
    agn_l3000_err : array
        Error on AGN-only luminosity (watts), same shape as obs_l3000
    stellar_l3000 : array
        Stellar-only luminosity (watts), same shape as obs_l3000
    stellar_l3000_err : array
        Error on stellar-only luminosity (watts), same shape as obs_l3000
    """
    # Convert inputs to numpy arrays
    obs_l3000 = np.asarray(obs_l3000)
    obs_l3000_err = np.asarray(obs_l3000_err)
    stellar_frac = np.asarray(stellar_frac)
    stellar_frac_err = np.asarray(stellar_frac_err)
    
    # Handle broadcasting for different input shapes
    if obs_l3000.ndim == 2:  # Multiple objects, multiple wavelengths
        # stellar_frac should be (n_objects,) - broadcast to (n_objects, n_wavelengths)
        if stellar_frac.ndim == 1:
            stellar_frac = stellar_frac[:, np.newaxis]
            stellar_frac_err = stellar_frac_err[:, np.newaxis]
    
    # Calculate AGN-only luminosity at each wavelength
    # L_AGN = L_total * (1 - f_stellar)
    agn_l3000 = obs_l3000 * (1 - stellar_frac)
    
    # Calculate stellar-only luminosity at each wavelength
    # L_stellar = L_total * f_stellar
    stellar_l3000 = obs_l3000 * stellar_frac
    
    # Error propagation for AGN: f = A × (1 - B)
    # σ_f² = (1-B)² × σ_A² + A² × σ_B²
    agn_l3000_err = np.sqrt(
        (1 - stellar_frac)**2 * obs_l3000_err**2 + 
        obs_l3000**2 * stellar_frac_err**2
    )
    
    # Error propagation for stellar: f = A × B
    # σ_f² = B² × σ_A² + A² × σ_B²
    stellar_l3000_err = np.sqrt(
        stellar_frac**2 * obs_l3000_err**2 + 
        obs_l3000**2 * stellar_frac_err**2
    )
    
    return agn_l3000, agn_l3000_err, stellar_l3000, stellar_l3000_err

def removing_the_stellar_continuum_enhanced(redshift, filter_wavelengths, stellar_fractions, 
                                          stellar_fraction_errors, obs_l3000, obs_l3000_err,
                                          wavelength_array=None):
    """
    Remove stellar continuum from L3000 measurement and return both AGN-only and stellar-only components.
    
    Parameters:
    -----------
    redshift : float or array-like
        Redshift of the AGN(s). Can be single float for one object.
    filter_wavelengths : array-like
        Central wavelengths of available filters (in Angstroms, observed frame)
    stellar_fractions : array-like
        Stellar fraction measurements for filters (dimensionless, 0-1). 
        For single object, can be 1D list/array. For multiple objects, 
        should be shape (n_objects, n_filters)
    stellar_fraction_errors : array-like
        Errors on stellar fraction measurements (dimensionless). 
        Same shape as stellar_fractions
    obs_l3000 : array-like
        Observed luminosity (erg/s). Can be 1D (n_wavelengths) for single object or 
        2D (n_objects, n_wavelengths) for multiple objects
    obs_l3000_err : array-like
        Error on observed luminosity (erg/s). Same shape as obs_l3000
    wavelength_array : array-like, optional
        Wavelength array corresponding to obs_l3000. If not provided,
        assumes evenly spaced wavelengths from 2950-3050 Å (observed frame)

    Returns:
    --------
    AGN_Only_L3000 : array
        Stellar-subtracted luminosity at each wavelength (erg/s). 
        Same shape as input obs_l3000
    AGN_Only_L3000_SD : array
        Error on stellar-subtracted luminosity (erg/s). 
        Same shape as input obs_l3000
    Stellar_Only_L3000 : array
        Stellar-only luminosity at each wavelength (erg/s). 
        Same shape as input obs_l3000
    Stellar_Only_L3000_SD : array
        Error on stellar-only luminosity (erg/s). 
        Same shape as input obs_l3000
    method_used : str or list
        Description of which method was used. String for single object, list for multiple
    """
    
    # Convert inputs to numpy arrays
    obs_l3000 = np.asarray(obs_l3000)
    obs_l3000_err = np.asarray(obs_l3000_err)
    stellar_fractions = np.asarray(stellar_fractions)
    stellar_fraction_errors = np.asarray(stellar_fraction_errors)
    filter_wavelengths = np.asarray(filter_wavelengths)
    
    # Handle different input scenarios
    single_redshift = isinstance(redshift, (int, float, np.integer, np.floating))
    
    # Ensure obs_l3000 is 2D (objects x wavelengths)
    if obs_l3000.ndim == 1:
        obs_l3000 = obs_l3000[np.newaxis, :]
        obs_l3000_err = obs_l3000_err[np.newaxis, :]
        truly_single_object = True
    else:
        truly_single_object = False
    
    # Get number of objects from obs_l3000 shape
    n_objects = obs_l3000.shape[0]
    
    # Handle redshift - can be single value applied to all objects
    if single_redshift:
        redshift_array = np.full(n_objects, redshift)
    else:
        redshift_array = np.atleast_1d(redshift)
        if len(redshift_array) == 1:
            redshift_array = np.repeat(redshift_array, n_objects)
    
    # Handle stellar fractions - can be single set applied to all objects
    if stellar_fractions.ndim == 1:
        # Single set of stellar fractions - replicate for all objects
        single_stellar_set = stellar_fractions.copy()
        single_stellar_err_set = stellar_fraction_errors.copy()
        stellar_fractions = np.tile(stellar_fractions, (n_objects, 1))
        stellar_fraction_errors = np.tile(stellar_fraction_errors, (n_objects, 1))
        same_stellar_fractions = True
    else:
        same_stellar_fractions = False
    
    n_wavelengths = obs_l3000.shape[1]
    
    # If all objects use the same stellar fractions, optimize processing
    if same_stellar_fractions:
        # Calculate filter selection method once or per redshift
        if single_redshift:
            # All objects have same redshift and stellar fractions
            z = redshift_array[0]
            
            # Convert rest-frame 3000 Å range to observed frame
            rest_l3000_min = 2950
            rest_l3000_max = 3050
            obs_l3000_min = rest_l3000_min * (1 + z)
            obs_l3000_max = rest_l3000_max * (1 + z)
            obs_l3000_center = 3000 * (1 + z)
            
            # Use the single set of stellar fractions
            obj_stellar_fractions = single_stellar_set
            obj_stellar_fraction_errors = single_stellar_err_set
            
            # Filter selection logic
            within_range_mask = (filter_wavelengths >= obs_l3000_min) & (filter_wavelengths <= obs_l3000_max)
            filters_within_range = np.sum(within_range_mask)
            
            if filters_within_range > 0:
                if filters_within_range == 1:
                    idx = np.where(within_range_mask)[0][0]
                    selected_stellar_frac = obj_stellar_fractions[idx]
                    selected_stellar_frac_err = obj_stellar_fraction_errors[idx]
                    method = f"single_filter_within_range_{filter_wavelengths[idx]:.0f}A"
                else:
                    within_range_fractions = obj_stellar_fractions[within_range_mask]
                    within_range_errors = obj_stellar_fraction_errors[within_range_mask]
                    weights = 1.0 / within_range_errors**2
                    selected_stellar_frac = np.sum(weights * within_range_fractions) / np.sum(weights)
                    selected_stellar_frac_err = 1.0 / np.sqrt(np.sum(weights))
                    method = f"weighted_average_{filters_within_range}_filters"
            else:
                below_range = filter_wavelengths < obs_l3000_min
                above_range = filter_wavelengths > obs_l3000_max
                
                if np.any(below_range) and np.any(above_range):
                    closest_below_idx = np.where(below_range)[0][np.argmax(filter_wavelengths[below_range])]
                    closest_above_idx = np.where(above_range)[0][np.argmin(filter_wavelengths[above_range])]
                    
                    wave_interp = [filter_wavelengths[closest_below_idx], filter_wavelengths[closest_above_idx]]
                    frac_interp = [obj_stellar_fractions[closest_below_idx], obj_stellar_fractions[closest_above_idx]]
                    err_interp = [obj_stellar_fraction_errors[closest_below_idx], obj_stellar_fraction_errors[closest_above_idx]]
                    
                    f_interp = interp1d(wave_interp, frac_interp, kind='linear')
                    selected_stellar_frac = float(f_interp(obs_l3000_center))
                    
                    w1 = (wave_interp[1] - obs_l3000_center) / (wave_interp[1] - wave_interp[0])
                    w2 = (obs_l3000_center - wave_interp[0]) / (wave_interp[1] - wave_interp[0])
                    selected_stellar_frac_err = np.sqrt((w1 * err_interp[0])**2 + (w2 * err_interp[1])**2)
                    
                    method = f"interpolation_between_{wave_interp[0]:.0f}A_and_{wave_interp[1]:.0f}A"
                else:
                    closest_idx = np.argmin(np.abs(filter_wavelengths - obs_l3000_center))
                    selected_stellar_frac = obj_stellar_fractions[closest_idx]
                    selected_stellar_frac_err = obj_stellar_fraction_errors[closest_idx]
                    
                    if np.all(above_range):
                        method = f"closest_filter_all_above_{filter_wavelengths[closest_idx]:.0f}A"
                    else:
                        method = f"closest_filter_all_below_{filter_wavelengths[closest_idx]:.0f}A"
            
            # Apply same stellar fraction to all objects
            selected_stellar_fractions = np.full(n_objects, selected_stellar_frac)
            selected_stellar_fraction_errors = np.full(n_objects, selected_stellar_frac_err)
            method_used = method
            
        else:
            # Different redshifts but same stellar fractions
            selected_stellar_fractions = np.zeros(n_objects)
            selected_stellar_fraction_errors = np.zeros(n_objects)
            method_used = []
            
            for i in range(n_objects):
                z = redshift_array[i]
                
                obs_l3000_min = 2950 * (1 + z)
                obs_l3000_max = 3050 * (1 + z)
                obs_l3000_center = 3000 * (1 + z)
                
                # Use the single set of stellar fractions for all objects
                obj_stellar_fractions = single_stellar_set
                obj_stellar_fraction_errors = single_stellar_err_set
                
                # Apply filter selection logic for this redshift
                within_range_mask = (filter_wavelengths >= obs_l3000_min) & (filter_wavelengths <= obs_l3000_max)
                filters_within_range = np.sum(within_range_mask)
                
                if filters_within_range > 0:
                    if filters_within_range == 1:
                        idx = np.where(within_range_mask)[0][0]
                        stellar_frac = obj_stellar_fractions[idx]
                        stellar_frac_err = obj_stellar_fraction_errors[idx]
                        method = f"single_filter_within_range_{filter_wavelengths[idx]:.0f}A"
                    else:
                        within_range_fractions = obj_stellar_fractions[within_range_mask]
                        within_range_errors = obj_stellar_fraction_errors[within_range_mask]
                        weights = 1.0 / within_range_errors**2
                        stellar_frac = np.sum(weights * within_range_fractions) / np.sum(weights)
                        stellar_frac_err = 1.0 / np.sqrt(np.sum(weights))
                        method = f"weighted_average_{filters_within_range}_filters"
                else:
                    below_range = filter_wavelengths < obs_l3000_min
                    above_range = filter_wavelengths > obs_l3000_max
                    
                    if np.any(below_range) and np.any(above_range):
                        closest_below_idx = np.where(below_range)[0][np.argmax(filter_wavelengths[below_range])]
                        closest_above_idx = np.where(above_range)[0][np.argmin(filter_wavelengths[above_range])]
                        
                        wave_interp = [filter_wavelengths[closest_below_idx], filter_wavelengths[closest_above_idx]]
                        frac_interp = [obj_stellar_fractions[closest_below_idx], obj_stellar_fractions[closest_above_idx]]
                        err_interp = [obj_stellar_fraction_errors[closest_below_idx], obj_stellar_fraction_errors[closest_above_idx]]
                        
                        f_interp = interp1d(wave_interp, frac_interp, kind='linear')
                        stellar_frac = float(f_interp(obs_l3000_center))
                        
                        w1 = (wave_interp[1] - obs_l3000_center) / (wave_interp[1] - wave_interp[0])
                        w2 = (obs_l3000_center - wave_interp[0]) / (wave_interp[1] - wave_interp[0])
                        stellar_frac_err = np.sqrt((w1 * err_interp[0])**2 + (w2 * err_interp[1])**2)
                        
                        method = f"interpolation_between_{wave_interp[0]:.0f}A_and_{wave_interp[1]:.0f}A"
                    else:
                        closest_idx = np.argmin(np.abs(filter_wavelengths - obs_l3000_center))
                        stellar_frac = obj_stellar_fractions[closest_idx]
                        stellar_frac_err = obj_stellar_fraction_errors[closest_idx]
                        
                        if np.all(above_range):
                            method = f"closest_filter_all_above_{filter_wavelengths[closest_idx]:.0f}A"
                        else:
                            method = f"closest_filter_all_below_{filter_wavelengths[closest_idx]:.0f}A"
                
                selected_stellar_fractions[i] = stellar_frac
                selected_stellar_fraction_errors[i] = stellar_frac_err
                method_used.append(method)
    
    else:
        # Different stellar fractions per object (original case)
        selected_stellar_fractions = np.zeros(n_objects)
        selected_stellar_fraction_errors = np.zeros(n_objects)
        method_used = []
        
        # Process each object individually with its own stellar fractions
        for i in range(n_objects):
            z = redshift_array[i]
            
            rest_l3000_min = 2950
            rest_l3000_max = 3050
            obs_l3000_min = rest_l3000_min * (1 + z)
            obs_l3000_max = rest_l3000_max * (1 + z)
            obs_l3000_center = 3000 * (1 + z)
            
            # This is the case where each object has different stellar fractions
            obj_stellar_fractions = stellar_fractions[i]
            obj_stellar_fraction_errors = stellar_fraction_errors[i]
            
            # Same filter selection logic
            within_range_mask = (filter_wavelengths >= obs_l3000_min) & (filter_wavelengths <= obs_l3000_max)
            filters_within_range = np.sum(within_range_mask)
            
            if filters_within_range > 0:
                if filters_within_range == 1:
                    idx = np.where(within_range_mask)[0][0]
                    stellar_frac = obj_stellar_fractions[idx]
                    stellar_frac_err = obj_stellar_fraction_errors[idx]
                    method = f"single_filter_within_range_{filter_wavelengths[idx]:.0f}A"
                else:
                    within_range_fractions = obj_stellar_fractions[within_range_mask]
                    within_range_errors = obj_stellar_fraction_errors[within_range_mask]
                    weights = 1.0 / within_range_errors**2
                    stellar_frac = np.sum(weights * within_range_fractions) / np.sum(weights)
                    stellar_frac_err = 1.0 / np.sqrt(np.sum(weights))
                    method = f"weighted_average_{filters_within_range}_filters"
            else:
                below_range = filter_wavelengths < obs_l3000_min
                above_range = filter_wavelengths > obs_l3000_max
                
                if np.any(below_range) and np.any(above_range):
                    closest_below_idx = np.where(below_range)[0][np.argmax(filter_wavelengths[below_range])]
                    closest_above_idx = np.where(above_range)[0][np.argmin(filter_wavelengths[above_range])]
                    
                    wave_interp = [filter_wavelengths[closest_below_idx], filter_wavelengths[closest_above_idx]]
                    frac_interp = [obj_stellar_fractions[closest_below_idx], obj_stellar_fractions[closest_above_idx]]
                    err_interp = [obj_stellar_fraction_errors[closest_below_idx], obj_stellar_fraction_errors[closest_above_idx]]
                    
                    f_interp = interp1d(wave_interp, frac_interp, kind='linear')
                    stellar_frac = float(f_interp(obs_l3000_center))
                    
                    w1 = (wave_interp[1] - obs_l3000_center) / (wave_interp[1] - wave_interp[0])
                    w2 = (obs_l3000_center - wave_interp[0]) / (wave_interp[1] - wave_interp[0])
                    stellar_frac_err = np.sqrt((w1 * err_interp[0])**2 + (w2 * err_interp[1])**2)
                    
                    method = f"interpolation_between_{wave_interp[0]:.0f}A_and_{wave_interp[1]:.0f}A"
                else:
                    closest_idx = np.argmin(np.abs(filter_wavelengths - obs_l3000_center))
                    stellar_frac = obj_stellar_fractions[closest_idx]
                    stellar_frac_err = obj_stellar_fraction_errors[closest_idx]
                    
                    if np.all(above_range):
                        method = f"closest_filter_all_above_{filter_wavelengths[closest_idx]:.0f}A"
                    else:
                        method = f"closest_filter_all_below_{filter_wavelengths[closest_idx]:.0f}A"
            
            selected_stellar_fractions[i] = stellar_frac
            selected_stellar_fraction_errors[i] = stellar_frac_err
            method_used.append(method)
    
    # Now calculate both AGN-only and stellar-only luminosity at each wavelength for each object
    AGN_Only_L3000, AGN_Only_L3000_SD, Stellar_Only_L3000, Stellar_Only_L3000_SD = calculate_agn_and_stellar_l3000_with_errors(
        obs_l3000, obs_l3000_err, selected_stellar_fractions, selected_stellar_fraction_errors
    )
    
    # If input was truly a single object (1D input), return 1D output
    if truly_single_object:
        AGN_Only_L3000 = AGN_Only_L3000[0]  # Remove extra dimension
        AGN_Only_L3000_SD = AGN_Only_L3000_SD[0]  # Remove extra dimension
        Stellar_Only_L3000 = Stellar_Only_L3000[0]  # Remove extra dimension
        Stellar_Only_L3000_SD = Stellar_Only_L3000_SD[0]  # Remove extra dimension
    
    return AGN_Only_L3000, AGN_Only_L3000_SD, Stellar_Only_L3000, Stellar_Only_L3000_SD, method_used



#Calculating the BH mass
def black_hole_mass(lam, wave_L_3000, FWHM, wave_L_3000_err=None, FWHM_err=None):
    """
    Calculate black hole mass using the MgII line width and continuum luminosity at 3000Å.
    
    This function computes black hole mass (in solar masses) based on the empirical relation:
    M_BH/M_☉ = 3.37 * (λL_3000/10^37 W)^0.47 * (FWHM_MgII/km s^-1)^2
    
    Parameters

    ----------
    lam : float
        Normalization constant, typically 1.0.
    wave_L_3000 : float or array-like
        Continuum luminosity at 3000Å in erg/s.
    FWHM : float or array-like
        Full Width at Half Maximum of the MgII line in km/s.
    wave_L_3000_err : float or array-like, optional
        Standard deviation of the continuum luminosity in erg/s.
    FWHM_err : float or array-like, optional
        Standard deviation of the FWHM in km/s.
        
    Returns
    -------
    M_bh_per_solar : float or array-like
        Black hole mass in units of solar masses.
    M_bh_per_solar_err : float or array-like or None
        Standard deviation of the black hole mass in solar masses.
        Returns None if both input errors are None.
    
    Notes
    -----
    The error propagation uses the standard formula for error propagation:
    For a function f(x,y) = A * x^a * y^b, the relative error is:
    σ_f/f = sqrt((a*σ_x/x)^2 + (b*σ_y/y)^2)
    
    References
    ----------
    Based on the calibration from McLure & Dunlop (2004) and Vestergaard & Peterson (2006).
    """
    # Convert input to numpy arrays if they aren't already
    wave_L_3000 = np.asarray(wave_L_3000)
    FWHM = np.asarray(FWHM)
    
    # Process input errors
    if wave_L_3000_err is not None:
        wave_L_3000_err = np.asarray(wave_L_3000_err)

    if FWHM_err is not None:
        FWHM_err = np.asarray(FWHM_err)
    
    # Compute black hole mass
    xx = lam * wave_L_3000  # erg/s
    xxx = xx / (10**7)  # watts (conversion from erg/s to watts)
    
    # Exponents in the formula
    power_lum = 0.47
    power_fwhm = 2.0
    
    # Calculate black hole mass
    M_bh_per_solar = 3.37 * ((xxx / 10**37)**power_lum) * (FWHM**power_fwhm)
    
    # If no errors are provided, return only the mass
    if wave_L_3000_err is None and FWHM_err is None:
        return M_bh_per_solar, None
    
    # Error propagation
    # Initialize error array with zeros
    M_bh_per_solar_err = np.zeros_like(M_bh_per_solar, dtype=float)
    
    # For elements where both errors are provided
    mask_both = np.logical_and(
        np.logical_and(wave_L_3000_err is not None, FWHM_err is not None),
        np.logical_and(wave_L_3000_err > 0, FWHM_err > 0)
    )
    
    # For elements where only luminosity error is provided
    mask_lum_only = np.logical_and(
        np.logical_and(wave_L_3000_err is not None, FWHM_err is None),
        wave_L_3000_err > 0
    )
    
    # For elements where only FWHM error is provided
    mask_fwhm_only = np.logical_and(
        np.logical_and(wave_L_3000_err is None, FWHM_err is not None),
        FWHM_err > 0
    )
    
    # Calculate relative errors for each component
    if np.any(mask_both) or np.any(mask_lum_only):
        # Relative error contribution from luminosity
        rel_err_lum = power_lum * (wave_L_3000_err / wave_L_3000)
    
    if np.any(mask_both) or np.any(mask_fwhm_only):
        # Relative error contribution from FWHM
        rel_err_fwhm = power_fwhm * (FWHM_err / FWHM)
    
    # Calculate total relative error and convert to absolute error
    if np.any(mask_both):
        rel_err_total = np.sqrt(rel_err_lum**2 + rel_err_fwhm**2)
        M_bh_per_solar_err = M_bh_per_solar * rel_err_total
    elif np.any(mask_lum_only):
        M_bh_per_solar_err = M_bh_per_solar * rel_err_lum
    elif np.any(mask_fwhm_only):
        M_bh_per_solar_err = M_bh_per_solar * rel_err_fwhm
    
    return M_bh_per_solar, M_bh_per_solar_err


def standard_deveation_delta_BH_Stellar(M_BH, M_BH_SD, M_Stellar, M_Stellar_SD, Local, Local_SD):
    """
    This code finds the standard deviation for the delta black hole stellar mass relation.

    Parameters
    ----------
    M_BH: Black hole mass [NOT Log]
    M_BH_SD: Black hole mass standard deviation

    M_Stellar: Stellar mass [NOT log]
    M_Stellar_SD: Stellar mass standard deviation average [NOT log] 

    Local: The local scaling relation
    Local_SD: The local scaling relation error

    Return
    ------
    Delta_BH_Stellar_SD: The standard deviation for the delta black hole stellar mass relation
    """


    term_one = ( ( 1 / ( M_BH * np.log(10) ) ) * M_BH_SD )**2
    term_two = ( ( -1 / (M_Stellar * np.log(10) ) ) * M_Stellar_SD )**2
    term_three = ( ( -1 / ( Local * np.log(10) ) ) * Local_SD )**2

    return np.sqrt(term_one + term_two + term_three)



def analyze_correction_results(luminosities, corrected_luminosities, redshifts, n_bins=7):
    """
    Analyze the results of luminosity evolution correction.
    
    Parameters:
    -----------
    luminosities : array-like
        Original luminosity values
    corrected_luminosities : array-like
        Corrected luminosity values
    redshifts : array-like
        Redshift values
    n_bins : int, default=7
        Number of redshift bins for analysis
    
    Returns:
    --------
    dict : Analysis results containing statistics and diagnostics
    """

    # Set MNRAS-compliant figure parameters
    plt.rcParams.update({
        'font.size': 12,
        'font.family': 'serif',
        'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'axes.linewidth': 2.5,
        'axes.grid': True,
        'grid.alpha': 0.7,
        'grid.linestyle': '--',
        'grid.linewidth': 0.8,
        'xtick.major.size': 8,
        'xtick.minor.size': 4,
        'ytick.major.size': 8,
        'ytick.minor.size': 4,
        'xtick.major.width': 2.0,
        'xtick.minor.width': 1.5,
        'ytick.major.width': 2.0,
        'ytick.minor.width': 1.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.top': True,
        'ytick.right': True,
        'legend.frameon': True,
        'legend.fancybox': True,
        'legend.edgecolor': 'black',
        'legend.facecolor': 'white',
        'legend.framealpha': 1.0
    })


    luminosities = np.asarray(luminosities)
    corrected_luminosities = np.asarray(corrected_luminosities)
    redshifts = np.asarray(redshifts)
    
    # Color scheme matching the original code
    purple = "#a714ff"
    pink = "#ff14f5"
    teal = "#14D8FF"
    main_blue = "#60B5FF"
    green = "#00FF9C"
    orange = "#ffbb14"
    red = "#FF5757"

    
    # Calculate correction factors
    correction_factors = luminosities / corrected_luminosities
    
    # Basic statistics
    original_stats = {
        'mean': np.mean(luminosities),
        'median': np.median(luminosities),
        'std': np.std(luminosities),
        'min': np.min(luminosities),
        'max': np.max(luminosities)
    }
    
    corrected_stats = {
        'mean': np.mean(corrected_luminosities),
        'median': np.median(corrected_luminosities),
        'std': np.std(corrected_luminosities),
        'min': np.min(corrected_luminosities),
        'max': np.max(corrected_luminosities)
    }
    
    # Create redshift bins for detailed analysis
    z_bins = np.linspace(np.min(redshifts), np.max(redshifts), n_bins + 1)
    z_bin_centers = (z_bins[:-1] + z_bins[1:]) / 2
    digitized = np.digitize(redshifts, z_bins)
    
    # Analyze each redshift bin
    bin_analysis = []
    for i in range(1, len(z_bins)):
        bin_mask = digitized == i
        if np.sum(bin_mask) > 0:
            bin_data = {
                'z_center': z_bin_centers[i-1],
                'z_range': (z_bins[i-1], z_bins[i]),
                'n_objects': np.sum(bin_mask),
                'original_lum_mean': np.mean(luminosities[bin_mask]),
                'original_lum_std': np.std(luminosities[bin_mask]),
                'corrected_lum_mean': np.mean(corrected_luminosities[bin_mask]),
                'corrected_lum_std': np.std(corrected_luminosities[bin_mask]),
                'mean_correction_factor': np.mean(correction_factors[bin_mask]),
                'correction_factor_std': np.std(correction_factors[bin_mask])
            }
            bin_analysis.append(bin_data)
    
    # Test for correlation with redshift (before and after correction)
    original_corr, original_p = stats.pearsonr(redshifts, np.log10(luminosities))
    corrected_corr, corrected_p = stats.pearsonr(redshifts, np.log10(corrected_luminosities))
    
    # Variance reduction analysis
    variance_reduction = (np.var(np.log10(luminosities)) - np.var(np.log10(corrected_luminosities))) / np.var(np.log10(luminosities))
    
    # Create comprehensive plots
    fig, axs = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Correction factors vs redshift
    axs[0, 0].scatter(redshifts, correction_factors, alpha=0.6, color=orange, s=40)
    axs[0, 0].set_xlabel('Redshift (z)', fontsize=12)
    axs[0, 0].set_ylabel('Correction Factor', fontsize=12)
    axs[0, 0].set_title('Correction Factors vs Redshift', fontsize=12)
    axs[0, 0].grid(True, alpha=0.3)
    axs[0, 0].set_yscale('log')
    
    # Plot 2: Log luminosity vs redshift correlation comparison
    axs[0, 1].scatter(redshifts, np.log10(luminosities), alpha=0.5, color=main_blue, 
                     label=f'Original (r={original_corr:.3f})', s=30)
    axs[0, 1].scatter(redshifts, np.log10(corrected_luminosities), alpha=0.5, color=green, 
                     label=f'Corrected (r={corrected_corr:.3f})', s=30)
    axs[0, 1].set_xlabel('Redshift (z)', fontsize=12)
    axs[0, 1].set_ylabel('Log₁₀ Luminosity', fontsize=12)
    axs[0, 1].set_title('Luminosity-Redshift Correlation', fontsize=12)
    axs[0, 1].legend(fontsize=10)
    axs[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Variance comparison by redshift bin
    if bin_analysis:
        z_centers = [bin_data['z_center'] for bin_data in bin_analysis]
        orig_vars = [bin_data['original_lum_std']**2 for bin_data in bin_analysis]
        corr_vars = [bin_data['corrected_lum_std']**2 for bin_data in bin_analysis]
        
        x = np.arange(len(z_centers))
        width = 0.35
        
        axs[0, 2].bar(x - width/2, orig_vars, width, label='Original', color=purple, alpha=0.7)
        axs[0, 2].bar(x + width/2, corr_vars, width, label='Corrected', color=teal, alpha=0.7)
        axs[0, 2].set_xlabel('Redshift Bin Center', fontsize=12)
        axs[0, 2].set_ylabel('Variance', fontsize=12)
        axs[0, 2].set_title('Variance by Redshift Bin', fontsize=12)
        axs[0, 2].set_xticks(x)
        axs[0, 2].set_xticklabels([f'{z:.2f}' for z in z_centers], rotation=45)
        axs[0, 2].legend(fontsize=10)
        axs[0, 2].grid(True, alpha=0.3)
    
    
    # Q-Q plot to compare distributions
    orig_quantiles = np.sort(np.log10(luminosities))
    corr_quantiles = np.sort(np.log10(corrected_luminosities))
    
    # Theoretical normal quantiles
    n = len(orig_quantiles)
    theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, n))
    
    axs[1, 0].scatter(theoretical_quantiles, orig_quantiles, alpha=0.6, color=purple, 
                     label='Original', s=20)
    axs[1, 0].scatter(theoretical_quantiles, corr_quantiles, alpha=0.6, color=teal, 
                     label='Corrected', s=20)
    axs[1, 0].plot(theoretical_quantiles, theoretical_quantiles, 'k--', alpha=0.5)
    axs[1, 0].set_xlabel('Theoretical Normal Quantiles', fontsize=12)
    axs[1, 0].set_ylabel('Sample Quantiles (Log Luminosity)', fontsize=12)
    axs[1, 0].set_title('Q-Q Plot vs Normal Distribution', fontsize=12)
    axs[1, 0].legend(fontsize=10)
    axs[1, 0].grid(True, alpha=0.3)
    
    # Plot 5: Residuals analysis
    # Calculate residuals from mean trend
    z_sorted_idx = np.argsort(redshifts)
    z_sorted = redshifts[z_sorted_idx]
    orig_sorted = np.log10(luminosities[z_sorted_idx])
    corr_sorted = np.log10(corrected_luminosities[z_sorted_idx])
    
    # Simple moving average for trend
    window = max(5, len(redshifts) // 10)
    orig_trend = np.convolve(orig_sorted, np.ones(window)/window, mode='same')
    corr_trend = np.convolve(corr_sorted, np.ones(window)/window, mode='same')
    
    orig_residuals = orig_sorted - orig_trend
    corr_residuals = corr_sorted - corr_trend
    
    axs[1, 1].scatter(z_sorted, orig_residuals, alpha=0.6, color=purple, 
                     label=f'Original (σ={np.std(orig_residuals):.3f})', s=20)
    axs[1, 1].scatter(z_sorted, corr_residuals, alpha=0.6, color=teal, 
                     label=f'Corrected (σ={np.std(corr_residuals):.3f})', s=20)
    axs[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.5)
    axs[1, 1].set_xlabel('Redshift (z)', fontsize=12)
    axs[1, 1].set_ylabel('Residuals (Log Luminosity)', fontsize=12)
    axs[1, 1].set_title('Residuals from Trend', fontsize=12)
    axs[1, 1].legend(fontsize=10)
    axs[1, 1].grid(True, alpha=0.3)
    
    # Plot 6: Statistical summary
    axs[1, 2].axis('off')
    
    # Create text summary
    summary_text = f"""Statistical Summary:

Original Data:
  Mean: {original_stats['mean']:.2e}
  Std:  {original_stats['std']:.2e}
  
Corrected Data:
  Mean: {corrected_stats['mean']:.2e}
  Std:  {corrected_stats['std']:.2e}

Correlation with Redshift:
  Original:  r = {original_corr:.4f} (p = {original_p:.4f})
  Corrected: r = {corrected_corr:.4f} (p = {corrected_p:.4f})

Variance Reduction: {variance_reduction:.1%}

Correction Factor Stats:
  Mean: {np.mean(correction_factors):.3f}
  Range: {np.min(correction_factors):.3f} - {np.max(correction_factors):.3f}
  
Data Points: {len(luminosities)}
Redshift Range: {np.min(redshifts):.3f} - {np.max(redshifts):.3f}"""
    
    axs[1, 2].text(0.05, 0.95, summary_text, transform=axs[1, 2].transAxes, 
                   fontsize=11, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    # Style all plots
    for ax in axs.flat:
        if ax.get_xlabel():  # Skip the text-only subplot
            ax.spines['top'].set_linewidth(1.5)
            ax.spines['right'].set_linewidth(1.5)
            ax.spines['left'].set_linewidth(1.5)
            ax.spines['bottom'].set_linewidth(1.5)
            ax.tick_params(axis='both', which='major', width=1.5, length=5, labelsize=10)
            ax.tick_params(axis='both', which='minor', width=1, length=3, labelsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Compile results
    analysis_results = {
        'original_statistics': original_stats,
        'corrected_statistics': corrected_stats,
        'correlation_analysis': {
            'original_correlation': original_corr,
            'original_p_value': original_p,
            'corrected_correlation': corrected_corr,
            'corrected_p_value': corrected_p
        },
        'correction_factors': {
            'mean': np.mean(correction_factors),
            'std': np.std(correction_factors),
            'min': np.min(correction_factors),
            'max': np.max(correction_factors)
        },
        'variance_reduction': variance_reduction,
        'bin_analysis': bin_analysis,
        'data_summary': {
            'n_objects': len(luminosities),
            'redshift_range': (np.min(redshifts), np.max(redshifts)),
            'luminosity_range_original': (np.min(luminosities), np.max(luminosities)),
            'luminosity_range_corrected': (np.min(corrected_luminosities), np.max(corrected_luminosities))
        }
    }
    
    return analysis_results

def calculate_simple_mean_with_std(values, std_values):
    """
    Calculate simple mean and its standard deviation using error propagation.
    
    Parameters
    ----------
    values : array-like
        Array of measurement values.
    std_values : array-like
        Array of standard deviations for each measurement value.
        
    Returns
    -------
    mean_value : float
        Simple mean of the input values.
    mean_std : float
        Standard deviation of the mean.
    """
    # Convert inputs to numpy arrays if they aren't already
    values = np.array(values)
    std_values = np.array(std_values)
    
    # Calculate simple mean
    mean_value = np.mean(values)
    
    # Calculate standard deviation of the mean using error propagation
    # For a mean of measurements with individual uncertainties:
    # σ_mean = sqrt(sum(σ_i²)) / n
    n = len(values)
    mean_std = np.sqrt(np.sum(std_values**2)) / n
    
    # Alternative: standard error of the mean if your std_values are not measurement errors
    # but rather standard deviations of the sample
    # mean_std_alt = np.std(values, ddof=1) / np.sqrt(n)
    
    return mean_value, mean_std

In [4]:
Table_1_AGN_Spectra = load_variables_from_hdf5("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/All_Variables.h5", auto_assign = True)

Table_2_AGN_Luminosity_Corrected_Spectra = load_variables_from_hdf5("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/All_Variables_Luminosity_Corrected_Bagpipes.h5", auto_assign = False)
# Add 'Bagpipes_' prefix and assign to global namespace
for key, value in Table_2_AGN_Luminosity_Corrected_Spectra.items():
    globals()['Bagpipes_Cor_' + key] = value

# Update the dictionary as well
Table_2_AGN_Luminosity_Corrected_Spectra = {'Bagpipes_Cor_' + key: value for key, value in Table_2_AGN_Luminosity_Corrected_Spectra.items()}

print(f"Successfully assigned {len(Table_2_AGN_Luminosity_Corrected_Spectra)} variables with Bagpipes_Cor_ prefix")

Successfully loaded and assigned 1316 variables
Sample variables:
  - AGN_Only_Continuum_L3000_025_035 (float64)
  - AGN_Only_Continuum_L3000_035_045 (float64)
  - AGN_Only_Continuum_L3000_045_055 (float64)
  ... and 1313 more


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/All_Variables_Luminosity_Corrected_Bagpipes.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
"""
I want to make a little test about how the malmquist bias will influence the black hole and stellar masses of my sample.
"""

In [ ]:
Bagpipes_Cor_Luminosity_Corrected_Flux_Corrected_025_035

Flux_Total_03 = {
                "G_Flux": Luminosity_Corrected_Flux_Corrected_025_035[0], 
                    "G_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_025_035[0],
                "R_Flux": Luminosity_Corrected_Flux_Corrected_025_035[1], 
                    "R_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_025_035[1],
                "I_Flux": Luminosity_Corrected_Flux_Corrected_025_035[2], 
                    "I_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_025_035[2],
                "Y_Flux": Luminosity_Corrected_Flux_Corrected_025_035[3], 
                    "Y_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_025_035[3], 
                "Z_Flux": Luminosity_Corrected_Flux_Corrected_025_035[4], 
                    "Z_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_025_035[5]
                }

Flux_Total_04 = {
                "G_Flux": Luminosity_Corrected_Flux_Corrected_035_045[0], 
                    "G_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_035_045[0],
                "R_Flux": Luminosity_Corrected_Flux_Corrected_035_045[1], 
                    "R_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_035_045[1],
                "I_Flux": Luminosity_Corrected_Flux_Corrected_035_045[2], 
                    "I_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_035_045[2],
                "Y_Flux": Luminosity_Corrected_Flux_Corrected_035_045[3], 
                    "Y_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_035_045[3], 
                "Z_Flux": Luminosity_Corrected_Flux_Corrected_035_045[4], 
                    "Z_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_035_045[5]
                }

Flux_Total_05 = {
                "G_Flux": Luminosity_Corrected_Flux_Corrected_045_055[0], 
                    "G_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_045_055[0],
                "R_Flux": Luminosity_Corrected_Flux_Corrected_045_055[1], 
                    "R_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_045_055[1],
                "I_Flux": Luminosity_Corrected_Flux_Corrected_045_055[2], 
                    "I_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_045_055[2],
                "Y_Flux": Luminosity_Corrected_Flux_Corrected_045_055[3], 
                    "Y_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_045_055[3], 
                "Z_Flux": Luminosity_Corrected_Flux_Corrected_045_055[4], 
                    "Z_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_045_055[5]
                }

Flux_Total_06 = {
                "G_Flux": Luminosity_Corrected_Flux_Corrected_055_065[0], 
                    "G_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_055_065[0],
                "R_Flux": Luminosity_Corrected_Flux_Corrected_055_065[1], 
                    "R_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_055_065[1],
                "I_Flux": Luminosity_Corrected_Flux_Corrected_055_065[2], 
                    "I_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_055_065[2],
                "Y_Flux": Luminosity_Corrected_Flux_Corrected_055_065[3], 
                    "Y_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_055_065[3], 
                "Z_Flux": Luminosity_Corrected_Flux_Corrected_055_065[4], 
                    "Z_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_055_065[5]
                }

Flux_Total_07 = {
                "G_Flux": Luminosity_Corrected_Flux_Corrected_065_075[0], 
                    "G_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_065_075[0],
                "R_Flux": Luminosity_Corrected_Flux_Corrected_065_075[1], 
                    "R_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_065_075[1],
                "I_Flux": Luminosity_Corrected_Flux_Corrected_065_075[2], 
                    "I_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_065_075[2],
                "Y_Flux": Luminosity_Corrected_Flux_Corrected_065_075[3], 
                    "Y_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_065_075[3], 
                "Z_Flux": Luminosity_Corrected_Flux_Corrected_065_075[4], 
                    "Z_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_065_075[5]
                }

Flux_Total_08 = {
                "G_Flux": Luminosity_Corrected_Flux_Corrected_075_085[0], 
                    "G_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_075_085[0],
                "R_Flux": Luminosity_Corrected_Flux_Corrected_075_085[1], 
                    "R_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_075_085[1],
                "I_Flux": Luminosity_Corrected_Flux_Corrected_075_085[2], 
                    "I_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_075_085[2],
                "Y_Flux": Luminosity_Corrected_Flux_Corrected_075_085[3], 
                    "Y_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_075_085[3], 
                "Z_Flux": Luminosity_Corrected_Flux_Corrected_075_085[4], 
                    "Z_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_075_085[5]
                }

Flux_Total_0905 = {
                "G_Flux": Luminosity_Corrected_Flux_Corrected_085_096[0], 
                    "G_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_085_096[0],
                "R_Flux": Luminosity_Corrected_Flux_Corrected_085_096[1], 
                    "R_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_085_096[1],
                "I_Flux": Luminosity_Corrected_Flux_Corrected_085_096[2], 
                    "I_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_085_096[2],
                "Y_Flux": Luminosity_Corrected_Flux_Corrected_085_096[3], 
                    "Y_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_085_096[3], 
                "Z_Flux": Luminosity_Corrected_Flux_Corrected_085_096[4], 
                    "Z_Flux_SD": Luminosity_Corrected_Flux_Corrected_SD_085_096[5]
                }


# Getting the ratio of the AGN+Stellar and the Stellar Flux
Ratio_of_Total_and_Stellar_03 = {
                                "G_Ratio": Flux_Total_03["G_Flux"] / Luminosity_Corrected_Flux_Corrected_025_035[0],
                                "G_Ratio_Err": (Flux_Total_03["G_Flux"]/Luminosity_Corrected_Flux_Corrected_025_035[0]) 
                                                * (np.sqrt((( Flux_Total_03["G_Flux_SD"] / Flux_Total_03["G_Flux"] ) ** 2) + 
                                                ((Luminosity_Corrected_Flux_Corrected_SD_025_035[0] / 
                                                  Luminosity_Corrected_Flux_Corrected_025_035[0])**2))),
                                
                                "R_Ratio": Flux_Total_03["R_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][1],
                                "R_Ratio_Err": (Flux_Total_03["R_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][1]) 
                                                * (np.sqrt((( Flux_Total_03["R_Flux_SD"] / Flux_Total_03["R_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][0][1] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][1])**2))),
                                
                                "I_Ratio": Flux_Total_03["I_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][2],
                                "I_Ratio_Err": (Flux_Total_03["I_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][2]) 
                                                * (np.sqrt((( Flux_Total_03["I_Flux_SD"] / Flux_Total_03["I_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][0][2] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][2])**2))),
                                
                                "Y_Ratio": Flux_Total_03["Y_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][3],
                                "Y_Ratio_Err": (Flux_Total_03["Y_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][3]) 
                                                * (np.sqrt((( Flux_Total_03["Y_Flux_SD"] / Flux_Total_03["Y_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][0][3] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][3])**2))),
                                
                                "Z_Ratio": Flux_Total_03["Z_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][4],
                                "Z_Ratio_Err": (Flux_Total_03["Z_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][4]) 
                                                * (np.sqrt((( Flux_Total_03["Z_Flux_SD"] / Flux_Total_03["Z_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][0][4] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][0][4])**2)))
                                }

Ratio_of_Total_and_Stellar_04 = {
                                "G_Ratio": Flux_Total_04["G_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][0],
                                "G_Ratio_Err": (Flux_Total_04["G_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][0]) 
                                                * (np.sqrt((( Flux_Total_04["G_Flux_SD"] / Flux_Total_04["G_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1][0] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][0])**2))),
                                
                                "R_Ratio": Flux_Total_04["R_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][1],
                                "R_Ratio_Err": (Flux_Total_04["R_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][1]) 
                                                * (np.sqrt((( Flux_Total_04["R_Flux_SD"] / Flux_Total_04["R_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1][1] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][1])**2))),
                                
                                "I_Ratio": Flux_Total_04["I_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][2],
                                "I_Ratio_Err": (Flux_Total_04["I_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][2]) 
                                                * (np.sqrt((( Flux_Total_04["I_Flux_SD"] / Flux_Total_04["I_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1][2] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][2])**2))),
                                
                                "Y_Ratio": Flux_Total_04["Y_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][3],
                                "Y_Ratio_Err": (Flux_Total_04["Y_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][3]) 
                                                * (np.sqrt((( Flux_Total_04["Y_Flux_SD"] / Flux_Total_04["Y_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1][3] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][3])**2))),
                                
                                "Z_Ratio": Flux_Total_04["Z_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][4],
                                "Z_Ratio_Err": (Flux_Total_04["Z_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][4]) 
                                                * (np.sqrt((( Flux_Total_04["Z_Flux_SD"] / Flux_Total_04["Z_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1][4] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][1][4])**2)))
                                }
                    
Ratio_of_Total_and_Stellar_05 = {
                                "G_Ratio": Flux_Total_05["G_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][0],
                                "G_Ratio_Err": (Flux_Total_05["G_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][0]) 
                                                * (np.sqrt((( Flux_Total_05["G_Flux_SD"] / Flux_Total_05["G_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2][0] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][0])**2))),
                                
                                "R_Ratio": Flux_Total_05["R_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][1],
                                "R_Ratio_Err": (Flux_Total_05["R_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][1]) 
                                                * (np.sqrt((( Flux_Total_05["R_Flux_SD"] / Flux_Total_05["R_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2][1] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][1])**2))),
                                
                                "I_Ratio": Flux_Total_05["I_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][2],
                                "I_Ratio_Err": (Flux_Total_05["I_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][2]) 
                                                * (np.sqrt((( Flux_Total_05["I_Flux_SD"] / Flux_Total_05["I_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2][2] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][2])**2))),
                                
                                "Y_Ratio": Flux_Total_05["Y_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][3],
                                "Y_Ratio_Err": (Flux_Total_05["Y_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][3]) 
                                                * (np.sqrt((( Flux_Total_05["Y_Flux_SD"] / Flux_Total_05["Y_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2][3] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][3])**2))),
                                
                                "Z_Ratio": Flux_Total_05["Z_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][4],
                                "Z_Ratio_Err": (Flux_Total_05["Z_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][4]) 
                                                * (np.sqrt((( Flux_Total_05["Z_Flux_SD"] / Flux_Total_05["Z_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2][4] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][2][4])**2)))
                                }

Ratio_of_Total_and_Stellar_06 = {
                                "G_Ratio": Flux_Total_06["G_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][0],
                                "G_Ratio_Err": (Flux_Total_06["G_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][0]) 
                                                * (np.sqrt((( Flux_Total_06["G_Flux_SD"] / Flux_Total_06["G_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3][0] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][0])**2))),
                                
                                "R_Ratio": Flux_Total_06["R_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][1],
                                "R_Ratio_Err": (Flux_Total_06["R_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][1]) 
                                                * (np.sqrt((( Flux_Total_06["R_Flux_SD"] / Flux_Total_06["R_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3][1] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][1])**2))),
                                
                                "I_Ratio": Flux_Total_06["I_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][2],
                                "I_Ratio_Err": (Flux_Total_06["I_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][2]) 
                                                * (np.sqrt((( Flux_Total_06["I_Flux_SD"] / Flux_Total_06["I_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3][2] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][2])**2))),
                                
                                "Y_Ratio": Flux_Total_06["Y_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][3],
                                "Y_Ratio_Err": (Flux_Total_06["Y_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][3]) 
                                                * (np.sqrt((( Flux_Total_06["Y_Flux_SD"] / Flux_Total_06["Y_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3][3] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][3])**2))),
                                
                                "Z_Ratio": Flux_Total_06["Z_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][4],
                                "Z_Ratio_Err": (Flux_Total_06["Z_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][4]) 
                                                * (np.sqrt((( Flux_Total_06["Z_Flux_SD"] / Flux_Total_06["Z_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3][4] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][3][4])**2)))
                                }

Ratio_of_Total_and_Stellar_07 = {
                                "G_Ratio": Flux_Total_07["G_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][0],
                                "G_Ratio_Err": (Flux_Total_07["G_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][0]) 
                                                * (np.sqrt((( Flux_Total_07["G_Flux_SD"] / Flux_Total_07["G_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4][0] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][0])**2))),
                                
                                "R_Ratio": Flux_Total_07["R_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][1],
                                "R_Ratio_Err": (Flux_Total_07["R_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][1]) 
                                                * (np.sqrt((( Flux_Total_07["R_Flux_SD"] / Flux_Total_07["R_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4][1] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][1])**2))),
                                
                                "I_Ratio": Flux_Total_07["I_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][2],
                                "I_Ratio_Err": (Flux_Total_07["I_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][2]) 
                                                * (np.sqrt((( Flux_Total_07["I_Flux_SD"] / Flux_Total_07["I_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4][2] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][2])**2))),
                                
                                "Y_Ratio": Flux_Total_07["Y_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][3],
                                "Y_Ratio_Err": (Flux_Total_07["Y_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][3]) 
                                                * (np.sqrt((( Flux_Total_07["Y_Flux_SD"] / Flux_Total_07["Y_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4][3] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][3])**2))),
                                
                                "Z_Ratio": Flux_Total_07["Z_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][4],
                                "Z_Ratio_Err": (Flux_Total_07["Z_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][4]) 
                                                * (np.sqrt((( Flux_Total_07["Z_Flux_SD"] / Flux_Total_07["Z_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4][4] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][4][4])**2)))
                                }

Ratio_of_Total_and_Stellar_08 = {
                                "G_Ratio": Flux_Total_08["G_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][0],
                                "G_Ratio_Err": (Flux_Total_08["G_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][0]) 
                                                * (np.sqrt((( Flux_Total_08["G_Flux_SD"] / Flux_Total_08["G_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5][0] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][0])**2))),
                                
                                "R_Ratio": Flux_Total_08["R_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][1],
                                "R_Ratio_Err": (Flux_Total_08["R_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][1]) 
                                                * (np.sqrt((( Flux_Total_08["R_Flux_SD"] / Flux_Total_08["R_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5][1] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][1])**2))),
                                
                                "I_Ratio": Flux_Total_08["I_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][2],
                                "I_Ratio_Err": (Flux_Total_08["I_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][2]) 
                                                * (np.sqrt((( Flux_Total_08["I_Flux_SD"] / Flux_Total_08["I_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5][2] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][2])**2))),
                                
                                "Y_Ratio": Flux_Total_08["Y_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][3],
                                "Y_Ratio_Err": (Flux_Total_08["Y_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][3]) 
                                                * (np.sqrt((( Flux_Total_08["Y_Flux_SD"] / Flux_Total_08["Y_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5][3] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][3])**2))),
                                
                                "Z_Ratio": Flux_Total_08["Z_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][4],
                                "Z_Ratio_Err": (Flux_Total_08["Z_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][4]) 
                                                * (np.sqrt((( Flux_Total_08["Z_Flux_SD"] / Flux_Total_08["Z_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5][4] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][5][4])**2)))
                                  }

Ratio_of_Total_and_Stellar_0905 = {
                                "G_Ratio": Flux_Total_0905["G_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][0],
                                "G_Ratio_Err": (Flux_Total_0905["G_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][0]) 
                                                * (np.sqrt((( Flux_Total_0905["G_Flux_SD"] / Flux_Total_0905["G_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6][0] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][0])**2))),
                                
                                "R_Ratio": Flux_Total_0905["R_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][1],
                                "R_Ratio_Err": (Flux_Total_0905["R_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][1]) 
                                                * (np.sqrt((( Flux_Total_0905["R_Flux_SD"] / Flux_Total_0905["R_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6][1] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][1])**2))),
                                
                                "I_Ratio": Flux_Total_0905["I_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][2],
                                "I_Ratio_Err": (Flux_Total_0905["I_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][2]) 
                                                * (np.sqrt((( Flux_Total_0905["I_Flux_SD"] / Flux_Total_0905["I_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6][2] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][2])**2))),
                                
                                "Y_Ratio": Flux_Total_0905["Y_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][3],
                                "Y_Ratio_Err": (Flux_Total_0905["Y_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][3]) 
                                                * (np.sqrt((( Flux_Total_0905["Y_Flux_SD"] / Flux_Total_0905["Y_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6][3] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][3])**2))),
                                
                                "Z_Ratio": Flux_Total_0905["Z_Flux"] / Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][4],
                                "Z_Ratio_Err": (Flux_Total_0905["Z_Flux"]/Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][4]) 
                                                * (np.sqrt((( Flux_Total_0905["Z_Flux_SD"] / Flux_Total_0905["Z_Flux"] ) ** 2) + 
                                                ((Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6][4] / 
                                                  Flux_Filters_File["Flux_SersicNOne_GaussianFixedSigma"][6][4])**2)))
                                }



# Convert ratios to stellar fractions with proper error propagation
# Stellar fraction = Stellar_Flux / Total_Flux = 1 / (Total_Flux / Stellar_Flux) = 1 / Ratio
# Error propagation: if y = 1/x, then σ_y = σ_x / x²

Stellar_Fraction_03 = {
    "G_Fraction": 1.0 / Ratio_of_Total_and_Stellar_03["G_Ratio"],
    "G_Fraction_Err": Ratio_of_Total_and_Stellar_03["G_Ratio_Err"] / (Ratio_of_Total_and_Stellar_03["G_Ratio"]**2),
    
    "R_Fraction": 1.0 / Ratio_of_Total_and_Stellar_03["R_Ratio"],
    "R_Fraction_Err": Ratio_of_Total_and_Stellar_03["R_Ratio_Err"] / (Ratio_of_Total_and_Stellar_03["R_Ratio"]**2),
    
    "I_Fraction": 1.0 / Ratio_of_Total_and_Stellar_03["I_Ratio"],
    "I_Fraction_Err": Ratio_of_Total_and_Stellar_03["I_Ratio_Err"] / (Ratio_of_Total_and_Stellar_03["I_Ratio"]**2),
    
    "Y_Fraction": 1.0 / Ratio_of_Total_and_Stellar_03["Y_Ratio"],
    "Y_Fraction_Err": Ratio_of_Total_and_Stellar_03["Y_Ratio_Err"] / (Ratio_of_Total_and_Stellar_03["Y_Ratio"]**2),
    
    "Z_Fraction": 1.0 / Ratio_of_Total_and_Stellar_03["Z_Ratio"],
    "Z_Fraction_Err": Ratio_of_Total_and_Stellar_03["Z_Ratio_Err"] / (Ratio_of_Total_and_Stellar_03["Z_Ratio"]**2)
}

Stellar_Fraction_04 = {
    "G_Fraction": 1.0 / Ratio_of_Total_and_Stellar_04["G_Ratio"],
    "G_Fraction_Err": Ratio_of_Total_and_Stellar_04["G_Ratio_Err"] / (Ratio_of_Total_and_Stellar_04["G_Ratio"]**2),
    
    "R_Fraction": 1.0 / Ratio_of_Total_and_Stellar_04["R_Ratio"],
    "R_Fraction_Err": Ratio_of_Total_and_Stellar_04["R_Ratio_Err"] / (Ratio_of_Total_and_Stellar_04["R_Ratio"]**2),
    
    "I_Fraction": 1.0 / Ratio_of_Total_and_Stellar_04["I_Ratio"],
    "I_Fraction_Err": Ratio_of_Total_and_Stellar_04["I_Ratio_Err"] / (Ratio_of_Total_and_Stellar_04["I_Ratio"]**2),
    
    "Y_Fraction": 1.0 / Ratio_of_Total_and_Stellar_04["Y_Ratio"],
    "Y_Fraction_Err": Ratio_of_Total_and_Stellar_04["Y_Ratio_Err"] / (Ratio_of_Total_and_Stellar_04["Y_Ratio"]**2),
    
    "Z_Fraction": 1.0 / Ratio_of_Total_and_Stellar_04["Z_Ratio"],
    "Z_Fraction_Err": Ratio_of_Total_and_Stellar_04["Z_Ratio_Err"] / (Ratio_of_Total_and_Stellar_04["Z_Ratio"]**2)
}

Stellar_Fraction_05 = {
    "G_Fraction": 1.0 / Ratio_of_Total_and_Stellar_05["G_Ratio"],
    "G_Fraction_Err": Ratio_of_Total_and_Stellar_05["G_Ratio_Err"] / (Ratio_of_Total_and_Stellar_05["G_Ratio"]**2),
    
    "R_Fraction": 1.0 / Ratio_of_Total_and_Stellar_05["R_Ratio"],
    "R_Fraction_Err": Ratio_of_Total_and_Stellar_05["R_Ratio_Err"] / (Ratio_of_Total_and_Stellar_05["R_Ratio"]**2),
    
    "I_Fraction": 1.0 / Ratio_of_Total_and_Stellar_05["I_Ratio"],
    "I_Fraction_Err": Ratio_of_Total_and_Stellar_05["I_Ratio_Err"] / (Ratio_of_Total_and_Stellar_05["I_Ratio"]**2),
    
    "Y_Fraction": 1.0 / Ratio_of_Total_and_Stellar_05["Y_Ratio"],
    "Y_Fraction_Err": Ratio_of_Total_and_Stellar_05["Y_Ratio_Err"] / (Ratio_of_Total_and_Stellar_05["Y_Ratio"]**2),
    
    "Z_Fraction": 1.0 / Ratio_of_Total_and_Stellar_05["Z_Ratio"],
    "Z_Fraction_Err": Ratio_of_Total_and_Stellar_05["Z_Ratio_Err"] / (Ratio_of_Total_and_Stellar_05["Z_Ratio"]**2)
}

Stellar_Fraction_06 = {
    "G_Fraction": 1.0 / Ratio_of_Total_and_Stellar_06["G_Ratio"],
    "G_Fraction_Err": Ratio_of_Total_and_Stellar_06["G_Ratio_Err"] / (Ratio_of_Total_and_Stellar_06["G_Ratio"]**2),
    
    "R_Fraction": 1.0 / Ratio_of_Total_and_Stellar_06["R_Ratio"],
    "R_Fraction_Err": Ratio_of_Total_and_Stellar_06["R_Ratio_Err"] / (Ratio_of_Total_and_Stellar_06["R_Ratio"]**2),
    
    "I_Fraction": 1.0 / Ratio_of_Total_and_Stellar_06["I_Ratio"],
    "I_Fraction_Err": Ratio_of_Total_and_Stellar_06["I_Ratio_Err"] / (Ratio_of_Total_and_Stellar_06["I_Ratio"]**2),
    
    "Y_Fraction": 1.0 / Ratio_of_Total_and_Stellar_06["Y_Ratio"],
    "Y_Fraction_Err": Ratio_of_Total_and_Stellar_06["Y_Ratio_Err"] / (Ratio_of_Total_and_Stellar_06["Y_Ratio"]**2),
    
    "Z_Fraction": 1.0 / Ratio_of_Total_and_Stellar_06["Z_Ratio"],
    "Z_Fraction_Err": Ratio_of_Total_and_Stellar_06["Z_Ratio_Err"] / (Ratio_of_Total_and_Stellar_06["Z_Ratio"]**2)
}

Stellar_Fraction_07 = {
    "G_Fraction": 1.0 / Ratio_of_Total_and_Stellar_07["G_Ratio"],
    "G_Fraction_Err": Ratio_of_Total_and_Stellar_07["G_Ratio_Err"] / (Ratio_of_Total_and_Stellar_07["G_Ratio"]**2),
    
    "R_Fraction": 1.0 / Ratio_of_Total_and_Stellar_07["R_Ratio"],
    "R_Fraction_Err": Ratio_of_Total_and_Stellar_07["R_Ratio_Err"] / (Ratio_of_Total_and_Stellar_07["R_Ratio"]**2),
    
    "I_Fraction": 1.0 / Ratio_of_Total_and_Stellar_07["I_Ratio"],
    "I_Fraction_Err": Ratio_of_Total_and_Stellar_07["I_Ratio_Err"] / (Ratio_of_Total_and_Stellar_07["I_Ratio"]**2),
    
    "Y_Fraction": 1.0 / Ratio_of_Total_and_Stellar_07["Y_Ratio"],
    "Y_Fraction_Err": Ratio_of_Total_and_Stellar_07["Y_Ratio_Err"] / (Ratio_of_Total_and_Stellar_07["Y_Ratio"]**2),
    
    "Z_Fraction": 1.0 / Ratio_of_Total_and_Stellar_07["Z_Ratio"],
    "Z_Fraction_Err": Ratio_of_Total_and_Stellar_07["Z_Ratio_Err"] / (Ratio_of_Total_and_Stellar_07["Z_Ratio"]**2)
}

Stellar_Fraction_08 = {
    "G_Fraction": 1.0 / Ratio_of_Total_and_Stellar_08["G_Ratio"],
    "G_Fraction_Err": Ratio_of_Total_and_Stellar_08["G_Ratio_Err"] / (Ratio_of_Total_and_Stellar_08["G_Ratio"]**2),
    
    "R_Fraction": 1.0 / Ratio_of_Total_and_Stellar_08["R_Ratio"],
    "R_Fraction_Err": Ratio_of_Total_and_Stellar_08["R_Ratio_Err"] / (Ratio_of_Total_and_Stellar_08["R_Ratio"]**2),
    
    "I_Fraction": 1.0 / Ratio_of_Total_and_Stellar_08["I_Ratio"],
    "I_Fraction_Err": Ratio_of_Total_and_Stellar_08["I_Ratio_Err"] / (Ratio_of_Total_and_Stellar_08["I_Ratio"]**2),
    
    "Y_Fraction": 1.0 / Ratio_of_Total_and_Stellar_08["Y_Ratio"],
    "Y_Fraction_Err": Ratio_of_Total_and_Stellar_08["Y_Ratio_Err"] / (Ratio_of_Total_and_Stellar_08["Y_Ratio"]**2),
    
    "Z_Fraction": 1.0 / Ratio_of_Total_and_Stellar_08["Z_Ratio"],
    "Z_Fraction_Err": Ratio_of_Total_and_Stellar_08["Z_Ratio_Err"] / (Ratio_of_Total_and_Stellar_08["Z_Ratio"]**2)
}

Stellar_Fraction_0905 = {
    "G_Fraction": 1.0 / Ratio_of_Total_and_Stellar_0905["G_Ratio"],
    "G_Fraction_Err": Ratio_of_Total_and_Stellar_0905["G_Ratio_Err"] / (Ratio_of_Total_and_Stellar_0905["G_Ratio"]**2),
    
    "R_Fraction": 1.0 / Ratio_of_Total_and_Stellar_0905["R_Ratio"],
    "R_Fraction_Err": Ratio_of_Total_and_Stellar_0905["R_Ratio_Err"] / (Ratio_of_Total_and_Stellar_0905["R_Ratio"]**2),
    
    "I_Fraction": 1.0 / Ratio_of_Total_and_Stellar_0905["I_Ratio"],
    "I_Fraction_Err": Ratio_of_Total_and_Stellar_0905["I_Ratio_Err"] / (Ratio_of_Total_and_Stellar_0905["I_Ratio"]**2),
    
    "Y_Fraction": 1.0 / Ratio_of_Total_and_Stellar_0905["Y_Ratio"],
    "Y_Fraction_Err": Ratio_of_Total_and_Stellar_0905["Y_Ratio_Err"] / (Ratio_of_Total_and_Stellar_0905["Y_Ratio"]**2),
    
    "Z_Fraction": 1.0 / Ratio_of_Total_and_Stellar_0905["Z_Ratio"],
    "Z_Fraction_Err": Ratio_of_Total_and_Stellar_0905["Z_Ratio_Err"] / (Ratio_of_Total_and_Stellar_0905["Z_Ratio"]**2)
}

In [ ]:
# Example usage for your existing code:
AGN_Only_L3000_025_035, AGN_Only_L3000_SD_025_035, Stellar_Only_L3000_025_035, Stellar_Only_L3000_SD_025_035, Method_Used_025_035 = removing_the_stellar_continuum_enhanced(
    Fixed_Z_025_035, 
    list(Bands_Rest_Data_03_Array.values()), 
    [Stellar_Fraction_03["G_Fraction"], Stellar_Fraction_03["R_Fraction"], 
     Stellar_Fraction_03["I_Fraction"], Stellar_Fraction_03["Y_Fraction"], 
     Stellar_Fraction_03["Z_Fraction"]], 
    [Stellar_Fraction_03["G_Fraction_Err"], Stellar_Fraction_03["R_Fraction_Err"], 
     Stellar_Fraction_03["I_Fraction_Err"], Stellar_Fraction_03["Y_Fraction_Err"], 
     Stellar_Fraction_03["Z_Fraction_Err"]], 
    L_3000_025_035, L_3000_SD_025_035)


AGN_Only_L3000_035_045, AGN_Only_L3000_SD_035_045, Stellar_Only_L3000_035_045, Stellar_Only_L3000_SD_035_045, Method_Used_035_045 = removing_the_stellar_continuum_enhanced(
    Fixed_Z_035_045, 
    list(Bands_Rest_Data_04_Array.values()), 
    [Stellar_Fraction_04["G_Fraction"], Stellar_Fraction_04["R_Fraction"], 
     Stellar_Fraction_04["I_Fraction"], Stellar_Fraction_04["Y_Fraction"], 
     Stellar_Fraction_04["Z_Fraction"]], 
    [Stellar_Fraction_04["G_Fraction_Err"], Stellar_Fraction_04["R_Fraction_Err"], 
     Stellar_Fraction_04["I_Fraction_Err"], Stellar_Fraction_04["Y_Fraction_Err"], 
     Stellar_Fraction_04["Z_Fraction_Err"]], 
    L_3000_035_045, L_3000_SD_035_045)

AGN_Only_L3000_045_055, AGN_Only_L3000_SD_045_055, Stellar_Only_L3000_045_055, Stellar_Only_L3000_SD_045_055, Method_Used_045_055 = removing_the_stellar_continuum_enhanced(
    Fixed_Z_045_055, 
    list(Bands_Rest_Data_05_Array.values()), 
    [Stellar_Fraction_05["G_Fraction"], Stellar_Fraction_05["R_Fraction"], 
     Stellar_Fraction_05["I_Fraction"], Stellar_Fraction_05["Y_Fraction"], 
     Stellar_Fraction_05["Z_Fraction"]], 
    [Stellar_Fraction_05["G_Fraction_Err"], Stellar_Fraction_05["R_Fraction_Err"], 
     Stellar_Fraction_05["I_Fraction_Err"], Stellar_Fraction_05["Y_Fraction_Err"], 
     Stellar_Fraction_05["Z_Fraction_Err"]], 
    L_3000_045_055, L_3000_SD_045_055)

AGN_Only_L3000_055_065, AGN_Only_L3000_SD_055_065, Stellar_Only_L3000_055_065, Stellar_Only_L3000_SD_055_065, Method_Used_055_065 = removing_the_stellar_continuum_enhanced(
    Fixed_Z_055_065, 
    list(Bands_Rest_Data_06_Array.values()), 
    [Stellar_Fraction_06["G_Fraction"], Stellar_Fraction_06["R_Fraction"], 
     Stellar_Fraction_06["I_Fraction"], Stellar_Fraction_06["Y_Fraction"], 
     Stellar_Fraction_06["Z_Fraction"]], 
    [Stellar_Fraction_06["G_Fraction_Err"], Stellar_Fraction_06["R_Fraction_Err"], 
     Stellar_Fraction_06["I_Fraction_Err"], Stellar_Fraction_06["Y_Fraction_Err"], 
     Stellar_Fraction_06["Z_Fraction_Err"]], 
    L_3000_055_065, L_3000_SD_055_065)

AGN_Only_L3000_065_075, AGN_Only_L3000_SD_065_075, Stellar_Only_L3000_065_075, Stellar_Only_L3000_SD_065_075, Method_Used_065_075 = removing_the_stellar_continuum_enhanced(
    Fixed_Z_065_075, 
    list(Bands_Rest_Data_07_Array.values()), 
    [Stellar_Fraction_07["G_Fraction"], Stellar_Fraction_07["R_Fraction"], 
     Stellar_Fraction_07["I_Fraction"], Stellar_Fraction_07["Y_Fraction"], 
     Stellar_Fraction_07["Z_Fraction"]], 
    [Stellar_Fraction_07["G_Fraction_Err"], Stellar_Fraction_07["R_Fraction_Err"], 
     Stellar_Fraction_07["I_Fraction_Err"], Stellar_Fraction_07["Y_Fraction_Err"], 
     Stellar_Fraction_07["Z_Fraction_Err"]], 
    L_3000_065_075, L_3000_SD_065_075)

AGN_Only_L3000_075_085, AGN_Only_L3000_SD_075_085, Stellar_Only_L3000_075_085, Stellar_Only_L3000_SD_075_085, Method_Used_075_085 = removing_the_stellar_continuum_enhanced(
    Fixed_Z_075_085, 
    list(Bands_Rest_Data_08_Array.values()), 
    [Stellar_Fraction_08["G_Fraction"], Stellar_Fraction_08["R_Fraction"], 
     Stellar_Fraction_08["I_Fraction"], Stellar_Fraction_08["Y_Fraction"], 
     Stellar_Fraction_08["Z_Fraction"]], 
    [Stellar_Fraction_08["G_Fraction_Err"], Stellar_Fraction_08["R_Fraction_Err"], 
     Stellar_Fraction_08["I_Fraction_Err"], Stellar_Fraction_08["Y_Fraction_Err"], 
     Stellar_Fraction_08["Z_Fraction_Err"]], 
    L_3000_075_085, L_3000_SD_075_085)

AGN_Only_L3000_085_096, AGN_Only_L3000_SD_085_096, Stellar_Only_L3000_085_096, Stellar_Only_L3000_SD_085_096, Method_Used_085_096 = removing_the_stellar_continuum_enhanced(
    Fixed_Z_085_096, 
    list(Bands_Rest_Data_0905_Array.values()), 
    [Stellar_Fraction_0905["G_Fraction"], Stellar_Fraction_0905["R_Fraction"], 
     Stellar_Fraction_0905["I_Fraction"], Stellar_Fraction_0905["Y_Fraction"], 
     Stellar_Fraction_0905["Z_Fraction"]], 
    [Stellar_Fraction_0905["G_Fraction_Err"], Stellar_Fraction_0905["R_Fraction_Err"], 
     Stellar_Fraction_0905["I_Fraction_Err"], Stellar_Fraction_0905["Y_Fraction_Err"], 
     Stellar_Fraction_0905["Z_Fraction_Err"]], 
    L_3000_085_096, L_3000_SD_085_096)


#################################################################################################################################################################
#################################################################################################################################
#####################################################################
############################


## Example usage for your existing code:
#Luminosity_Corrected_AGN_Only_L3000_025_035, Luminosity_Corrected_AGN_Only_L3000_SD_025_035, Luminosity_Corrected_Stellar_Only_L3000_025_035, Luminosity_Corrected_Stellar_Only_L3000_SD_025_035, Method_Used_025_035 = removing_the_stellar_continuum_enhanced(
#    Fixed_Z_025_035, 
#    list(Bands_Rest_Data_03_Array.values()), 
#    [Luminosity_Corrected_Stellar_Fraction_03["G_Fraction"], Luminosity_Corrected_Stellar_Fraction_03["R_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_03["I_Fraction"], Luminosity_Corrected_Stellar_Fraction_03["Y_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_03["Z_Fraction"]], 
#    [Luminosity_Corrected_Stellar_Fraction_03["G_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_03["R_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_03["I_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_03["Y_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_03["Z_Fraction_Err"]], 
#    L_3000_025_035, L_3000_SD_025_035)
#
#
#Luminosity_Corrected_AGN_Only_L3000_035_045, Luminosity_Corrected_AGN_Only_L3000_SD_035_045, Luminosity_Corrected_Stellar_Only_L3000_035_045, Luminosity_Corrected_Stellar_Only_L3000_SD_035_045, Method_Used_035_045 = removing_the_stellar_continuum_enhanced(
#    Fixed_Z_035_045, 
#    list(Bands_Rest_Data_04_Array.values()), 
#    [Luminosity_Corrected_Stellar_Fraction_04["G_Fraction"], Luminosity_Corrected_Stellar_Fraction_04["R_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_04["I_Fraction"], Luminosity_Corrected_Stellar_Fraction_04["Y_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_04["Z_Fraction"]], 
#    [Luminosity_Corrected_Stellar_Fraction_04["G_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_04["R_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_04["I_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_04["Y_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_04["Z_Fraction_Err"]], 
#    L_3000_035_045, L_3000_SD_035_045)
#
#Luminosity_Corrected_AGN_Only_L3000_045_055, Luminosity_Corrected_AGN_Only_L3000_SD_045_055, Luminosity_Corrected_Stellar_Only_L3000_045_055, Luminosity_Corrected_Stellar_Only_L3000_SD_045_055, Method_Used_045_055 = removing_the_stellar_continuum_enhanced(
#    Fixed_Z_045_055, 
#    list(Bands_Rest_Data_05_Array.values()), 
#    [Luminosity_Corrected_Stellar_Fraction_05["G_Fraction"], Luminosity_Corrected_Stellar_Fraction_05["R_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_05["I_Fraction"], Luminosity_Corrected_Stellar_Fraction_05["Y_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_05["Z_Fraction"]], 
#    [Luminosity_Corrected_Stellar_Fraction_05["G_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_05["R_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_05["I_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_05["Y_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_05["Z_Fraction_Err"]], 
#    L_3000_045_055, L_3000_SD_045_055)
#
#Luminosity_Corrected_AGN_Only_L3000_055_065, Luminosity_Corrected_AGN_Only_L3000_SD_055_065, Luminosity_Corrected_Stellar_Only_L3000_055_065, Luminosity_Corrected_Stellar_Only_L3000_SD_055_065, Method_Used_055_065 = removing_the_stellar_continuum_enhanced(
#    Fixed_Z_055_065, 
#    list(Bands_Rest_Data_06_Array.values()), 
#    [Luminosity_Corrected_Stellar_Fraction_06["G_Fraction"], Luminosity_Corrected_Stellar_Fraction_06["R_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_06["I_Fraction"], Luminosity_Corrected_Stellar_Fraction_06["Y_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_06["Z_Fraction"]], 
#    [Luminosity_Corrected_Stellar_Fraction_06["G_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_06["R_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_06["I_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_06["Y_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_06["Z_Fraction_Err"]], 
#    L_3000_055_065, L_3000_SD_055_065)
#
#Luminosity_Corrected_AGN_Only_L3000_065_075, Luminosity_Corrected_AGN_Only_L3000_SD_065_075, Luminosity_Corrected_Stellar_Only_L3000_065_075, Luminosity_Corrected_Stellar_Only_L3000_SD_065_075, Method_Used_065_075 = removing_the_stellar_continuum_enhanced(
#    Fixed_Z_065_075, 
#    list(Bands_Rest_Data_07_Array.values()), 
#    [Luminosity_Corrected_Stellar_Fraction_07["G_Fraction"], Luminosity_Corrected_Stellar_Fraction_07["R_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_07["I_Fraction"], Luminosity_Corrected_Stellar_Fraction_07["Y_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_07["Z_Fraction"]], 
#    [Luminosity_Corrected_Stellar_Fraction_07["G_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_07["R_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_07["I_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_07["Y_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_07["Z_Fraction_Err"]], 
#    L_3000_065_075, L_3000_SD_065_075)
#
#Luminosity_Corrected_AGN_Only_L3000_075_085, Luminosity_Corrected_AGN_Only_L3000_SD_075_085, Luminosity_Corrected_Stellar_Only_L3000_075_085, Luminosity_Corrected_Stellar_Only_L3000_SD_075_085, Method_Used_075_085 = removing_the_stellar_continuum_enhanced(
#    Fixed_Z_075_085, 
#    list(Bands_Rest_Data_08_Array.values()), 
#    [Luminosity_Corrected_Stellar_Fraction_08["G_Fraction"], Luminosity_Corrected_Stellar_Fraction_08["R_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_08["I_Fraction"], Luminosity_Corrected_Stellar_Fraction_08["Y_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_08["Z_Fraction"]], 
#    [Luminosity_Corrected_Stellar_Fraction_08["G_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_08["R_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_08["I_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_08["Y_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_08["Z_Fraction_Err"]], 
#    L_3000_075_085, L_3000_SD_075_085)
#
#Luminosity_Corrected_AGN_Only_L3000_085_096, Luminosity_Corrected_AGN_Only_L3000_SD_085_096, Luminosity_Corrected_Stellar_Only_L3000_085_096, Luminosity_Corrected_Stellar_Only_L3000_SD_085_096, Method_Used_085_096 = removing_the_stellar_continuum_enhanced(
#    Fixed_Z_085_096, 
#    list(Bands_Rest_Data_0905_Array.values()), 
#    [Luminosity_Corrected_Stellar_Fraction_0905["G_Fraction"], Luminosity_Corrected_Stellar_Fraction_0905["R_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_0905["I_Fraction"], Luminosity_Corrected_Stellar_Fraction_0905["Y_Fraction"], 
#     Luminosity_Corrected_Stellar_Fraction_0905["Z_Fraction"]], 
#    [Luminosity_Corrected_Stellar_Fraction_0905["G_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_0905["R_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_0905["I_Fraction_Err"], Luminosity_Corrected_Stellar_Fraction_0905["Y_Fraction_Err"], 
#     Luminosity_Corrected_Stellar_Fraction_0905["Z_Fraction_Err"]], 
#    L_3000_085_096, L_3000_SD_085_096)
#



In [ ]:
"""
Making an array of all the Redshifts and Luminosity_Average_From_Each_AGN (getting_the_average_luminosity_for_each_AGN).
"""

Fixed_Z_Array = np.concatenate([
    np.array(Fixed_Z_025_035),
    np.array(Fixed_Z_035_045),
    np.array(Fixed_Z_045_055),
    np.array(Fixed_Z_055_065),
    np.array(Fixed_Z_065_075),
    np.array(Fixed_Z_075_085),
    np.array(Fixed_Z_085_096)
])


Luminosity_3000_AGN_Only_Array = np.concatenate([
    np.mean(AGN_Only_Continuums_L3000_025_035, axis=1),
    np.mean(AGN_Only_Continuums_L3000_035_045, axis=1),
    np.mean(AGN_Only_Continuums_L3000_045_055, axis=1),
    np.mean(AGN_Only_Continuums_L3000_055_065, axis=1),
    np.mean(AGN_Only_Continuums_L3000_065_075, axis=1),
    np.mean(AGN_Only_Continuums_L3000_075_085, axis=1),
    np.mean(AGN_Only_Continuums_L3000_085_096, axis=1)
])

Luminosity_3000_AGN_Only_SD_Array = np.concatenate([
    np.mean(AGN_Only_Continuums_L3000_SD_025_035, axis=1),
    np.mean(AGN_Only_Continuums_L3000_SD_035_045, axis=1),
    np.mean(AGN_Only_Continuums_L3000_SD_045_055, axis=1),
    np.mean(AGN_Only_Continuums_L3000_SD_055_065, axis=1),
    np.mean(AGN_Only_Continuums_L3000_SD_065_075, axis=1),
    np.mean(AGN_Only_Continuums_L3000_SD_075_085, axis=1),
    np.mean(AGN_Only_Continuums_L3000_SD_085_096, axis=1)
])



Luminosity_3000_Stellar_Only_Array = np.concatenate([
    np.mean(Stellar_Only_L3000_025_035, axis=1),
    np.mean(Stellar_Only_L3000_035_045, axis=1),
    np.mean(Stellar_Only_L3000_045_055, axis=1),
    np.mean(Stellar_Only_L3000_055_065, axis=1),
    np.mean(Stellar_Only_L3000_065_075, axis=1),
    np.mean(Stellar_Only_L3000_075_085, axis=1),
    np.mean(Stellar_Only_L3000_085_096, axis=1)
])

Luminosity_3000_Stellar_Only_SD_Array = np.concatenate([
    np.mean(Stellar_Only_L3000_SD_025_035, axis=1),
    np.mean(Stellar_Only_L3000_SD_035_045, axis=1),
    np.mean(Stellar_Only_L3000_SD_045_055, axis=1),
    np.mean(Stellar_Only_L3000_SD_055_065, axis=1),
    np.mean(Stellar_Only_L3000_SD_065_075, axis=1),
    np.mean(Stellar_Only_L3000_SD_075_085, axis=1),
    np.mean(Stellar_Only_L3000_SD_085_096, axis=1)
])



Luminosity_3000_Array = np.concatenate([
    np.mean(L_3000_025_035, axis=1),
    np.mean(L_3000_035_045, axis=1),
    np.mean(L_3000_045_055, axis=1),
    np.mean(L_3000_055_065, axis=1),
    np.mean(L_3000_065_075, axis=1),
    np.mean(L_3000_075_085, axis=1),
    np.mean(L_3000_085_096, axis=1)
])

Luminosity_3000_SD_Array = np.concatenate([
    np.mean(L_3000_SD_025_035, axis=1),
    np.mean(L_3000_SD_035_045, axis=1),
    np.mean(L_3000_SD_045_055, axis=1),
    np.mean(L_3000_SD_055_065, axis=1),
    np.mean(L_3000_SD_065_075, axis=1),
    np.mean(L_3000_SD_075_085, axis=1),
    np.mean(L_3000_SD_085_096, axis=1)
])





In [ ]:
Corrected_Luminosities_AGN_Only, Corrected_Luminosities_AGN_Only_SD, Correction_Params_AGN_Only, Corrected_Luminosities_AGN_Only_Binned_Data = luminosity_evolution_correction(Luminosity_3000_AGN_Only_Array, Luminosity_3000_AGN_Only_SD_Array, Fixed_Z_Array, file_name = "/home/jovyan/work/stampede3/AGN-Black-Hole-Research/Plots_For_Paper/Malmquist_Bias_Test_AGN_Only_MNRAS.png", ref_z=0.25, evolution_model='power_law', n_bins=7)

# Analyze results
analysis_AGN_Only = analyze_correction_results(Luminosity_3000_AGN_Only_Array, Corrected_Luminosities_AGN_Only, Fixed_Z_Array, n_bins=7)


In [ ]:
Corrected_Luminosities_Stellar_Only, Corrected_Luminosities_Stellar_Only_SD, Correction_Params_Stellar_Only, Corrected_Luminosities_Stellar_Only_Binned_Data = luminosity_evolution_correction(Luminosity_3000_Stellar_Only_Array, Luminosity_3000_Stellar_Only_SD_Array, Fixed_Z_Array, file_name = "/home/jovyan/work/stampede3/AGN-Black-Hole-Research/Plots_For_Paper/Malmquist_Bias_Test_Stellar_Only_MNRAS.png", ref_z=0.25, evolution_model='power_law', n_bins=7)

# Analyze results
analysis_Stellar_Only = analyze_correction_results(Luminosity_3000_Stellar_Only_Array, Corrected_Luminosities_Stellar_Only, Fixed_Z_Array, n_bins=7)



In [ ]:
Corrected_Luminosities_AGN_Stellar, Corrected_Luminosities_AGN_Stellar_SD, Correction_Params_AGN_Stellar, Corrected_Luminosities_AGN_Stellar_Binned_Data = luminosity_evolution_correction(Luminosity_3000_Array, Luminosity_3000_SD_Array, Fixed_Z_Array, file_name = "/home/jovyan/work/stampede3/AGN-Black-Hole-Research/Plots_For_Paper/Malmquist_Bias_Test_AGN_Stellar_Only_MNRAS.png", ref_z=0.25, evolution_model='power_law', n_bins=7)

# Analyze results
analysis_AGN_Stellar = analyze_correction_results(Luminosity_3000_Array, Corrected_Luminosities_AGN_Stellar, Fixed_Z_Array, n_bins=7)



In [ ]:
Corrected_Luminosities_AGN_Stellar_Binned_Data_025_035 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_by_bin'][0]
Corrected_Luminosities_AGN_Stellar_Binned_Data_035_045 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_by_bin'][1]
Corrected_Luminosities_AGN_Stellar_Binned_Data_045_055 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_by_bin'][2]
Corrected_Luminosities_AGN_Stellar_Binned_Data_055_065 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_by_bin'][3]
Corrected_Luminosities_AGN_Stellar_Binned_Data_065_075 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_by_bin'][4]
Corrected_Luminosities_AGN_Stellar_Binned_Data_075_085 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_by_bin'][5]
Corrected_Luminosities_AGN_Stellar_Binned_Data_085_096 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_by_bin'][6]


Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_025_035 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_SD_by_bin'][0]
Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_035_045 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_SD_by_bin'][1]
Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_045_055 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_SD_by_bin'][2]
Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_055_065 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_SD_by_bin'][3]
Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_065_075 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_SD_by_bin'][4]
Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_075_085 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_SD_by_bin'][5]
Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_085_096 = Corrected_Luminosities_AGN_Stellar_Binned_Data['corrected_luminosities_SD_by_bin'][6]

###################################################################################################################################################
############################################################################################
#########################################

Corrected_Luminosities_AGN_Only_Binned_Data_025_035 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_by_bin'][0]
Corrected_Luminosities_AGN_Only_Binned_Data_035_045 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_by_bin'][1]
Corrected_Luminosities_AGN_Only_Binned_Data_045_055 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_by_bin'][2]
Corrected_Luminosities_AGN_Only_Binned_Data_055_065 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_by_bin'][3]
Corrected_Luminosities_AGN_Only_Binned_Data_065_075 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_by_bin'][4]
Corrected_Luminosities_AGN_Only_Binned_Data_075_085 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_by_bin'][5]
Corrected_Luminosities_AGN_Only_Binned_Data_085_096 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_by_bin'][6]


Corrected_Luminosities_AGN_Only_Binned_Data_SD_025_035 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_SD_by_bin'][0]
Corrected_Luminosities_AGN_Only_Binned_Data_SD_035_045 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_SD_by_bin'][1]
Corrected_Luminosities_AGN_Only_Binned_Data_SD_045_055 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_SD_by_bin'][2]
Corrected_Luminosities_AGN_Only_Binned_Data_SD_055_065 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_SD_by_bin'][3]
Corrected_Luminosities_AGN_Only_Binned_Data_SD_065_075 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_SD_by_bin'][4]
Corrected_Luminosities_AGN_Only_Binned_Data_SD_075_085 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_SD_by_bin'][5]
Corrected_Luminosities_AGN_Only_Binned_Data_SD_085_096 = Corrected_Luminosities_AGN_Only_Binned_Data['corrected_luminosities_SD_by_bin'][6]

###################################################################################################################################################
############################################################################################
#########################################

Corrected_Luminosities_Stellar_Only_Binned_Data_025_035 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_by_bin'][0]
Corrected_Luminosities_Stellar_Only_Binned_Data_035_045 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_by_bin'][1]
Corrected_Luminosities_Stellar_Only_Binned_Data_045_055 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_by_bin'][2]
Corrected_Luminosities_Stellar_Only_Binned_Data_055_065 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_by_bin'][3]
Corrected_Luminosities_Stellar_Only_Binned_Data_065_075 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_by_bin'][4]
Corrected_Luminosities_Stellar_Only_Binned_Data_075_085 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_by_bin'][5]
Corrected_Luminosities_Stellar_Only_Binned_Data_085_096 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_by_bin'][6]


Corrected_Luminosities_Stellar_Only_Binned_Data_SD_025_035 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_SD_by_bin'][0]
Corrected_Luminosities_Stellar_Only_Binned_Data_SD_035_045 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_SD_by_bin'][1]
Corrected_Luminosities_Stellar_Only_Binned_Data_SD_045_055 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_SD_by_bin'][2]
Corrected_Luminosities_Stellar_Only_Binned_Data_SD_055_065 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_SD_by_bin'][3]
Corrected_Luminosities_Stellar_Only_Binned_Data_SD_065_075 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_SD_by_bin'][4]
Corrected_Luminosities_Stellar_Only_Binned_Data_SD_075_085 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_SD_by_bin'][5]
Corrected_Luminosities_Stellar_Only_Binned_Data_SD_085_096 = Corrected_Luminosities_Stellar_Only_Binned_Data['corrected_luminosities_SD_by_bin'][6]


In [ ]:
"""
Now with the corrected luminosities I want to recalculate the mean luminosity in each redshift bin.
"""
Luminosities_AGN_Stellar_Binned_Data_Mean_025_035, Luminosities_AGN_Stellar_Binned_Data_Mean_SD_025_035 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Stellar_Binned_Data_025_035, Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_025_035)
Luminosities_AGN_Stellar_Binned_Data_Mean_035_045, Luminosities_AGN_Stellar_Binned_Data_Mean_SD_035_045 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Stellar_Binned_Data_035_045, Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_035_045)
Luminosities_AGN_Stellar_Binned_Data_Mean_045_055, Luminosities_AGN_Stellar_Binned_Data_Mean_SD_045_055 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Stellar_Binned_Data_045_055, Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_045_055)
Luminosities_AGN_Stellar_Binned_Data_Mean_055_065, Luminosities_AGN_Stellar_Binned_Data_Mean_SD_055_065 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Stellar_Binned_Data_055_065, Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_055_065)
Luminosities_AGN_Stellar_Binned_Data_Mean_065_075, Luminosities_AGN_Stellar_Binned_Data_Mean_SD_065_075 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Stellar_Binned_Data_065_075, Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_065_075)
Luminosities_AGN_Stellar_Binned_Data_Mean_075_085, Luminosities_AGN_Stellar_Binned_Data_Mean_SD_075_085 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Stellar_Binned_Data_075_085, Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_075_085)
Luminosities_AGN_Stellar_Binned_Data_Mean_085_096, Luminosities_AGN_Stellar_Binned_Data_Mean_SD_085_096 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Stellar_Binned_Data_085_096, Corrected_Luminosities_AGN_Stellar_Binned_Data_SD_085_096)

###################################################################################################################################################
############################################################################################
#########################################

Luminosities_AGN_Only_Binned_Data_Mean_025_035, Luminosities_AGN_Only_Binned_Data_Mean_SD_025_035 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Only_Binned_Data_025_035, Corrected_Luminosities_AGN_Only_Binned_Data_SD_025_035)
Luminosities_AGN_Only_Binned_Data_Mean_035_045, Luminosities_AGN_Only_Binned_Data_Mean_SD_035_045 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Only_Binned_Data_035_045, Corrected_Luminosities_AGN_Only_Binned_Data_SD_035_045)
Luminosities_AGN_Only_Binned_Data_Mean_045_055, Luminosities_AGN_Only_Binned_Data_Mean_SD_045_055 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Only_Binned_Data_045_055, Corrected_Luminosities_AGN_Only_Binned_Data_SD_045_055)
Luminosities_AGN_Only_Binned_Data_Mean_055_065, Luminosities_AGN_Only_Binned_Data_Mean_SD_055_065 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Only_Binned_Data_055_065, Corrected_Luminosities_AGN_Only_Binned_Data_SD_055_065)
Luminosities_AGN_Only_Binned_Data_Mean_065_075, Luminosities_AGN_Only_Binned_Data_Mean_SD_065_075 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Only_Binned_Data_065_075, Corrected_Luminosities_AGN_Only_Binned_Data_SD_065_075)
Luminosities_AGN_Only_Binned_Data_Mean_075_085, Luminosities_AGN_Only_Binned_Data_Mean_SD_075_085 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Only_Binned_Data_075_085, Corrected_Luminosities_AGN_Only_Binned_Data_SD_075_085)
Luminosities_AGN_Only_Binned_Data_Mean_085_096, Luminosities_AGN_Only_Binned_Data_Mean_SD_085_096 = calculate_simple_mean_with_std(Corrected_Luminosities_AGN_Only_Binned_Data_085_096, Corrected_Luminosities_AGN_Only_Binned_Data_SD_085_096)

###################################################################################################################################################
############################################################################################
#########################################

Luminosities_Stellar_Only_Binned_Data_Mean_025_035, Luminosities_Stellar_Only_Binned_Data_Mean_SD_025_035 = calculate_simple_mean_with_std(Corrected_Luminosities_Stellar_Only_Binned_Data_025_035, Corrected_Luminosities_Stellar_Only_Binned_Data_SD_025_035)
Luminosities_Stellar_Only_Binned_Data_Mean_035_045, Luminosities_Stellar_Only_Binned_Data_Mean_SD_035_045 = calculate_simple_mean_with_std(Corrected_Luminosities_Stellar_Only_Binned_Data_035_045, Corrected_Luminosities_Stellar_Only_Binned_Data_SD_035_045)
Luminosities_Stellar_Only_Binned_Data_Mean_045_055, Luminosities_Stellar_Only_Binned_Data_Mean_SD_045_055 = calculate_simple_mean_with_std(Corrected_Luminosities_Stellar_Only_Binned_Data_045_055, Corrected_Luminosities_Stellar_Only_Binned_Data_SD_045_055)
Luminosities_Stellar_Only_Binned_Data_Mean_055_065, Luminosities_Stellar_Only_Binned_Data_Mean_SD_055_065 = calculate_simple_mean_with_std(Corrected_Luminosities_Stellar_Only_Binned_Data_055_065, Corrected_Luminosities_Stellar_Only_Binned_Data_SD_055_065)
Luminosities_Stellar_Only_Binned_Data_Mean_065_075, Luminosities_Stellar_Only_Binned_Data_Mean_SD_065_075 = calculate_simple_mean_with_std(Corrected_Luminosities_Stellar_Only_Binned_Data_065_075, Corrected_Luminosities_Stellar_Only_Binned_Data_SD_065_075)
Luminosities_Stellar_Only_Binned_Data_Mean_075_085, Luminosities_Stellar_Only_Binned_Data_Mean_SD_075_085 = calculate_simple_mean_with_std(Corrected_Luminosities_Stellar_Only_Binned_Data_075_085, Corrected_Luminosities_Stellar_Only_Binned_Data_SD_075_085)
Luminosities_Stellar_Only_Binned_Data_Mean_085_096, Luminosities_Stellar_Only_Binned_Data_Mean_SD_085_096 = calculate_simple_mean_with_std(Corrected_Luminosities_Stellar_Only_Binned_Data_085_096, Corrected_Luminosities_Stellar_Only_Binned_Data_SD_085_096)


###################################################################################################################################################
############################################################################################
#########################################

Luminosities_NOT_Corrected_Binned_Data_Mean_025_035, Luminosities_NOT_Corrected_Binned_Data_Mean_SD_025_035 = calculate_simple_mean_with_std(L_3000_025_035, L_3000_SD_025_035)
Luminosities_NOT_Corrected_Binned_Data_Mean_035_045, Luminosities_NOT_Corrected_Binned_Data_Mean_SD_035_045 = calculate_simple_mean_with_std(L_3000_035_045, L_3000_SD_035_045)
Luminosities_NOT_Corrected_Binned_Data_Mean_045_055, Luminosities_NOT_Corrected_Binned_Data_Mean_SD_045_055 = calculate_simple_mean_with_std(L_3000_045_055, L_3000_SD_045_055)
Luminosities_NOT_Corrected_Binned_Data_Mean_055_065, Luminosities_NOT_Corrected_Binned_Data_Mean_SD_055_065 = calculate_simple_mean_with_std(L_3000_055_065, L_3000_SD_055_065)
Luminosities_NOT_Corrected_Binned_Data_Mean_065_075, Luminosities_NOT_Corrected_Binned_Data_Mean_SD_065_075 = calculate_simple_mean_with_std(L_3000_065_075, L_3000_SD_065_075)
Luminosities_NOT_Corrected_Binned_Data_Mean_075_085, Luminosities_NOT_Corrected_Binned_Data_Mean_SD_075_085 = calculate_simple_mean_with_std(L_3000_075_085, L_3000_SD_075_085)
Luminosities_NOT_Corrected_Binned_Data_Mean_085_096, Luminosities_NOT_Corrected_Binned_Data_Mean_SD_085_096 = calculate_simple_mean_with_std(L_3000_085_096, L_3000_SD_085_096)



In [ ]:
"""
Recalculating the black hole masses from the Uncorrected luminosities. 
This one it for the two gaussian measurement with Uniform Narrow restrictions. (example of restrictions: PSF limited or uniform narrow.)

For the Broad part of the MgII.

Calculate black hole mass using the MgII line width and continuum luminosity at 3000Å.
    
This function computes black hole mass (in solar masses) based on the empirical relation:
    M_BH/M_☉ = 3.37 * (λL_3000/10^37 W)^0.47 * (FWHM_MgII/km s^-1)^2
"""

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_SD_025_035 = black_hole_mass(3000, Luminosities_NOT_Corrected_Binned_Data_Mean_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035, 
                                                                                                                                  Luminosities_NOT_Corrected_Binned_Data_Mean_SD_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_SD_025_035)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_035_045 = black_hole_mass(3000, Luminosities_NOT_Corrected_Binned_Data_Mean_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045, 
                                                                                                                                  Luminosities_NOT_Corrected_Binned_Data_Mean_SD_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_035_045)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_045_055 = black_hole_mass(3000, Luminosities_NOT_Corrected_Binned_Data_Mean_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055, 
                                                                                                                                  Luminosities_NOT_Corrected_Binned_Data_Mean_SD_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_045_055)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_055_065 = black_hole_mass(3000, Luminosities_NOT_Corrected_Binned_Data_Mean_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065, 
                                                                                                                                  Luminosities_NOT_Corrected_Binned_Data_Mean_SD_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_055_065)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_065_075 = black_hole_mass(3000, Luminosities_NOT_Corrected_Binned_Data_Mean_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075, 
                                                                                                                                  Luminosities_NOT_Corrected_Binned_Data_Mean_SD_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_065_075)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_075_085 = black_hole_mass(3000, Luminosities_NOT_Corrected_Binned_Data_Mean_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085, 
                                                                                                                                  Luminosities_NOT_Corrected_Binned_Data_Mean_SD_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_075_085)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_085_096 = black_hole_mass(3000, Luminosities_NOT_Corrected_Binned_Data_Mean_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096, 
                                                                                                                                  Luminosities_NOT_Corrected_Binned_Data_Mean_SD_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_085_096)



In [ ]:
"""
Recalculating the black hole masses from the AGN and Stellar Corrected luminosities. 
This one it for the two gaussian measurement with Uniform Narrow restrictions. (example of restrictions: PSF limited or uniform narrow.)

For the Broad part of the MgII.

Calculate black hole mass using the MgII line width and continuum luminosity at 3000Å.
    
This function computes black hole mass (in solar masses) based on the empirical relation:
    M_BH/M_☉ = 3.37 * (λL_3000/10^37 W)^0.47 * (FWHM_MgII/km s^-1)^2
"""

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_SD_025_035 = black_hole_mass(3000, Luminosities_AGN_Stellar_Binned_Data_Mean_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035, 
                                                                                                                                  Luminosities_AGN_Stellar_Binned_Data_Mean_SD_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_SD_025_035)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_035_045 = black_hole_mass(3000, Luminosities_AGN_Stellar_Binned_Data_Mean_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045, 
                                                                                                                                  Luminosities_AGN_Stellar_Binned_Data_Mean_SD_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_035_045)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_045_055 = black_hole_mass(3000, Luminosities_AGN_Stellar_Binned_Data_Mean_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055, 
                                                                                                                                  Luminosities_AGN_Stellar_Binned_Data_Mean_SD_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_045_055)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_055_065 = black_hole_mass(3000, Luminosities_AGN_Stellar_Binned_Data_Mean_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065, 
                                                                                                                                  Luminosities_AGN_Stellar_Binned_Data_Mean_SD_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_055_065)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_065_075 = black_hole_mass(3000, Luminosities_AGN_Stellar_Binned_Data_Mean_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075, 
                                                                                                                                  Luminosities_AGN_Stellar_Binned_Data_Mean_SD_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_065_075)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_075_085 = black_hole_mass(3000, Luminosities_AGN_Stellar_Binned_Data_Mean_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085, 
                                                                                                                                  Luminosities_AGN_Stellar_Binned_Data_Mean_SD_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_075_085)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_085_096 = black_hole_mass(3000, Luminosities_AGN_Stellar_Binned_Data_Mean_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096, 
                                                                                                                                  Luminosities_AGN_Stellar_Binned_Data_Mean_SD_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_085_096)



In [ ]:
"""
Recalculating the black hole masses from the AGN Only Corrected luminosities. 
This one it for the two gaussian measurement with Uniform Narrow restrictions. (example of restrictions: PSF limited or uniform narrow.)

For the Broad part of the MgII.

Calculate black hole mass using the MgII line width and continuum luminosity at 3000Å.
    
This function computes black hole mass (in solar masses) based on the empirical relation:
    M_BH/M_☉ = 3.37 * (λL_3000/10^37 W)^0.47 * (FWHM_MgII/km s^-1)^2
"""

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_SD_025_035 = black_hole_mass(3000, Luminosities_AGN_Only_Binned_Data_Mean_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035, 
                                                                                                                                  Luminosities_AGN_Only_Binned_Data_Mean_SD_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_SD_025_035)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_035_045 = black_hole_mass(3000, Luminosities_AGN_Only_Binned_Data_Mean_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045, 
                                                                                                                                  Luminosities_AGN_Only_Binned_Data_Mean_SD_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_035_045)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_045_055 = black_hole_mass(3000, Luminosities_AGN_Only_Binned_Data_Mean_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055, 
                                                                                                                                  Luminosities_AGN_Only_Binned_Data_Mean_SD_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_045_055)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_055_065 = black_hole_mass(3000, Luminosities_AGN_Only_Binned_Data_Mean_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065, 
                                                                                                                                  Luminosities_AGN_Only_Binned_Data_Mean_SD_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_055_065)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_065_075 = black_hole_mass(3000, Luminosities_AGN_Only_Binned_Data_Mean_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075, 
                                                                                                                                  Luminosities_AGN_Only_Binned_Data_Mean_SD_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_065_075)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_075_085 = black_hole_mass(3000, Luminosities_AGN_Only_Binned_Data_Mean_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085, 
                                                                                                                                  Luminosities_AGN_Only_Binned_Data_Mean_SD_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_075_085)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_085_096 = black_hole_mass(3000, Luminosities_AGN_Only_Binned_Data_Mean_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096, 
                                                                                                                                  Luminosities_AGN_Only_Binned_Data_Mean_SD_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_085_096)



In [ ]:
"""
Recalculating the black hole masses from the Stellar Only Corrected luminosities. 
This one it for the two gaussian measurement with Uniform Narrow restrictions. (example of restrictions: PSF limited or uniform narrow.)

For the Broad part of the MgII.

Calculate black hole mass using the MgII line width and continuum luminosity at 3000Å.
    
This function computes black hole mass (in solar masses) based on the empirical relation:
    M_BH/M_☉ = 3.37 * (λL_3000/10^37 W)^0.47 * (FWHM_MgII/km s^-1)^2
"""

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_SD_025_035 = black_hole_mass(3000, Luminosities_Stellar_Only_Binned_Data_Mean_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035, 
                                                                                                                                  Luminosities_Stellar_Only_Binned_Data_Mean_SD_025_035, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_SD_025_035)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_035_045 = black_hole_mass(3000, Luminosities_Stellar_Only_Binned_Data_Mean_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045, 
                                                                                                                                  Luminosities_Stellar_Only_Binned_Data_Mean_SD_035_045, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_035_045)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_045_055 = black_hole_mass(3000, Luminosities_Stellar_Only_Binned_Data_Mean_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055, 
                                                                                                                                  Luminosities_Stellar_Only_Binned_Data_Mean_SD_045_055, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_045_055)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_055_065 = black_hole_mass(3000, Luminosities_Stellar_Only_Binned_Data_Mean_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065, 
                                                                                                                                  Luminosities_Stellar_Only_Binned_Data_Mean_SD_055_065, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_055_065)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_065_075 = black_hole_mass(3000, Luminosities_Stellar_Only_Binned_Data_Mean_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075, 
                                                                                                                                  Luminosities_Stellar_Only_Binned_Data_Mean_SD_065_075, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_065_075)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_075_085 = black_hole_mass(3000, Luminosities_Stellar_Only_Binned_Data_Mean_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085, 
                                                                                                                                  Luminosities_Stellar_Only_Binned_Data_Mean_SD_075_085, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_075_085)

BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096, BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_085_096 = black_hole_mass(3000, Luminosities_Stellar_Only_Binned_Data_Mean_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096, 
                                                                                                                                  Luminosities_Stellar_Only_Binned_Data_Mean_SD_085_096, MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_SD_085_096)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure with specified dimensions
fig, ax = plt.subplots(1, 1, figsize=(9, 5), facecolor='white')

# Color scheme (same as original)
colors = {
    '025_035': "#a714ff",  # Purple (deep/cool)
    '035_045': "#ff14f5",  # Pink
    '045_055': "#14D8FF",  # Teal
    '055_065': "#60B5FF",  # Blue
    '065_075': "#00FF9C",  # Green
    '075_085': "#ffbb14",  # Orange
    '085_096': "#FF5757"   # Red (warm)
}

# Marker shapes for different datasets
markers = {
    'NOT_Corrected': "d",  # Diamond
    'AGN_Stellar': 's',    # Square
    'AGN_Only': '^',       # Triangle up
    'Stellar_Only': 'o'    # Circle
}

# Dataset configuration using your actual variables
datasets = [
    # NOT_Corrected datasets
    ('025_035', 'NOT_Corrected',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035*np.log(10))),
    ('035_045', 'NOT_Corrected',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045*np.log(10))),
    ('045_055', 'NOT_Corrected',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055*np.log(10))),
    ('055_065', 'NOT_Corrected',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065*np.log(10))),
    ('065_075', 'NOT_Corrected',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075*np.log(10))),
    ('075_085', 'NOT_Corrected',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085*np.log(10))),
    ('085_096', 'NOT_Corrected',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096*np.log(10))),
    
    # AGN_Stellar datasets
    ('025_035', 'AGN_Stellar',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Stellar',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Stellar',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Stellar',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Stellar',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Stellar',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Stellar',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096*np.log(10))),
    
    # AGN_Only datasets
    ('025_035', 'AGN_Only',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Only',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Only',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Only',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Only',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Only',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Only',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096*np.log(10))),
    
    # Stellar_Only datasets
    ('025_035', 'Stellar_Only',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035*np.log(10))),
    ('035_045', 'Stellar_Only',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045*np.log(10))),
    ('045_055', 'Stellar_Only',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055*np.log(10))),
    ('055_065', 'Stellar_Only',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065*np.log(10))),
    ('065_075', 'Stellar_Only',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075*np.log(10))),
    ('075_085', 'Stellar_Only',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085*np.log(10))),
    ('085_096', 'Stellar_Only',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096*np.log(10)))
]

# Organize data by dataset type and redshift bin
data_by_type = {'NOT_Corrected': [], 'AGN_Stellar': [], 'AGN_Only': [], 'Stellar_Only': []}
redshift_bins = []
colors_for_bar = ["#ff14f5", "#14D8FF", "#00FF9C",  "#ffbb14"]

for i, (z_range, dataset_type, x_data, y_data, x_err, y_err) in enumerate(datasets):
    z_index = i % 7
    if z_index == 0:  # First occurrence of each redshift bin
        redshift_bins.append(z_range)
    data_by_type[dataset_type].append((y_data, y_err))


# Create bar chart
redshift_labels = ['0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.905']
x_pos = np.arange(len(redshift_labels))
bar_width = 0.22

# Create legend elements for different dataset types
legend_elements = []
marker_labels = {
    'NOT_Corrected': 'NOT Corrected',
    'AGN_Stellar': 'AGN Stellar', 
    'AGN_Only': 'AGN Only',
    'Stellar_Only': 'Stellar Only'
}

for i, (dataset_type, data_points) in enumerate(data_by_type.items()):
    if len(data_points) > 0:
        y_coords = [point[0] for point in data_points]
        y_errs = [point[1] for point in data_points]
        
        # Plot bars with dataset colors
        bars = ax.bar(x_pos + i * bar_width, y_coords, bar_width, 
               yerr=y_errs, capsize=3,
               color=colors_for_bar[i], alpha=1.0, 
               edgecolor='black', linewidth=1, zorder=10)
        
        # Add to legend using the dataset colors
        legend_elements.append(plt.Rectangle((0,0),1,1, 
                                           facecolor=colors_for_bar[i], 
                                           edgecolor='black',
                                           label=marker_labels[dataset_type]))

# Add the legend
ax.legend(handles=legend_elements, loc='upper left', fontsize=10, ncol=4)

# Configure axes labels
ax.set_xlabel('Redshift', fontsize=14, color="black")
ax.set_ylabel(r"log$_{10}$(M$_{BH}$/M$_{\odot}$)", fontsize=14, color="black")
ax.set_xticks(x_pos + bar_width * 1.5)
ax.set_xticklabels(redshift_labels)

# Configure grid
ax.grid(visible=True, which='both', axis='y', 
        linestyle='--', alpha=0.7, zorder=-10)

# Set minor ticks
ax.minorticks_on()

# Configure spine thickness (MNRAS style)
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax.tick_params(axis='both', which='major', labelsize=12,
               length=8, width=2.0, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10,
               length=4, width=1.5, direction='in')

# Enable ticks on all sides
ax.tick_params(top=True, right=True)

ax.set_ylim(0, 10)
# Adjust layout
plt.tight_layout(pad=0.5)

plt.show()


In [ ]:
"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035 / 10**Stellar_Mass_Average_025_035) - np.log10(0.0025)
Delta_BH_Stellar_Broad_NOT_Corrected_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045 / 10**Stellar_Mass_Average_035_045) - np.log10(0.0025)
Delta_BH_Stellar_Broad_NOT_Corrected_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055 / 10**Stellar_Mass_Average_045_055) - np.log10(0.0025)
Delta_BH_Stellar_Broad_NOT_Corrected_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065 / 10**Stellar_Mass_Average_055_065) - np.log10(0.0025)
Delta_BH_Stellar_Broad_NOT_Corrected_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075 / 10**Stellar_Mass_Average_065_075) - np.log10(0.0025)
Delta_BH_Stellar_Broad_NOT_Corrected_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085 / 10**Stellar_Mass_Average_075_085) - np.log10(0.0025)
Delta_BH_Stellar_Broad_NOT_Corrected_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096 / 10**Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_SD_025_035, 
                                                                  10**Stellar_Mass_Average_025_035, 
                                                                  10**Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_035_045, 
                                                                  10**Stellar_Mass_Average_035_045, 
                                                                  10**Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_045_055, 
                                                                  10**Stellar_Mass_Average_045_055, 
                                                                  10**Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_055_065, 
                                                                  10**Stellar_Mass_Average_055_065, 
                                                                  10**Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_065_075, 
                                                                  10**Stellar_Mass_Average_065_075, 
                                                                  10**Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_075_085, 
                                                                  10**Stellar_Mass_Average_075_085, 
                                                                  10**Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_085_096, 
                                                                  10**Stellar_Mass_Average_085_096, 
                                                                  10**Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)

###############################################################################################################################################################
#################################################################################################################
###################################################
#########################



"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035 / 10**Stellar_Mass_Average_025_035) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Stellar_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045 / 10**Stellar_Mass_Average_035_045) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Stellar_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055 / 10**Stellar_Mass_Average_045_055) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Stellar_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065 / 10**Stellar_Mass_Average_055_065) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Stellar_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075 / 10**Stellar_Mass_Average_065_075) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Stellar_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085 / 10**Stellar_Mass_Average_075_085) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Stellar_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096 / 10**Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_SD_025_035, 
                                                                  10**Stellar_Mass_Average_025_035, 
                                                                  10**Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_035_045, 
                                                                  10**Stellar_Mass_Average_035_045, 
                                                                  10**Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_045_055, 
                                                                  10**Stellar_Mass_Average_045_055, 
                                                                  10**Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_055_065, 
                                                                  10**Stellar_Mass_Average_055_065, 
                                                                  10**Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_065_075, 
                                                                  10**Stellar_Mass_Average_065_075, 
                                                                  10**Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_075_085, 
                                                                  10**Stellar_Mass_Average_075_085, 
                                                                  10**Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_085_096, 
                                                                  10**Stellar_Mass_Average_085_096, 
                                                                  10**Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)

###############################################################################################################################################################
#################################################################################################################
###################################################
#########################



"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035 / 10**Stellar_Mass_Average_025_035) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Only_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045 / 10**Stellar_Mass_Average_035_045) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Only_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055 / 10**Stellar_Mass_Average_045_055) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Only_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065 / 10**Stellar_Mass_Average_055_065) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Only_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075 / 10**Stellar_Mass_Average_065_075) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Only_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085 / 10**Stellar_Mass_Average_075_085) - np.log10(0.0025)
Delta_BH_Stellar_Broad_AGN_Only_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096 / 10**Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_SD_025_035, 
                                                                  10**Stellar_Mass_Average_025_035, 
                                                                  10**Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Only_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_035_045, 
                                                                  10**Stellar_Mass_Average_035_045, 
                                                                  10**Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Delta_BH_Stellar_Broad_AGN_Only_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_045_055, 
                                                                  10**Stellar_Mass_Average_045_055, 
                                                                  10**Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Only_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_055_065, 
                                                                  10**Stellar_Mass_Average_055_065, 
                                                                  10**Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Only_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_065_075, 
                                                                  10**Stellar_Mass_Average_065_075, 
                                                                  10**Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Only_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_075_085, 
                                                                  10**Stellar_Mass_Average_075_085, 
                                                                  10**Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_AGN_Only_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_085_096, 
                                                                  10**Stellar_Mass_Average_085_096, 
                                                                  10**Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)

###############################################################################################################################################################
#################################################################################################################
###################################################
#########################



"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035 / 10**Stellar_Mass_Average_025_035) - np.log10(0.0025)
Delta_BH_Stellar_Broad_Stellar_Only_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045 / 10**Stellar_Mass_Average_035_045) - np.log10(0.0025)
Delta_BH_Stellar_Broad_Stellar_Only_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055 / 10**Stellar_Mass_Average_045_055) - np.log10(0.0025)
Delta_BH_Stellar_Broad_Stellar_Only_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065 / 10**Stellar_Mass_Average_055_065) - np.log10(0.0025)
Delta_BH_Stellar_Broad_Stellar_Only_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075 / 10**Stellar_Mass_Average_065_075) - np.log10(0.0025)
Delta_BH_Stellar_Broad_Stellar_Only_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085 / 10**Stellar_Mass_Average_075_085) - np.log10(0.0025)
Delta_BH_Stellar_Broad_Stellar_Only_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096 / 10**Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_SD_025_035, 
                                                                  10**Stellar_Mass_Average_025_035, 
                                                                  10**Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_035_045, 
                                                                  10**Stellar_Mass_Average_035_045, 
                                                                  10**Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_045_055, 
                                                                  10**Stellar_Mass_Average_045_055, 
                                                                  10**Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_055_065, 
                                                                  10**Stellar_Mass_Average_055_065, 
                                                                  10**Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_065_075, 
                                                                  10**Stellar_Mass_Average_065_075, 
                                                                  10**Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_075_085, 
                                                                  10**Stellar_Mass_Average_075_085, 
                                                                  10**Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_085_096, 
                                                                  10**Stellar_Mass_Average_085_096, 
                                                                  10**Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.cm as cm

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure with specified dimensions
fig, ax = plt.subplots(1, 1, figsize=(8, 6), facecolor='white')

# Color scheme (organized in dictionary)
colors = {
    '025_035': "#a714ff",  # Purple (deep/cool)
    '035_045': "#ff14f5",  # Pink
    '045_055': "#14D8FF",  # Teal
    '055_065': "#60B5FF",  # Blue
    '065_075': "#00FF9C",  # Green
    '075_085': "#ffbb14",  # Orange
    '085_096': "#FF5757"   # Red (warm)
}

# Dataset configuration for plotting (Black Hole Mass vs Luminosity)
# Four different datasets with different markers
datasets_not_corrected = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_NOT_Corrected_085_096,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096)
]

datasets_agn_stellar = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_AGN_Stellar_085_096,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096)
]

datasets_agn_only = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_AGN_Only_035_045,
     Delta_BH_Stellar_Broad_AGN_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_AGN_Only_045_055,
     Delta_BH_Stellar_Broad_AGN_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_AGN_Only_055_065,
     Delta_BH_Stellar_Broad_AGN_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_AGN_Only_065_075,
     Delta_BH_Stellar_Broad_AGN_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_AGN_Only_075_085,
     Delta_BH_Stellar_Broad_AGN_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_AGN_Only_085_096,
     Delta_BH_Stellar_Broad_AGN_Only_SD_085_096)
]

datasets_stellar_only = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_Stellar_Only_035_045,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_Stellar_Only_045_055,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_Stellar_Only_055_065,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_Stellar_Only_065_075,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_Stellar_Only_075_085,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_Stellar_Only_085_096,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096)
]

x_fit = np.linspace(0.1, 1.0, 100)

# Define power law function
def delta_Z_fit(x, delta):
    return delta * np.log10(1 + x)

# Extract data for fitting (using AGN_Only as reference)
x_data = np.array([dataset[1] for dataset in datasets_agn_only])
y_data = np.array([dataset[2] for dataset in datasets_agn_only])
y_errors = np.array([dataset[3] for dataset in datasets_agn_only])

# Print data for debugging
print("X data:", x_data)
print("Y data:", y_data)
print("Y errors:", y_errors)

# Perform fitting with better error handling and initial guess
try:
    initial_guess = [0.5]
    popt, pcov = curve_fit(
        delta_Z_fit, 
        x_data, 
        y_data, 
        sigma=y_errors, 
        absolute_sigma=True,
        p0=initial_guess,
        maxfev=5000
    )
    
    delta_fit = popt[0]
    delta_err = np.sqrt(np.diag(pcov))[0]
    fit_successful = True
    
    print(f"Fit successful! Delta = {delta_fit:.3f} ± {delta_err:.3f}")
    
except Exception as e:
    fit_successful = False
    print(f"Fit failed with error: {e}")

# Plot Merloni et al. 2010 reference line
z_ref = np.array([0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.905, 1.0])
merloni_line = 0.68 * np.log10(1 + z_ref)

ax.plot(z_ref, merloni_line, linewidth=2, linestyle="--", color="#5E386A", label="Merloni et al. 2010", zorder=5) 
ax.fill_between(z_ref, 
                merloni_line - 0.12, 
                merloni_line + 0.12, 
                color="#F3C5FF", alpha=0.4, zorder=5)

# ADD MATT ET AL. 2025 RELATION
# Parameters from Matt et al. 2025
alpha0 = 8.69
beta0 = 1.17
alpha_z = 1.04
alpha_z_err = 0.5

# Calculate Matt et al. 2025 relation
# M_BH = alpha0 * (1+z)^alpha_z * (M_bulge/10^11 M_sun)^beta0
# Converting to delta log10(M_BH/M_*) form
# Assuming M_bulge ≈ M_* and using a reference point at z=0

def matt_relation(z, alpha_z_val):
    """Calculate the Matt et al. 2025 relation in delta form"""
    # Reference at z=0
    ref_term = alpha0 * (1.0)**alpha_z_val
    # Evolution term
    evol_term = alpha0 * (1 + z)**alpha_z_val
    # Delta in log space
    return np.log10(evol_term) - np.log10(ref_term)

z_matt = np.linspace(0.0, 1.0, 100)
matt_line = matt_relation(z_matt, alpha_z)
matt_upper = matt_relation(z_matt, alpha_z + alpha_z_err)
matt_lower = matt_relation(z_matt, alpha_z - alpha_z_err)

ax.plot(z_matt, matt_line, linewidth=2, color="#0047AB", 
        label="Matt et al. 2025", linestyle='-.', zorder=-10)
ax.fill_between(z_matt, matt_lower, matt_upper, 
                color="#8ABBFF", alpha=0.4, zorder=-10)

# Plot fitted curve if successful
if fit_successful:
    y_fit = delta_Z_fit(x_fit, delta_fit)
    y_fit_upper = delta_Z_fit(x_fit, delta_fit + delta_err)
    y_fit_lower = delta_Z_fit(x_fit, delta_fit - delta_err)
    
    ax.plot(x_fit, y_fit, color='black', linewidth=2, 
            label=f'Best Fit (δ = {delta_fit:.3f})', zorder=30)
    ax.fill_between(x_fit, y_fit_lower, y_fit_upper, 
                   color='gray', alpha=0.3, zorder=25)

# Plot each dataset with different markers
all_datasets = [
    (datasets_not_corrected, "d", 'NOT Corrected'),
    (datasets_agn_stellar, 's', 'AGN + Stellar'),
    (datasets_agn_only, '^', 'AGN Only'),
    (datasets_stellar_only, 'o', 'Stellar Only')
]

for datasets, marker, label in all_datasets:
    for i, (key, x_data_point, y_data_point, y_err) in enumerate(datasets):
        # Plot data points with different markers
        ax.scatter(x_data_point, y_data_point,
                  color=colors[key],
                  edgecolor="black",
                  linewidth=1,
                  s=50,
                  marker=marker,
                  alpha=1.0,
                  zorder=10000,
                  label=label if i == 0 else "")

        # Plot error bars
        ax.errorbar(x_data_point, y_data_point,
                   yerr=y_err,
                   linestyle='',
                   ecolor='black',
                   capsize=5,
                   capthick=2,
                   elinewidth=1.5,
                   alpha=0.8,
                   zorder=100)

# Configure axes labels and title
ax.set_xlabel(r"Z", fontsize=14, color="black")
ax.set_ylabel(r"$\Delta$ log$_{10}$(M$_{BH}$/M$_{*}$)", fontsize=14, color="black")

# Create colorbar
from matplotlib.colors import LinearSegmentedColormap
color_list = list(colors.values())
custom_cmap = LinearSegmentedColormap.from_list("custom", color_list, N=256)

labels_2 = [
    r'$0.3$',
    r'$0.4$',
    r'$0.5$',
    r'$0.6$',
    r'$0.7$',
    r'$0.8$',
    r'$0.905$'
]

sm = cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=len(labels_2)-1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02, shrink=1.0)
cbar.set_ticks(np.arange(len(labels_2)))
cbar.set_ticklabels(labels_2)
cbar.ax.tick_params(labelsize=12)
cbar.set_label('Redshift', fontsize=14, labelpad=15)

# Create legend with marker types (matching second script style)
legend_elements = []

# Add marker types to legend
markers_dict = {'NOT Corrected': "d", 'AGN + Stellar': 's', 'AGN Only': '^', 'Stellar Only': 'o'}
for label, marker in markers_dict.items():
    legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w', 
                                    markerfacecolor=None, 
                                    markeredgecolor='black', markeredgewidth=1,
                                    markersize=8,
                                    label=label))

# Add reference lines to legend
legend_elements.extend([
    plt.Line2D([0], [0], linestyle="--", color="#5E386A", linewidth=2, label="Merloni et al. 2010"),
    plt.Line2D([0], [0], linestyle="-.", color="#0047AB", linewidth=2, label="Matt et al. 2025")
])

if fit_successful:
    legend_elements.append(plt.Line2D([0], [0], color='black', linewidth=2, 
                                    label=f'Best Fit (δ = {delta_fit:.3f})'))

# Create legend
legend1 = ax.legend(handles=legend_elements, loc='upper left', fontsize=10, 
                   frameon=True, fancybox=True, 
                   edgecolor='black', facecolor='white', framealpha=1.0)

# Configure grid
ax.grid(visible=True, which='major', axis='both', 
        linestyle='--', alpha=0.7, zorder=-10)

# Set minor ticks
ax.minorticks_on()

# Configure spine thickness
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax.tick_params(axis='both', which='major', labelsize=12,
               length=8, width=2.0, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10,
               length=4, width=1.5, direction='in')

# Enable ticks on all sides
ax.tick_params(top=True, right=True)

# Extend the upper y-limit slightly
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin/1.05, ymax * 1.5)

# Set x-axis limits
ax.set_xlim(0.2, 0.95)
ax.set_ylim(-0.43, 1.25)

# Adjust layout
plt.tight_layout(pad=0.5)


# Show the figure
plt.show(block=False)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8), facecolor='white')

# Colors for each dataset type
colors = {
    'NOT_Corrected': "#ff14f5",  # Purple
    'AGN_Stellar': "#14D8FF",    # Pink  
    'AGN_Only': "#00FF9C",       # Teal
    'Stellar_Only': "#ffbb14"    # Green
}

# Collect all data points and group by FWHM value
data_groups = {}

# Helper function to add data to groups
def add_to_group(fwhm, bh_mass, dataset_type):
    fwhm_key = round(fwhm, 4)  # Round to avoid floating point issues
    if fwhm_key not in data_groups:
        data_groups[fwhm_key] = {}
    data_groups[fwhm_key][dataset_type] = bh_mass

# NOT_Corrected data
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'NOT_Corrected')

# AGN_Stellar data
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'AGN_Stellar')

# AGN_Only data  
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'AGN_Only')

# Stellar_Only data
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'Stellar_Only')

# Create grouped bar chart with equal spacing (not to scale)
bar_width = 0.15
group_spacing = 1.0  # Space between FWHM groups

# Sort FWHM values and assign equal spacing positions
sorted_fwhm_values = sorted(data_groups.keys())
x_positions = {fwhm: i * group_spacing for i, fwhm in enumerate(sorted_fwhm_values)}

offset_map = {'NOT_Corrected': -1.5, 'AGN_Stellar': -0.5, 'AGN_Only': 0.5, 'Stellar_Only': 1.5}

for fwhm_val in sorted_fwhm_values:
    datasets = data_groups[fwhm_val]
    base_x = x_positions[fwhm_val]
    
    for dataset_type, bh_mass in datasets.items():
        x_pos = base_x + offset_map[dataset_type] * bar_width
        ax.bar(x_pos, bh_mass, bar_width, 
               color=colors[dataset_type], alpha=1.0, 
               edgecolor='black', linewidth=0.5, zorder=10)

# Set custom x-tick labels showing actual FWHM values
ax.set_xticks([x_positions[fwhm] for fwhm in sorted_fwhm_values])
ax.set_xticklabels([f'{fwhm:.3f}' for fwhm in sorted_fwhm_values], rotation=0)

# Create legend
legend_elements = [
    plt.Rectangle((0,0),1,1, facecolor=colors['NOT_Corrected'], edgecolor='black', alpha=0.8, label='NOT Corrected'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Stellar'], edgecolor='black', alpha=0.8, label='AGN Stellar'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Only'], edgecolor='black', alpha=0.8, label='AGN Only'),
    plt.Rectangle((0,0),1,1, facecolor=colors['Stellar_Only'], edgecolor='black', alpha=0.8, label='Stellar Only')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10, ncol=4)

# Configure axes
ax.set_xlabel(r"log$_{10}$(FWHM [km/s])", fontsize=14, color="black")
ax.set_ylabel(r"log$_{10}$(M$_{BH}$/M$_{\odot}$)", fontsize=14, color="black")

# Configure grid
ax.grid(visible=True, which='both', axis='both', linestyle='--', alpha=0.7, zorder=-10)
ax.minorticks_on()

# Configure spine thickness
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')
ax.tick_params(top=True, right=True)

plt.tight_layout(pad=0.5)
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure with specified dimensions
fig, ax = plt.subplots(1, 1, figsize=(8, 6), facecolor='white')

# Color scheme (organized in dictionary)
colors = {
    '025_035': "#a714ff",  # Purple (deep/cool)
    '035_045': "#ff14f5",  # Pink
    '045_055': "#14D8FF",  # Teal
    '055_065': "#60B5FF",  # Blue
    '065_075': "#00FF9C",  # Green
    '075_085': "#ffbb14",  # Orange
    '085_096': "#FF5757"   # Red (warm)
}

# Marker shapes for different datasets
markers = {
    'NOT_Corrected': "d",  # Heart (original)
    'AGN_Stellar': 's',           # Square
    'AGN_Only': '^',              # Triangle up
    'Stellar_Only': 'o'           # Circle
}

# Labels for legend (formatted for better readability)
labels = [
    r'$0.25 < z < 0.35$',
    r'$0.35 < z < 0.45$',
    r'$0.45 < z < 0.55$',
    r'$0.55 < z < 0.65$',
    r'$0.65 < z < 0.75$',
    r'$0.75 < z < 0.85$',
    r'$0.85 < z < 0.96$'
]

# Dataset configuration for plotting (Redshift vs Black Hole Mass)
# Format: (z_range, dataset_type, x_data, y_data, y_err)
datasets = [
    # NOT_Corrected datasets
    ('025_035', 'NOT_Corrected', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035*np.log(10))),
    ('035_045', 'NOT_Corrected', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045*np.log(10))),
    ('045_055', 'NOT_Corrected', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055*np.log(10))),
    ('055_065', 'NOT_Corrected', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065*np.log(10))),
    ('065_075', 'NOT_Corrected', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075*np.log(10))),
    ('075_085', 'NOT_Corrected', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085*np.log(10))),
    ('085_096', 'NOT_Corrected', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096*np.log(10))),
    
    # AGN_Stellar datasets
    ('025_035', 'AGN_Stellar', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Stellar', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Stellar', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Stellar', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Stellar', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Stellar', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Stellar', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096*np.log(10))),
    
    # AGN_Only datasets (these are the original ones in the code)
    ('025_035', 'AGN_Only', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Only', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Only', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Only', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Only', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Only', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Only', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096*np.log(10))),
    
    # Stellar_Only datasets
    ('025_035', 'Stellar_Only', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035*np.log(10))),
    ('035_045', 'Stellar_Only', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045*np.log(10))),
    ('045_055', 'Stellar_Only', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055*np.log(10))),
    ('055_065', 'Stellar_Only', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065*np.log(10))),
    ('065_075', 'Stellar_Only', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075*np.log(10))),
    ('075_085', 'Stellar_Only', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085*np.log(10))),
    ('085_096', 'Stellar_Only', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096*np.log(10)))
]

# Plot each dataset
for i, (z_range, dataset_type, x_data, y_data, y_err) in enumerate(datasets):
    # Get the z-range index for color
    z_index = int(i % 7)  # 7 redshift bins
    
    # Plot data points with different markers for each dataset type
    ax.scatter(x_data, y_data,
              color=colors[z_range],
              edgecolor="black",
              linewidth=1,
              s=50,  # Made points smaller (was 200/100)
              marker=markers[dataset_type],
              alpha=1.0,
              zorder=10+i)
    
    # Plot error bars (y direction only)
    ax.errorbar(x_data, y_data,
               yerr=y_err,
               linestyle='',
               ecolor='black',
               capsize=5,
               capthick=2,
               elinewidth=1.5,
               alpha=0.8,
               zorder=i)

# Create colorbar-inspired legend for shapes
from matplotlib.colors import LinearSegmentedColormap
color_list = list(colors.values())
custom_cmap = LinearSegmentedColormap.from_list("custom", color_list, N=256)

# Create legend with rainbow-filled shapes using matplotlib patches
from matplotlib.patches import Circle, Rectangle, Polygon
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# Create a simple legend with single gradient-colored shapes
legend_patches = []

# Define the shape creation for each marker type
def create_gradient_patch(marker_type, label):
    if marker_type == 'o':  # Circle
        patch = mpatches.Circle((0, 0), 0.5, facecolor=custom_cmap(0.5), 
                               edgecolor='black', linewidth=1)
    elif marker_type == 's':  # Square  
        patch = mpatches.Rectangle((-0.5, -0.5), 1, 1, facecolor=custom_cmap(0.5),
                                  edgecolor='black', linewidth=1)
    elif marker_type == '^':  # Triangle
        triangle = mpatches.RegularPolygon((0, 0), 3, 0.5, orientation=np.pi/2,
                                          facecolor=custom_cmap(0.5), 
                                          edgecolor='black', linewidth=1)
        patch = triangle
    elif marker_type == "d":  # Heart - approximate with circle for now
        patch = mpatches.Circle((0, 0), 0.5, facecolor=custom_cmap(0.5),
                               edgecolor='black', linewidth=1)
    else:
        patch = mpatches.Circle((0, 0), 0.5, facecolor=custom_cmap(0.5),
                               edgecolor='black', linewidth=1)
    
    # Set the label
    patch.set_label(label)
    return patch

# Create legend elements - let's use simple colored markers for now
legend_elements = []
marker_labels = {
    'NOT_Corrected': 'NOT Corrected',
    'AGN_Stellar': 'AGN Stellar', 
    'AGN_Only': 'AGN Only',
    'Stellar_Only': 'Stellar Only'
}

# Use different colors from the colormap for each shape
colors_for_legend = ["black", "black", "black", "black"]

for i, (dataset_type, marker) in enumerate(markers.items()):
    legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w', 
                                    markerfacecolor=None, 
                                    markeredgecolor='black', markeredgewidth=1,
                                    markersize=8,
                                    label=marker_labels[dataset_type]))

# Add the legend
ax.legend(handles=legend_elements, loc='best', fontsize=10)

# Configure axes labels and title
ax.set_xlabel(r"Z", fontsize=14, color="black")
ax.set_ylabel(r"log$_{10}$(M$_{BH}$/M$_{\odot}$)", fontsize=14, color="black")
#ax.set_title("Fixed Narrow Component Double Gaussian - Luminosity Corrected - Broad - FeII Removed", fontsize=14, pad=15)

labels_2 = [
    r'$0.3$',
    r'$0.4$',
    r'$0.5$',
    r'$0.6$',
    r'$0.7$',
    r'$0.8$',
    r'$0.905$'
]

sm = cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=len(labels_2)-1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02, shrink=1.0)
cbar.set_ticks(np.arange(len(labels_2)))
cbar.set_ticklabels(labels_2)
cbar.ax.tick_params(labelsize=12)
cbar.set_label('Redshift', fontsize=14, labelpad=15)

# Configure grid
ax.grid(visible=True, which='both', axis='both', 
        linestyle='--', alpha=0.7, zorder=-10)

# Set minor ticks
ax.minorticks_on()

# Configure spine thickness (MNRAS style)
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax.tick_params(axis='both', which='major', labelsize=12,
               length=8, width=2.0, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10,
               length=4, width=1.5, direction='in')

# Enable ticks on all sides
ax.tick_params(top=True, right=True)

# Extend the upper y-limit slightly
#ax.set_ylim(, ymax * 1.05)

# Adjust layout
plt.tight_layout(pad=0.5)

# Save figure in MNRAS-ready format
# Uncomment the following lines to save:
# fig.savefig("AGN_Gaussian_Sersic_Fit_MNRAS_Ready_085_096.pdf", dpi=300, 
#             bbox_inches='tight', facecolor='white', edgecolor='none')
# fig.savefig("AGN_Gaussian_Sersic_Fit_MNRAS_Ready_085_096.png", dpi=300, 
#             bbox_inches='tight', facecolor='white', edgecolor='none')

# Save and show the figure
plt.show(block=False)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8), facecolor='white')

# Colors for each dataset type
colors = {
    'NOT_Corrected': "#a714ff",  # Purple
    'AGN_Stellar': "#ff14f5",    # Pink  
    'AGN_Only': "#14D8FF",       # Teal
    'Stellar_Only': "#00FF9C"    # Green
}

# Collect all data points and group by redshift value
data_groups = {}

# Helper function to add data to groups
def add_to_group(redshift, bh_mass, dataset_type):
    redshift_key = redshift  # Use exact redshift values
    if redshift_key not in data_groups:
        data_groups[redshift_key] = {}
    data_groups[redshift_key][dataset_type] = bh_mass

# NOT_Corrected data
redshift_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.905]
bh_mass_vars_not_corrected = [
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096)
]

for z, bh_mass in zip(redshift_values, bh_mass_vars_not_corrected):
    add_to_group(z, bh_mass, 'NOT_Corrected')

# AGN_Stellar data
bh_mass_vars_agn_stellar = [
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096)
]

for z, bh_mass in zip(redshift_values, bh_mass_vars_agn_stellar):
    add_to_group(z, bh_mass, 'AGN_Stellar')

# AGN_Only data
bh_mass_vars_agn_only = [
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096)
]

for z, bh_mass in zip(redshift_values, bh_mass_vars_agn_only):
    add_to_group(z, bh_mass, 'AGN_Only')

# Stellar_Only data
bh_mass_vars_stellar_only = [
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085),
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096)
]

for z, bh_mass in zip(redshift_values, bh_mass_vars_stellar_only):
    add_to_group(z, bh_mass, 'Stellar_Only')

# Create grouped bar chart with equal spacing (not to scale)
bar_width = 0.15
group_spacing = 1.0  # Space between redshift groups

# Sort redshift values and assign equal spacing positions
sorted_redshift_values = sorted(data_groups.keys())
x_positions = {z: i * group_spacing for i, z in enumerate(sorted_redshift_values)}

offset_map = {'NOT_Corrected': -1.5, 'AGN_Stellar': -0.5, 'AGN_Only': 0.5, 'Stellar_Only': 1.5}

for z_val in sorted_redshift_values:
    datasets = data_groups[z_val]
    base_x = x_positions[z_val]
    
    for dataset_type, bh_mass in datasets.items():
        x_pos = base_x + offset_map[dataset_type] * bar_width
        ax.bar(x_pos, bh_mass, bar_width, 
               color=colors[dataset_type], alpha=0.8, 
               edgecolor='black', linewidth=0.5)

# Set custom x-tick labels showing actual redshift values
ax.set_xticks([x_positions[z] for z in sorted_redshift_values])
ax.set_xticklabels([f'{z}' for z in sorted_redshift_values], rotation=0)

# Create legend
legend_elements = [
    plt.Rectangle((0,0),1,1, facecolor=colors['NOT_Corrected'], edgecolor='black', alpha=0.8, label='NOT Corrected'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Stellar'], edgecolor='black', alpha=0.8, label='AGN Stellar'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Only'], edgecolor='black', alpha=0.8, label='AGN Only'),
    plt.Rectangle((0,0),1,1, facecolor=colors['Stellar_Only'], edgecolor='black', alpha=0.8, label='Stellar Only')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

# Configure axes
ax.set_xlabel(r"Redshift", fontsize=14, color="black")
ax.set_ylabel(r"log$_{10}$(M$_{BH}$/M$_{\odot}$)", fontsize=14, color="black")

# Configure grid
ax.grid(visible=True, which='both', axis='both', linestyle='--', alpha=0.7, zorder=-10)
ax.minorticks_on()

# Configure spine thickness
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')
ax.tick_params(top=True, right=True)

plt.tight_layout(pad=0.5)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import curve_fit

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure with 2x2 subplots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(18, 12), facecolor='white')

#########################################
# PLOT 1 (top-left) - Bar chart
#########################################

datasets = [
    # NOT_Corrected datasets
    ('025_035', 'NOT_Corrected',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035*np.log(10))),
    ('035_045', 'NOT_Corrected',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045*np.log(10))),
    ('045_055', 'NOT_Corrected',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055*np.log(10))),
    ('055_065', 'NOT_Corrected',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065*np.log(10))),
    ('065_075', 'NOT_Corrected',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075*np.log(10))),
    ('075_085', 'NOT_Corrected',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085*np.log(10))),
    ('085_096', 'NOT_Corrected',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096*np.log(10))),
    
    # AGN_Stellar datasets
    ('025_035', 'AGN_Stellar',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Stellar',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Stellar',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Stellar',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Stellar',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Stellar',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Stellar',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096*np.log(10))),
    
    # AGN_Only datasets
    ('025_035', 'AGN_Only',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Only',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Only',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Only',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Only',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Only',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Only',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096*np.log(10))),
    
    # Stellar_Only datasets
    ('025_035', 'Stellar_Only',
    Stellar_Mass_Average_025_035,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035),
    Stellar_Mass_SD_025_035,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035*np.log(10))),
    ('035_045', 'Stellar_Only',
    Stellar_Mass_Average_035_045,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045),
    Stellar_Mass_SD_035_045,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045*np.log(10))),
    ('045_055', 'Stellar_Only',
    Stellar_Mass_Average_045_055,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055),
    Stellar_Mass_SD_045_055,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055*np.log(10))),
    ('055_065', 'Stellar_Only',
    Stellar_Mass_Average_055_065,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065),
    Stellar_Mass_SD_055_065,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065*np.log(10))),
    ('065_075', 'Stellar_Only',
    Stellar_Mass_Average_065_075,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075),
    Stellar_Mass_SD_065_075,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075*np.log(10))),
    ('075_085', 'Stellar_Only',
    Stellar_Mass_Average_075_085,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085),
    Stellar_Mass_SD_075_085,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085*np.log(10))),
    ('085_096', 'Stellar_Only',
    Stellar_Mass_Average_085_096,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096),
    Stellar_Mass_SD_085_096,
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096*np.log(10)))
]


colors = {
    '025_035': "#a714ff",
    '035_045': "#ff14f5",
    '045_055': "#14D8FF",
    '055_065': "#60B5FF",
    '065_075': "#00FF9C",
    '075_085': "#ffbb14",
    '085_096': "#FF5757"
}

# Organize data by dataset type and redshift bin
data_by_type = {'NOT_Corrected': [], 'AGN_Stellar': [], 'AGN_Only': [], 'Stellar_Only': []}
redshift_bins = []
colors_for_bar = ["#ff14f5", "#14D8FF", "#00FF9C", "#ffbb14"]

for i, (z_range, dataset_type, x_data, y_data, x_err, y_err) in enumerate(datasets):
    z_index = i % 7
    if z_index == 0:  # First occurrence of each redshift bin
        redshift_bins.append(z_range)
    data_by_type[dataset_type].append((y_data, y_err))


# Create bar chart
redshift_labels = ['0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.905']
x_pos = np.arange(len(redshift_labels))
bar_width = 0.22

# Create legend elements for different dataset types
legend_elements = []
marker_labels = {
    'NOT_Corrected': 'NOT Corrected',
    'AGN_Stellar': 'AGN Stellar', 
    'AGN_Only': 'AGN Only',
    'Stellar_Only': 'Stellar Only'
}

for i, (dataset_type, data_points) in enumerate(data_by_type.items()):
    if len(data_points) > 0:
        y_coords = [point[0] for point in data_points]
        y_errs = [point[1] for point in data_points]
        
        # Plot bars with dataset colors
        bars = ax1.bar(x_pos + i * bar_width, y_coords, bar_width, 
               yerr=y_errs, capsize=3,
               color=colors_for_bar[i], alpha=1.0, 
               edgecolor='black', linewidth=1, zorder=10)
        
        # Add to legend using the dataset colors
        legend_elements.append(plt.Rectangle((0,0),1,1, 
                                           facecolor=colors_for_bar[i], 
                                           edgecolor='black',
                                           label=marker_labels[dataset_type]))


# You'll organize your data like this:
# data_by_type = {'NOT_Corrected': [], 'AGN_Stellar': [], 'AGN_Only': [], 'Stellar_Only': []}
# Then create bars for each type

# Add the legend
ax1.legend(handles=legend_elements, loc='upper left', fontsize=10, ncol=4)

ax1.set_xlabel('Redshift', fontsize=14, color="black")
ax1.set_ylabel(r"log$_{10}$(M$_{BH}$/M$_{\odot}$)", fontsize=14, color="black")

ax1.set_xticks(x_pos + bar_width * 1.5)
ax1.set_xticklabels(redshift_labels)

ax1.grid(visible=True, which='both', axis='y', linestyle='--', alpha=0.7, zorder=-10)

ax1.minorticks_on()

for spine in ax1.spines.values():
    spine.set_linewidth(2.5)
    
ax1.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
ax1.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')

ax1.tick_params(top=True, right=True)
ax1.set_ylim(0, 10)

#########################################
# PLOT 2 (top-right) - Delta vs Z with fitting
#########################################

datasets_not_corrected = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_NOT_Corrected_085_096,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096)
]

datasets_agn_stellar = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_AGN_Stellar_085_096,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096)
]

datasets_agn_only = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_AGN_Only_035_045,
     Delta_BH_Stellar_Broad_AGN_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_AGN_Only_045_055,
     Delta_BH_Stellar_Broad_AGN_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_AGN_Only_055_065,
     Delta_BH_Stellar_Broad_AGN_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_AGN_Only_065_075,
     Delta_BH_Stellar_Broad_AGN_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_AGN_Only_075_085,
     Delta_BH_Stellar_Broad_AGN_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_AGN_Only_085_096,
     Delta_BH_Stellar_Broad_AGN_Only_SD_085_096)
]

datasets_stellar_only = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_Stellar_Only_035_045,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_Stellar_Only_045_055,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_Stellar_Only_055_065,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_Stellar_Only_065_075,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_Stellar_Only_075_085,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_Stellar_Only_085_096,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096)
]




x_fit = np.linspace(0.1, 1.0, 100)

# Define power law function
def delta_Z_fit(x, delta):
    return delta * np.log10(1 + x)

# Extract data for fitting (using AGN_Only as reference)
x_data = np.array([dataset[1] for dataset in datasets_agn_only])
y_data = np.array([dataset[2] for dataset in datasets_agn_only])
y_errors = np.array([dataset[3] for dataset in datasets_agn_only])

# Print data for debugging
print("X data:", x_data)
print("Y data:", y_data)
print("Y errors:", y_errors)

# Perform fitting with better error handling and initial guess
try:
    initial_guess = [0.5]
    popt, pcov = curve_fit(
        delta_Z_fit, 
        x_data, 
        y_data, 
        sigma=y_errors, 
        absolute_sigma=True,
        p0=initial_guess,
        maxfev=5000
    )
    
    delta_fit = popt[0]
    delta_err = np.sqrt(np.diag(pcov))[0]
    fit_successful = True
    
    print(f"Fit successful! Delta = {delta_fit:.3f} ± {delta_err:.3f}")
    
except Exception as e:
    fit_successful = False
    print(f"Fit failed with error: {e}")

# Plot Merloni et al. 2010 reference line
z_ref = np.array([0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.905, 1.0])
merloni_line = 0.68 * np.log10(1 + z_ref)

ax2.plot(z_ref, merloni_line, linewidth=2, linestyle="--", color="#5E386A", label="Merloni et al. 2010", zorder=5) 
ax2.fill_between(z_ref, 
                merloni_line - 0.12, 
                merloni_line + 0.12, 
                color="#F3C5FF", alpha=0.4, zorder=5)

# ADD MATT ET AL. 2025 RELATION
# Parameters from Matt et al. 2025
alpha0 = 8.69
beta0 = 1.17
alpha_z = 1.04
alpha_z_err = 0.5

# Calculate Matt et al. 2025 relation
# M_BH = alpha0 * (1+z)^alpha_z * (M_bulge/10^11 M_sun)^beta0
# Converting to delta log10(M_BH/M_*) form
# Assuming M_bulge ≈ M_* and using a reference point at z=0

def matt_relation(z, alpha_z_val):
    """Calculate the Matt et al. 2025 relation in delta form"""
    # Reference at z=0
    ref_term = alpha0 * (1.0)**alpha_z_val
    # Evolution term
    evol_term = alpha0 * (1 + z)**alpha_z_val
    # Delta in log space
    return np.log10(evol_term) - np.log10(ref_term)

z_matt = np.linspace(0.0, 1.0, 100)
matt_line = matt_relation(z_matt, alpha_z)
matt_upper = matt_relation(z_matt, alpha_z + alpha_z_err)
matt_lower = matt_relation(z_matt, alpha_z - alpha_z_err)

ax2.plot(z_matt, matt_line, linewidth=2, color="#0047AB", 
        label="Matt et al. 2025", linestyle='-.', zorder=-10)
ax2.fill_between(z_matt, matt_lower, matt_upper, 
                color="#8ABBFF", alpha=0.4, zorder=-10)

# Plot fitted curve if successful
if fit_successful:
    y_fit = delta_Z_fit(x_fit, delta_fit)
    y_fit_upper = delta_Z_fit(x_fit, delta_fit + delta_err)
    y_fit_lower = delta_Z_fit(x_fit, delta_fit - delta_err)
    
    ax2.plot(x_fit, y_fit, color='black', linewidth=2, 
            label=f'Best Fit (δ = {delta_fit:.3f})', zorder=30)
    ax2.fill_between(x_fit, y_fit_lower, y_fit_upper, 
                   color='gray', alpha=0.3, zorder=25)

# Plot each dataset with different markers
all_datasets = [
    (datasets_not_corrected, "d", 'NOT Corrected'),
    (datasets_agn_stellar, 's', 'AGN + Stellar'),
    (datasets_agn_only, '^', 'AGN Only'),
    (datasets_stellar_only, 'o', 'Stellar Only')
]

for datasets, marker, label in all_datasets:
    for i, (key, x_data_point, y_data_point, y_err) in enumerate(datasets):
        # Plot data points with different markers
        ax2.scatter(x_data_point, y_data_point,
                  color=colors[key],
                  edgecolor="black",
                  linewidth=1,
                  s=50,
                  marker=marker,
                  alpha=1.0,
                  zorder=10000,
                  label=label if i == 0 else "")

        # Plot error bars
        ax2.errorbar(x_data_point, y_data_point,
                   yerr=y_err,
                   linestyle='',
                   ecolor='black',
                   capsize=5,
                   capthick=2,
                   elinewidth=1.5,
                   alpha=0.8,
                   zorder=100)

ax2.set_xlabel(r"Z", fontsize=14, color="black")
ax2.set_ylabel(r"$\Delta$ log$_{10}$(M$_{BH}$/M$_{*}$)", fontsize=14, color="black")

# Create colorbar
from matplotlib.colors import LinearSegmentedColormap
color_list = list(colors.values())
custom_cmap = LinearSegmentedColormap.from_list("custom", color_list, N=256)
labels_2 = [r'$0.3$', r'$0.4$', r'$0.5$', r'$0.6$', r'$0.7$', r'$0.8$', r'$0.905$']

sm = cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=len(labels_2)-1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax2, pad=0.02, shrink=1.0)
cbar.set_ticks(np.arange(len(labels_2)))
cbar.set_ticklabels(labels_2)
cbar.ax.tick_params(labelsize=12)
cbar.set_label('Redshift', fontsize=14, labelpad=15)

# Create legend with marker types (matching second script style)
legend_elements = []

# Add marker types to legend
markers_dict = {'NOT Corrected': "d", 'AGN + Stellar': 's', 'AGN Only': '^', 'Stellar Only': 'o'}
for label, marker in markers_dict.items():
    legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w', 
                                    markerfacecolor=None, 
                                    markeredgecolor='black', markeredgewidth=1,
                                    markersize=8,
                                    label=label))

# Add reference lines to legend
legend_elements.extend([
    plt.Line2D([0], [0], linestyle="--", color="#5E386A", linewidth=2, label="Merloni et al. 2010"),
    plt.Line2D([0], [0], linestyle="-.", color="#0047AB", linewidth=2, label="Matt et al. 2025")
])

if fit_successful:
    legend_elements.append(plt.Line2D([0], [0], color='black', linewidth=2, 
                                    label=f'Best Fit (δ = {delta_fit:.3f})'))

# Create legend
legend1 = ax2.legend(handles=legend_elements, loc='upper left', fontsize=10, 
                   frameon=True, fancybox=True, 
                   edgecolor='black', facecolor='white', framealpha=1.0)


ax2.grid(visible=True, which='major', axis='both', linestyle='--', alpha=0.7, zorder=-10)

ax2.minorticks_on()

for spine in ax2.spines.values():
    spine.set_linewidth(2.5)
    
ax2.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
ax2.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')

ax2.tick_params(top=True, right=True)

ax2.set_xlim(0.2, 0.95)
ax2.set_ylim(-0.43, 1.25)

#########################################
# PLOT 3 (bottom-left) - FWHM vs BH Mass
#########################################


# Colors for each dataset type
colors = {
    'NOT_Corrected': "#ff14f5",  # Purple
    'AGN_Stellar': "#14D8FF",    # Pink  
    'AGN_Only': "#00FF9C",       # Teal
    'Stellar_Only': "#ffbb14"    # Green
}

# Collect all data points and group by FWHM value
data_groups = {}

# Helper function to add data to groups
def add_to_group(fwhm, bh_mass, dataset_type):
    fwhm_key = round(fwhm, 4)  # Round to avoid floating point issues
    if fwhm_key not in data_groups:
        data_groups[fwhm_key] = {}
    data_groups[fwhm_key][dataset_type] = bh_mass

# NOT_Corrected data
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'NOT_Corrected')

# AGN_Stellar data
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'AGN_Stellar')

# AGN_Only data  
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'AGN_Only')

# Stellar_Only data
for suffix in ['025_035', '035_045', '045_055', '055_065', '065_075', '075_085', '085_096']:
    if suffix == '025_035':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_FeII_Removed_Broad_Sigma_025_035)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035)
    elif suffix == '035_045':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_035_045)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045)
    elif suffix == '045_055':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_045_055)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055)
    elif suffix == '055_065':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_055_065)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065)
    elif suffix == '065_075':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_065_075)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075)
    elif suffix == '075_085':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_075_085)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085)
    elif suffix == '085_096':
        fwhm = np.log10(MgII_ReStacked_Continuum_Scaled_Two_Gaussian_Velocity_Uniform_Narrow_Sigma_Broad_Sigma_085_096)
        bh_mass = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096)
    add_to_group(fwhm, bh_mass, 'Stellar_Only')

# Create grouped bar chart with equal spacing (not to scale)
bar_width = 0.15
group_spacing = 1.0  # Space between FWHM groups

# Sort FWHM values and assign equal spacing positions
sorted_fwhm_values = sorted(data_groups.keys())
x_positions = {fwhm: i * group_spacing for i, fwhm in enumerate(sorted_fwhm_values)}

offset_map = {'NOT_Corrected': -1.5, 'AGN_Stellar': -0.5, 'AGN_Only': 0.5, 'Stellar_Only': 1.5}

for fwhm_val in sorted_fwhm_values:
    datasets = data_groups[fwhm_val]
    base_x = x_positions[fwhm_val]
    
    for dataset_type, bh_mass in datasets.items():
        x_pos = base_x + offset_map[dataset_type] * bar_width
        ax3.bar(x_pos, bh_mass, bar_width, 
               color=colors[dataset_type], alpha=1.0, 
               edgecolor='black', linewidth=0.5, zorder=10)

# Set custom x-tick labels showing actual FWHM values
ax3.set_xticks([x_positions[fwhm] for fwhm in sorted_fwhm_values])
ax3.set_xticklabels([f'{fwhm:.3f}' for fwhm in sorted_fwhm_values], rotation=0)

# Create legend
legend_elements = [
    plt.Rectangle((0,0),1,1, facecolor=colors['NOT_Corrected'], edgecolor='black', alpha=0.8, label='NOT Corrected'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Stellar'], edgecolor='black', alpha=0.8, label='AGN Stellar'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Only'], edgecolor='black', alpha=0.8, label='AGN Only'),
    plt.Rectangle((0,0),1,1, facecolor=colors['Stellar_Only'], edgecolor='black', alpha=0.8, label='Stellar Only')
]
ax3.legend(handles=legend_elements, loc='upper left', fontsize=10, ncol=4)

# Configure axes
ax3.set_xlabel(r"log$_{10}$(FWHM [km/s])", fontsize=14, color="black")
ax3.set_ylabel(r"log$_{10}$(M$_{BH}$/M$_{\odot}$)", fontsize=14, color="black")

# Configure grid
ax3.grid(visible=True, which='both', axis='both', linestyle='--', alpha=0.7, zorder=-10)
ax3.minorticks_on()

# Configure spine thickness
for spine in ax3.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax3.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
ax3.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')
ax3.tick_params(top=True, right=True)

#########################################
# PLOT 4 (bottom-right) - Z vs BH Mass scatter
#########################################

# Color scheme (organized in dictionary)
colors = {
    '025_035': "#a714ff",  # Purple (deep/cool)
    '035_045': "#ff14f5",  # Pink
    '045_055': "#14D8FF",  # Teal
    '055_065': "#60B5FF",  # Blue
    '065_075': "#00FF9C",  # Green
    '075_085': "#ffbb14",  # Orange
    '085_096': "#FF5757"   # Red (warm)
}

# Marker shapes for different datasets
markers = {
    'NOT_Corrected': "d",  # Heart (original)
    'AGN_Stellar': 's',           # Square
    'AGN_Only': '^',              # Triangle up
    'Stellar_Only': 'o'           # Circle
}

# Labels for legend (formatted for better readability)
labels = [
    r'$0.25 < z < 0.35$',
    r'$0.35 < z < 0.45$',
    r'$0.45 < z < 0.55$',
    r'$0.55 < z < 0.65$',
    r'$0.65 < z < 0.75$',
    r'$0.75 < z < 0.85$',
    r'$0.85 < z < 0.96$'
]

# Dataset configuration for plotting (Redshift vs Black Hole Mass)
# Format: (z_range, dataset_type, x_data, y_data, y_err)
datasets = [
    # NOT_Corrected datasets
    ('025_035', 'NOT_Corrected', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035*np.log(10))),
    ('035_045', 'NOT_Corrected', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045*np.log(10))),
    ('045_055', 'NOT_Corrected', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055*np.log(10))),
    ('055_065', 'NOT_Corrected', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065*np.log(10))),
    ('065_075', 'NOT_Corrected', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075*np.log(10))),
    ('075_085', 'NOT_Corrected', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085*np.log(10))),
    ('085_096', 'NOT_Corrected', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096*np.log(10))),
    
    # AGN_Stellar datasets
    ('025_035', 'AGN_Stellar', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Stellar', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Stellar', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Stellar', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Stellar', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Stellar', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Stellar', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096*np.log(10))),
    
    # AGN_Only datasets (these are the original ones in the code)
    ('025_035', 'AGN_Only', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035*np.log(10))),
    ('035_045', 'AGN_Only', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045*np.log(10))),
    ('045_055', 'AGN_Only', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055*np.log(10))),
    ('055_065', 'AGN_Only', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065*np.log(10))),
    ('065_075', 'AGN_Only', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075*np.log(10))),
    ('075_085', 'AGN_Only', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085*np.log(10))),
    ('085_096', 'AGN_Only', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096*np.log(10))),
    
    # Stellar_Only datasets
    ('025_035', 'Stellar_Only', 0.3,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_SD_025_035 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035*np.log(10))),
    ('035_045', 'Stellar_Only', 0.4,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_035_045 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045*np.log(10))),
    ('045_055', 'Stellar_Only', 0.5,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_045_055 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055*np.log(10))),
    ('055_065', 'Stellar_Only', 0.6,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_055_065 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065*np.log(10))),
    ('065_075', 'Stellar_Only', 0.7,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_065_075 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075*np.log(10))),
    ('075_085', 'Stellar_Only', 0.8,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_075_085 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085*np.log(10))),
    ('085_096', 'Stellar_Only', 0.905,
    np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096),
    BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_085_096 / (BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096*np.log(10)))
]

# Plot each dataset
for i, (z_range, dataset_type, x_data, y_data, y_err) in enumerate(datasets):
    # Get the z-range index for color
    z_index = int(i % 7)  # 7 redshift bins
    
    # Plot data points with different markers for each dataset type
    ax4.scatter(x_data, y_data,
              color=colors[z_range],
              edgecolor="black",
              linewidth=1,
              s=50,  # Made points smaller (was 200/100)
              marker=markers[dataset_type],
              alpha=1.0,
              zorder=10+i)
    
    # Plot error bars (y direction only)
    ax4.errorbar(x_data, y_data,
               yerr=y_err,
               linestyle='',
               ecolor='black',
               capsize=5,
               capthick=2,
               elinewidth=1.5,
               alpha=0.8,
               zorder=i)

# Create colorbar-inspired legend for shapes
from matplotlib.colors import LinearSegmentedColormap
color_list = list(colors.values())
custom_cmap = LinearSegmentedColormap.from_list("custom", color_list, N=256)

# Create legend with rainbow-filled shapes using matplotlib patches
from matplotlib.patches import Circle, Rectangle, Polygon
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# Create a simple legend with single gradient-colored shapes
legend_patches = []

# Define the shape creation for each marker type
def create_gradient_patch(marker_type, label):
    if marker_type == 'o':  # Circle
        patch = mpatches.Circle((0, 0), 0.5, facecolor=custom_cmap(0.5), 
                               edgecolor='black', linewidth=1)
    elif marker_type == 's':  # Square  
        patch = mpatches.Rectangle((-0.5, -0.5), 1, 1, facecolor=custom_cmap(0.5),
                                  edgecolor='black', linewidth=1)
    elif marker_type == '^':  # Triangle
        triangle = mpatches.RegularPolygon((0, 0), 3, 0.5, orientation=np.pi/2,
                                          facecolor=custom_cmap(0.5), 
                                          edgecolor='black', linewidth=1)
        patch = triangle
    elif marker_type == "d":  # Heart - approximate with circle for now
        patch = mpatches.Circle((0, 0), 0.5, facecolor=custom_cmap(0.5),
                               edgecolor='black', linewidth=1)
    else:
        patch = mpatches.Circle((0, 0), 0.5, facecolor=custom_cmap(0.5),
                               edgecolor='black', linewidth=1)
    
    # Set the label
    patch.set_label(label)
    return patch

# Create legend elements - let's use simple colored markers for now
legend_elements = []
marker_labels = {
    'NOT_Corrected': 'NOT Corrected',
    'AGN_Stellar': 'AGN + Stellar', 
    'AGN_Only': 'AGN Only',
    'Stellar_Only': 'Stellar Only'
}

# Use different colors from the colormap for each shape
colors_for_legend = ["black", "black", "black", "black"]

for i, (dataset_type, marker) in enumerate(markers.items()):
    legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w', 
                                    markerfacecolor=None, 
                                    markeredgecolor='black', markeredgewidth=1,
                                    markersize=8,
                                    label=marker_labels[dataset_type]))

# Add the legend
ax4.legend(handles=legend_elements, loc='best', fontsize=10)

# Configure axes labels and title
ax4.set_xlabel(r"Z", fontsize=14, color="black")
ax4.set_ylabel(r"log$_{10}$(M$_{BH}$/M$_{\odot}$)", fontsize=14, color="black")
#ax.set_title("Fixed Narrow Component Double Gaussian - Luminosity Corrected - Broad - FeII Removed", fontsize=14, pad=15)

labels_2 = [
    r'$0.3$',
    r'$0.4$',
    r'$0.5$',
    r'$0.6$',
    r'$0.7$',
    r'$0.8$',
    r'$0.905$'
]

sm = cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=len(labels_2)-1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax4, pad=0.02, shrink=1.0)
cbar.set_ticks(np.arange(len(labels_2)))
cbar.set_ticklabels(labels_2)
cbar.ax.tick_params(labelsize=12)
cbar.set_label('Redshift', fontsize=14, labelpad=15)

# Configure grid
ax4.grid(visible=True, which='both', axis='both', 
        linestyle='--', alpha=0.7, zorder=-10)

# Set minor ticks
ax4.minorticks_on()

# Configure spine thickness (MNRAS style)
for spine in ax4.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax4.tick_params(axis='both', which='major', labelsize=12,
               length=8, width=2.0, direction='in')
ax4.tick_params(axis='both', which='minor', labelsize=10,
               length=4, width=1.5, direction='in')

# Enable ticks on all sides
ax4.tick_params(top=True, right=True)

plt.tight_layout(pad=2.0)

fig.savefig("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/Plots_For_Paper/Test_Luminositys_No_Bagpipes.png", dpi=300, 
             bbox_inches='tight', facecolor='white', edgecolor='none')

plt.show()


In [ ]:
# Getting the average stellar mass from the Bagpipes SED Fitting
Bagpipes_Corrected_Stellar_Mass_Average_025_035 = np.average(Bagpipes_Cor_Stellar_Mass_025_035)
Bagpipes_Corrected_Stellar_Mass_Average_035_045 = np.average(Bagpipes_Cor_Stellar_Mass_035_045)
Bagpipes_Corrected_Stellar_Mass_Average_045_055 = np.average(Bagpipes_Cor_Stellar_Mass_045_055)
Bagpipes_Corrected_Stellar_Mass_Average_055_065 = np.average(Bagpipes_Cor_Stellar_Mass_055_065)
Bagpipes_Corrected_Stellar_Mass_Average_065_075 = np.average(Bagpipes_Cor_Stellar_Mass_065_075)
Bagpipes_Corrected_Stellar_Mass_Average_075_085 = np.average(Bagpipes_Cor_Stellar_Mass_075_085)
Bagpipes_Corrected_Stellar_Mass_Average_085_096 = np.average(Bagpipes_Cor_Stellar_Mass_085_096)

# Getting the lower bound for each of the averaged values
Bagpipes_Corrected_Stellar_Mass_SD_025_035 = np.std(Bagpipes_Cor_Stellar_Mass_025_035)
Bagpipes_Corrected_Stellar_Mass_SD_035_045 = np.std(Bagpipes_Cor_Stellar_Mass_035_045)
Bagpipes_Corrected_Stellar_Mass_SD_045_055 = np.std(Bagpipes_Cor_Stellar_Mass_045_055)
Bagpipes_Corrected_Stellar_Mass_SD_055_065 = np.std(Bagpipes_Cor_Stellar_Mass_055_065)
Bagpipes_Corrected_Stellar_Mass_SD_065_075 = np.std(Bagpipes_Cor_Stellar_Mass_065_075)
Bagpipes_Corrected_Stellar_Mass_SD_075_085 = np.std(Bagpipes_Cor_Stellar_Mass_075_085)
Bagpipes_Corrected_Stellar_Mass_SD_085_096 = np.std(Bagpipes_Cor_Stellar_Mass_085_096)

In [ ]:
"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035 / 10**Bagpipes_Corrected_Stellar_Mass_Average_025_035) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045 / 10**Bagpipes_Corrected_Stellar_Mass_Average_035_045) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055 / 10**Bagpipes_Corrected_Stellar_Mass_Average_045_055) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065 / 10**Bagpipes_Corrected_Stellar_Mass_Average_055_065) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075 / 10**Bagpipes_Corrected_Stellar_Mass_Average_065_075) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085 / 10**Bagpipes_Corrected_Stellar_Mass_Average_075_085) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096 / 10**Bagpipes_Corrected_Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_NOT_Corrected_Mean_SD_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_NOT_Corrected_Mean_SD_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)

###############################################################################################################################################################
#################################################################################################################
###################################################
#########################



"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035 / 10**Bagpipes_Corrected_Stellar_Mass_Average_025_035) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045 / 10**Bagpipes_Corrected_Stellar_Mass_Average_035_045) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055 / 10**Bagpipes_Corrected_Stellar_Mass_Average_045_055) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065 / 10**Bagpipes_Corrected_Stellar_Mass_Average_055_065) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075 / 10**Bagpipes_Corrected_Stellar_Mass_Average_065_075) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085 / 10**Bagpipes_Corrected_Stellar_Mass_Average_075_085) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096 / 10**Bagpipes_Corrected_Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Stellar_Mean_SD_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Stellar_Mean_SD_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)

###############################################################################################################################################################
#################################################################################################################
###################################################
#########################



"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035 / 10**Bagpipes_Corrected_Stellar_Mass_Average_025_035) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045 / 10**Bagpipes_Corrected_Stellar_Mass_Average_035_045) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055 / 10**Bagpipes_Corrected_Stellar_Mass_Average_045_055) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065 / 10**Bagpipes_Corrected_Stellar_Mass_Average_055_065) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075 / 10**Bagpipes_Corrected_Stellar_Mass_Average_065_075) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085 / 10**Bagpipes_Corrected_Stellar_Mass_Average_075_085) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096 / 10**Bagpipes_Corrected_Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_AGN_Only_Mean_SD_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_AGN_Only_Mean_SD_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)

###############################################################################################################################################################
#################################################################################################################
###################################################
#########################



"""
Getting the delta black hole mass over stellar mass and the standard deviation
"""
#Getting the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035 / 10**Bagpipes_Corrected_Stellar_Mass_Average_025_035) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_035_045 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045 / 10**Bagpipes_Corrected_Stellar_Mass_Average_035_045) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_045_055 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055 / 10**Bagpipes_Corrected_Stellar_Mass_Average_045_055) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_055_065 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065 / 10**Bagpipes_Corrected_Stellar_Mass_Average_055_065) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_065_075 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075 / 10**Bagpipes_Corrected_Stellar_Mass_Average_065_075) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_075_085 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085 / 10**Bagpipes_Corrected_Stellar_Mass_Average_075_085) - np.log10(0.0025)
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_085_096 = np.log10(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096 / 10**Bagpipes_Corrected_Stellar_Mass_Average_085_096) - np.log10(0.0025)


#Getting the standard deviation for the delta black hole stellar mass relation
Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_025_035, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_FeII_Removed_Luminosity_Stellar_Only_Mean_SD_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_025_035, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_025_035, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_035_045, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_035_045, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_035_045, 
                                                                  0.0025, 
                                                                  0.0005)


Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_045_055, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_045_055, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_045_055, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_055_065, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_055_065, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_055_065, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_065_075, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_065_075, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_065_075, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_075_085, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_075_085, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_075_085, 
                                                                  0.0025, 
                                                                  0.0005)

Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096 = standard_deveation_delta_BH_Stellar(BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_085_096, 
                                                                  BH_Mass_Broad_MgII_Uniform_Narrow_Sigma_Luminosity_Stellar_Only_Mean_SD_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_Average_085_096, 
                                                                  10**Bagpipes_Corrected_Stellar_Mass_SD_085_096, 
                                                                  0.0025, 
                                                                  0.0005)



In [ ]:
print(Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096)
print("")
print(Stellar_Mass_Average_085_096, Bagpipes_Corrected_Stellar_Mass_Average_085_096)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.cm as cm

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='white')

# Color scheme (organized in dictionary)
colors = {
    '025_035': "#a714ff",  # Purple (deep/cool)
    '035_045': "#ff14f5",  # Pink
    '045_055': "#14D8FF",  # Teal
    '055_065': "#60B5FF",  # Blue
    '065_075': "#00FF9C",  # Green
    '075_085': "#ffbb14",  # Orange
    '085_096': "#FF5757"   # Red (warm)
}

# Dataset configuration for plotting (Black Hole Mass vs Luminosity)
# Four different datasets with different markers
datasets_not_corrected = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_NOT_Corrected_085_096,
     Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096)
]

datasets_agn_stellar = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_AGN_Stellar_085_096,
     Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096)
]

datasets_agn_only = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_AGN_Only_035_045,
     Delta_BH_Stellar_Broad_AGN_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_AGN_Only_045_055,
     Delta_BH_Stellar_Broad_AGN_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_AGN_Only_055_065,
     Delta_BH_Stellar_Broad_AGN_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_AGN_Only_065_075,
     Delta_BH_Stellar_Broad_AGN_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_AGN_Only_075_085,
     Delta_BH_Stellar_Broad_AGN_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_AGN_Only_085_096,
     Delta_BH_Stellar_Broad_AGN_Only_SD_085_096)
]

datasets_stellar_only = [
    ('025_035', 
     0.3,
     Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
     Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Delta_BH_Stellar_Broad_Stellar_Only_035_045,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Delta_BH_Stellar_Broad_Stellar_Only_045_055,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Delta_BH_Stellar_Broad_Stellar_Only_055_065,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Delta_BH_Stellar_Broad_Stellar_Only_065_075,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Delta_BH_Stellar_Broad_Stellar_Only_075_085,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Delta_BH_Stellar_Broad_Stellar_Only_085_096,
     Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096)
]

# SECOND SUBPLOT DATASETS (using Bagpipes_Corrected_Delta_BH)
bagpipes_datasets_not_corrected = [
    ('025_035', 
     0.3,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045),
    
    ('045_055', 
     0.5,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055),
    
    ('055_065',
     0.6,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065),
    
    ('065_075', 
     0.7,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075),
    
    ('075_085', 
     0.8,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085),
    
    ('085_096', 
     0.905,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_085_096,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096)
]

bagpipes_datasets_agn_stellar = [
    ('025_035', 
     0.3,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045),
    
    ('045_055', 
     0.5,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055),
    
    ('055_065',
     0.6,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065),
    
    ('065_075', 
     0.7,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075),
    
    ('075_085', 
     0.8,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085),
    
    ('085_096', 
     0.905,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_085_096,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096)
]

bagpipes_datasets_agn_only = [
    ('025_035', 
     0.3,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_035_045,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_045_055,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_055_065,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_065_075,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_075_085,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_085_096,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_085_096)
]

bagpipes_datasets_stellar_only = [
    ('025_035', 
     0.3,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035),
    
    ('035_045',
     0.4,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_035_045,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045),
    
    ('045_055', 
     0.5,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_045_055,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055),
    
    ('055_065',
     0.6,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_055_065,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065),
    
    ('065_075', 
     0.7,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_065_075,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075),
    
    ('075_085', 
     0.8,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_075_085,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085),
    
    ('085_096', 
     0.905,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_085_096,
     Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096)
]

x_fit = np.linspace(0.1, 1.0, 100)

# Define power law function
def delta_Z_fit(x, delta):
    return delta * np.log10(1 + x)

# Function to plot on a given axis
def plot_data(ax, datasets_nc, datasets_as, datasets_ao, datasets_so, is_bagpipes=False):
    # Extract data for fitting (using AGN_Only as reference)
    x_data = np.array([dataset[1] for dataset in datasets_ao])
    y_data = np.array([dataset[2] for dataset in datasets_ao])
    y_errors = np.array([dataset[3] for dataset in datasets_ao])
    
    # Print data for debugging
    prefix = "Bagpipes " if is_bagpipes else ""
    print(f"{prefix}X data:", x_data)
    print(f"{prefix}Y data:", y_data)
    print(f"{prefix}Y errors:", y_errors)
    
    # Perform fitting with better error handling and initial guess
    try:
        initial_guess = [0.5]
        popt, pcov = curve_fit(
            delta_Z_fit, 
            x_data, 
            y_data, 
            sigma=y_errors, 
            absolute_sigma=True,
            p0=initial_guess,
            maxfev=5000
        )
        
        delta_fit = popt[0]
        delta_err = np.sqrt(np.diag(pcov))[0]
        fit_successful = True
        
        print(f"{prefix}Fit successful! Delta = {delta_fit:.3f} ± {delta_err:.3f}")
        
    except Exception as e:
        fit_successful = False
        print(f"{prefix}Fit failed with error: {e}")
    
    # Plot Merloni et al. 2010 reference line
    z_ref = np.array([0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.905, 1.0])
    merloni_line = 0.68 * np.log10(1 + z_ref)
    
    ax.plot(z_ref, merloni_line, linewidth=2, linestyle="--", color="#5E386A", label="Merloni et al. 2010", zorder=5) 
    ax.fill_between(z_ref, 
                    merloni_line - 0.12, 
                    merloni_line + 0.12, 
                    color="#F3C5FF", alpha=0.4, zorder=5)
    
    # ADD MATT ET AL. 2025 RELATION
    # Parameters from Matt et al. 2025
    alpha0 = 8.69
    beta0 = 1.17
    alpha_z = 1.04
    alpha_z_err = 0.5
    
    # Calculate Matt et al. 2025 relation
    def matt_relation(z, alpha_z_val):
        """Calculate the Matt et al. 2025 relation in delta form"""
        ref_term = alpha0 * (1.0)**alpha_z_val
        evol_term = alpha0 * (1 + z)**alpha_z_val
        return np.log10(evol_term) - np.log10(ref_term)
    
    z_matt = np.linspace(0.0, 1.0, 100)
    matt_line = matt_relation(z_matt, alpha_z)
    matt_upper = matt_relation(z_matt, alpha_z + alpha_z_err)
    matt_lower = matt_relation(z_matt, alpha_z - alpha_z_err)
    
    ax.plot(z_matt, matt_line, linewidth=2, color="#0047AB", 
            label="Matt et al. 2025", linestyle='-.', zorder=-10)
    ax.fill_between(z_matt, matt_lower, matt_upper, 
                    color="#8ABBFF", alpha=0.4, zorder=-10)
    
    # Plot fitted curve if successful
    if fit_successful:
        y_fit = delta_Z_fit(x_fit, delta_fit)
        y_fit_upper = delta_Z_fit(x_fit, delta_fit + delta_err)
        y_fit_lower = delta_Z_fit(x_fit, delta_fit - delta_err)
        
        ax.plot(x_fit, y_fit, color='black', linewidth=2, 
                label=f'Best Fit (δ = {delta_fit:.3f})', zorder=30)
        ax.fill_between(x_fit, y_fit_lower, y_fit_upper, 
                       color='gray', alpha=0.3, zorder=25)
    
    # Plot each dataset with different markers
    all_datasets = [
        (datasets_nc, "d", 'NOT Corrected'),
        (datasets_as, 's', 'AGN + Stellar'),
        (datasets_ao, '^', 'AGN Only'),
        (datasets_so, 'o', 'Stellar Only')
    ]
    
    for datasets, marker, label in all_datasets:
        for i, (key, x_data_point, y_data_point, y_err) in enumerate(datasets):
            # Plot data points with different markers
            ax.scatter(x_data_point, y_data_point,
                      color=colors[key],
                      edgecolor="black",
                      linewidth=1,
                      s=50,
                      marker=marker,
                      alpha=1.0,
                      zorder=10000,
                      label=label if i == 0 else "")
    
            # Plot error bars
            ax.errorbar(x_data_point, y_data_point,
                       yerr=y_err,
                       linestyle='',
                       ecolor='black',
                       capsize=5,
                       capthick=2,
                       elinewidth=1.5,
                       alpha=0.8,
                       zorder=100)
    
    # Configure axes labels and title
    ax.set_xlabel(r"Z", fontsize=14, color="black")
    ax.set_ylabel(r"$\Delta$ log$_{10}$(M$_{BH}$/M$_{*}$)", fontsize=14, color="black")
    
    # Create colorbar
    from matplotlib.colors import LinearSegmentedColormap
    color_list = list(colors.values())
    custom_cmap = LinearSegmentedColormap.from_list("custom", color_list, N=256)
    
    labels_2 = [
        r'$0.3$',
        r'$0.4$',
        r'$0.5$',
        r'$0.6$',
        r'$0.7$',
        r'$0.8$',
        r'$0.905$'
    ]
    
    sm = cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=len(labels_2)-1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02, shrink=1.0)
    cbar.set_ticks(np.arange(len(labels_2)))
    cbar.set_ticklabels(labels_2)
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label('Redshift', fontsize=14, labelpad=15)
    
    # Create legend with marker types
    legend_elements = []
    
    # Add marker types to legend
    markers_dict = {'NOT Corrected': "d", 'AGN + Stellar': 's', 'AGN Only': '^', 'Stellar Only': 'o'}
    for label, marker in markers_dict.items():
        legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w', 
                                        markerfacecolor=None, 
                                        markeredgecolor='black', markeredgewidth=1,
                                        markersize=8,
                                        label=label))
    
    # Add reference lines to legend
    legend_elements.extend([
        plt.Line2D([0], [0], linestyle="--", color="#5E386A", linewidth=2, label="Merloni et al. 2010"),
        plt.Line2D([0], [0], linestyle="-.", color="#0047AB", linewidth=2, label="Matt et al. 2025")
    ])
    
    if fit_successful:
        legend_elements.append(plt.Line2D([0], [0], color='black', linewidth=2, 
                                        label=f'Best Fit (δ = {delta_fit:.3f})'))
    
    # Create legend
    legend1 = ax.legend(handles=legend_elements, loc='upper left', fontsize=10, 
                       frameon=True, fancybox=True, 
                       edgecolor='black', facecolor='white', framealpha=1.0)
    
    # Configure grid
    ax.grid(visible=True, which='major', axis='both', 
            linestyle='--', alpha=0.7, zorder=-10)
    
    # Set minor ticks
    ax.minorticks_on()
    
    # Configure spine thickness
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
    
    # Configure tick parameters
    ax.tick_params(axis='both', which='major', labelsize=12,
                   length=8, width=2.0, direction='in')
    ax.tick_params(axis='both', which='minor', labelsize=10,
                   length=4, width=1.5, direction='in')
    
    # Enable ticks on all sides
    ax.tick_params(top=True, right=True)
    
    # Extend the upper y-limit slightly
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin/1.05, ymax * 1.5)
    
    # Set x-axis limits
    ax.set_xlim(0.2, 0.95)
    ax.set_ylim(-0.43, 1.25)

# Plot first subplot (original data)
plot_data(ax1, datasets_not_corrected, datasets_agn_stellar, 
          datasets_agn_only, datasets_stellar_only, is_bagpipes=False)

# Plot second subplot (Bagpipes corrected data)
plot_data(ax2, bagpipes_datasets_not_corrected, bagpipes_datasets_agn_stellar, 
          bagpipes_datasets_agn_only, bagpipes_datasets_stellar_only, is_bagpipes=True)

# Adjust layout
plt.tight_layout(pad=0.5)

# Show the figure
plt.show(block=False)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Your actual data variables - DO NOT CHANGE
# These need to be defined before running this script

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(8, 8), facecolor='white')

# Colors for each dataset type - lighter colors for original, contrasting colors for Bagpipes corrected
colors = {
    'NOT_Corrected': "#a714ff",  # Purple
    'AGN_Stellar': "#ff14f5",    # Pink  
    'AGN_Only': "#14D8FF",       # Teal
    'Stellar_Only': "#00FF9C"    # Green
}

# Contrasting colors for Bagpipes corrected data
colors_dark = {
    'NOT_Corrected': "#FFD700",  # Gold (contrasts with purple)
    'AGN_Stellar': "#00CED1",    # Dark Turquoise (contrasts with pink)  
    'AGN_Only': "#FF6347",       # Tomato Red (contrasts with teal)
    'Stellar_Only': "#FF1493"    # Deep Pink (contrasts with green)
}

# Collect all data points and group by redshift value
data_groups = {}
error_groups = {}

# Helper function to add data to groups
def add_to_group(redshift, delta_value, dataset_type, error_value=None):
    redshift_key = redshift
    if redshift_key not in data_groups:
        data_groups[redshift_key] = {}
        error_groups[redshift_key] = {}
    data_groups[redshift_key][dataset_type] = delta_value
    if error_value is not None:
        error_groups[redshift_key][dataset_type] = error_value

# Redshift values
redshift_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.905]

# NOT_Corrected data
delta_vars_not_corrected = [
    Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
    Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
    Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
    Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
    Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
    Delta_BH_Stellar_Broad_NOT_Corrected_085_096
]

error_vars_not_corrected = [
    Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_not_corrected, error_vars_not_corrected):
    add_to_group(z, delta, 'NOT_Corrected', error)

# AGN_Stellar data
delta_vars_agn_stellar = [
    Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
    Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
    Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
    Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
    Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
    Delta_BH_Stellar_Broad_AGN_Stellar_085_096
]

error_vars_agn_stellar = [
    Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_agn_stellar, error_vars_agn_stellar):
    add_to_group(z, delta, 'AGN_Stellar', error)

# AGN_Only data
delta_vars_agn_only = [
    Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_AGN_Only_035_045,
    Delta_BH_Stellar_Broad_AGN_Only_045_055,
    Delta_BH_Stellar_Broad_AGN_Only_055_065,
    Delta_BH_Stellar_Broad_AGN_Only_065_075,
    Delta_BH_Stellar_Broad_AGN_Only_075_085,
    Delta_BH_Stellar_Broad_AGN_Only_085_096
]

error_vars_agn_only = [
    Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_AGN_Only_SD_035_045,
    Delta_BH_Stellar_Broad_AGN_Only_SD_045_055,
    Delta_BH_Stellar_Broad_AGN_Only_SD_055_065,
    Delta_BH_Stellar_Broad_AGN_Only_SD_065_075,
    Delta_BH_Stellar_Broad_AGN_Only_SD_075_085,
    Delta_BH_Stellar_Broad_AGN_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_agn_only, error_vars_agn_only):
    add_to_group(z, delta, 'AGN_Only', error)

# Stellar_Only data
delta_vars_stellar_only = [
    Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_Stellar_Only_035_045,
    Delta_BH_Stellar_Broad_Stellar_Only_045_055,
    Delta_BH_Stellar_Broad_Stellar_Only_055_065,
    Delta_BH_Stellar_Broad_Stellar_Only_065_075,
    Delta_BH_Stellar_Broad_Stellar_Only_075_085,
    Delta_BH_Stellar_Broad_Stellar_Only_085_096
]

error_vars_stellar_only = [
    Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_stellar_only, error_vars_stellar_only):
    add_to_group(z, delta, 'Stellar_Only', error)

# Bagpipes Corrected NOT_Corrected data
bagpipes_delta_vars_not_corrected = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_085_096
]

bagpipes_error_vars_not_corrected = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_not_corrected, bagpipes_error_vars_not_corrected):
    add_to_group(z, delta, 'Bagpipes_NOT_Corrected', error)

# Bagpipes Corrected AGN_Stellar data
bagpipes_delta_vars_agn_stellar = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_085_096
]

bagpipes_error_vars_agn_stellar = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_agn_stellar, bagpipes_error_vars_agn_stellar):
    add_to_group(z, delta, 'Bagpipes_AGN_Stellar', error)

# Bagpipes Corrected AGN_Only data
bagpipes_delta_vars_agn_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_085_096
]

bagpipes_error_vars_agn_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_agn_only, bagpipes_error_vars_agn_only):
    add_to_group(z, delta, 'Bagpipes_AGN_Only', error)

# Bagpipes Corrected Stellar_Only data
bagpipes_delta_vars_stellar_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_085_096
]

bagpipes_error_vars_stellar_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_stellar_only, bagpipes_error_vars_stellar_only):
    add_to_group(z, delta, 'Bagpipes_Stellar_Only', error)

# Create grouped bar chart with equal spacing
bar_width = 0.15  # Back to original width since bars will overlap
group_spacing = 1.0

# Sort redshift values and assign equal spacing positions
sorted_redshift_values = sorted(data_groups.keys())
x_positions = {z: i * group_spacing for i, z in enumerate(sorted_redshift_values)}

# Update offset map - same positions for original and Bagpipes (they'll overlap)
offset_map = {
    'NOT_Corrected': -1.5,
    'AGN_Stellar': -0.5,
    'AGN_Only': 0.5,
    'Stellar_Only': 1.5,
    'Bagpipes_NOT_Corrected': -1.5,  # Same as NOT_Corrected
    'Bagpipes_AGN_Stellar': -0.5,     # Same as AGN_Stellar
    'Bagpipes_AGN_Only': 0.5,         # Same as AGN_Only
    'Bagpipes_Stellar_Only': 1.5      # Same as Stellar_Only
}

for z_val in sorted_redshift_values:
    datasets = data_groups[z_val]
    errors = error_groups[z_val]
    base_x = x_positions[z_val]
    
    # First plot the original data (will be in back)
    for dataset_type in ['NOT_Corrected', 'AGN_Stellar', 'AGN_Only', 'Stellar_Only']:
        if dataset_type in datasets:
            delta_value = datasets[dataset_type]
            error_value = errors.get(dataset_type, 0)
            x_pos = base_x + offset_map[dataset_type] * bar_width
            ax.bar(x_pos, delta_value, bar_width, 
                   color=colors[dataset_type], alpha=0.9, 
                   edgecolor='black', linewidth=1.0, zorder=1)
            # Add error bars for original data
            ax.errorbar(x_pos, delta_value, yerr=error_value,
                       fmt='none', ecolor='black', capsize=3, 
                       capthick=1.5, elinewidth=1.2, alpha=0.8, zorder=2)
    
    # Then plot the Bagpipes data on top
    for dataset_type in ['Bagpipes_NOT_Corrected', 'Bagpipes_AGN_Stellar', 'Bagpipes_AGN_Only', 'Bagpipes_Stellar_Only']:
        if dataset_type in datasets:
            delta_value = datasets[dataset_type]
            error_value = errors.get(dataset_type, 0)
            x_pos = base_x + offset_map[dataset_type] * bar_width
            base_type = dataset_type.replace('Bagpipes_', '')
            color = colors_dark[base_type]
            ax.bar(x_pos, delta_value, bar_width, 
                   color=color, alpha=0.3, 
                   edgecolor='black', linewidth=1.0, zorder=3)
            # Add error bars for Bagpipes data
            ax.errorbar(x_pos, delta_value, yerr=error_value,
                       fmt='none', ecolor='black', capsize=3, 
                       capthick=1.5, elinewidth=1.2, alpha=0.6, zorder=4)

# Set custom x-tick labels showing actual redshift values
ax.set_xticks([x_positions[z] for z in sorted_redshift_values])
ax.set_xticklabels([f'{z}' for z in sorted_redshift_values], rotation=0)

# Create legend with 2 columns
legend_elements_col1 = [
    plt.Rectangle((0,0),1,1, facecolor=colors['NOT_Corrected'], edgecolor='black', alpha=0.9, label='NOT Corrected'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Stellar'], edgecolor='black', alpha=0.9, label='AGN Stellar'),
    plt.Rectangle((0,0),1,1, facecolor=colors['AGN_Only'], edgecolor='black', alpha=0.9, label='AGN Only'),
    plt.Rectangle((0,0),1,1, facecolor=colors['Stellar_Only'], edgecolor='black', alpha=0.9, label='Stellar Only')
]

legend_elements_col2 = [
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['NOT_Corrected'], edgecolor='black', alpha=0.3, label='NOT Corrected (Corrected M*)'),
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['AGN_Stellar'], edgecolor='black', alpha=0.3, label='AGN Stellar (Corrected M*)'),
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['AGN_Only'], edgecolor='black', alpha=0.3, label='AGN Only (Corrected M*)'),
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['Stellar_Only'], edgecolor='black', alpha=0.3, label='Stellar Only (Corrected M*)')
]

legend_elements = legend_elements_col1 + legend_elements_col2
ax.legend(handles=legend_elements, loc='upper left', fontsize=9, ncol=2)

# Configure axes
ax.set_xlabel(r"Redshift", fontsize=14, color="black")
ax.set_ylabel(r"$\Delta$ log$_{10}$(M$_{BH}$/M$_{*}$)", fontsize=14, color="black")

# Configure grid
ax.grid(visible=True, which='both', axis='both', linestyle='-', alpha=0.3, zorder=0)
ax.minorticks_on()

# Configure spine thickness
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

# Configure tick parameters
ax.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')
ax.tick_params(top=True, right=True)

plt.tight_layout(pad=0.5)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.cm as cm

# Set MNRAS-compliant figure parameters
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 2.5,
    'axes.grid': True,
    'grid.alpha': 0.7,
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.major.width': 2.0,
    'ytick.minor.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0
})

# Create figure with GridSpec: 2 plots on top, 1 plot on bottom
fig = plt.figure(figsize=(14, 14), facecolor='white')
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.3, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

# ============================================================================
# FIRST TWO PLOTS (from document 1)
# ============================================================================

# Color scheme (organized in dictionary)
colors = {
    '025_035': "#a714ff",  # Purple (deep/cool)
    '035_045': "#ff14f5",  # Pink
    '045_055': "#14D8FF",  # Teal
    '055_065': "#60B5FF",  # Blue
    '065_075': "#00FF9C",  # Green
    '075_085': "#ffbb14",  # Orange
    '085_096': "#FF5757"   # Red (warm)
}

# Dataset configuration for plotting (Black Hole Mass vs Luminosity)
datasets_not_corrected = [
    ('025_035', 0.3, Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035, Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Delta_BH_Stellar_Broad_NOT_Corrected_035_045, Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045),
    ('045_055', 0.5, Delta_BH_Stellar_Broad_NOT_Corrected_045_055, Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055),
    ('055_065', 0.6, Delta_BH_Stellar_Broad_NOT_Corrected_055_065, Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065),
    ('065_075', 0.7, Delta_BH_Stellar_Broad_NOT_Corrected_065_075, Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075),
    ('075_085', 0.8, Delta_BH_Stellar_Broad_NOT_Corrected_075_085, Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085),
    ('085_096', 0.905, Delta_BH_Stellar_Broad_NOT_Corrected_085_096, Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096)
]

datasets_agn_stellar = [
    ('025_035', 0.3, Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035, Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Delta_BH_Stellar_Broad_AGN_Stellar_035_045, Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045),
    ('045_055', 0.5, Delta_BH_Stellar_Broad_AGN_Stellar_045_055, Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055),
    ('055_065', 0.6, Delta_BH_Stellar_Broad_AGN_Stellar_055_065, Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065),
    ('065_075', 0.7, Delta_BH_Stellar_Broad_AGN_Stellar_065_075, Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075),
    ('075_085', 0.8, Delta_BH_Stellar_Broad_AGN_Stellar_075_085, Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085),
    ('085_096', 0.905, Delta_BH_Stellar_Broad_AGN_Stellar_085_096, Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096)
]

datasets_agn_only = [
    ('025_035', 0.3, Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035, Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Delta_BH_Stellar_Broad_AGN_Only_035_045, Delta_BH_Stellar_Broad_AGN_Only_SD_035_045),
    ('045_055', 0.5, Delta_BH_Stellar_Broad_AGN_Only_045_055, Delta_BH_Stellar_Broad_AGN_Only_SD_045_055),
    ('055_065', 0.6, Delta_BH_Stellar_Broad_AGN_Only_055_065, Delta_BH_Stellar_Broad_AGN_Only_SD_055_065),
    ('065_075', 0.7, Delta_BH_Stellar_Broad_AGN_Only_065_075, Delta_BH_Stellar_Broad_AGN_Only_SD_065_075),
    ('075_085', 0.8, Delta_BH_Stellar_Broad_AGN_Only_075_085, Delta_BH_Stellar_Broad_AGN_Only_SD_075_085),
    ('085_096', 0.905, Delta_BH_Stellar_Broad_AGN_Only_085_096, Delta_BH_Stellar_Broad_AGN_Only_SD_085_096)
]

datasets_stellar_only = [
    ('025_035', 0.3, Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035, Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Delta_BH_Stellar_Broad_Stellar_Only_035_045, Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045),
    ('045_055', 0.5, Delta_BH_Stellar_Broad_Stellar_Only_045_055, Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055),
    ('055_065', 0.6, Delta_BH_Stellar_Broad_Stellar_Only_055_065, Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065),
    ('065_075', 0.7, Delta_BH_Stellar_Broad_Stellar_Only_065_075, Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075),
    ('075_085', 0.8, Delta_BH_Stellar_Broad_Stellar_Only_075_085, Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085),
    ('085_096', 0.905, Delta_BH_Stellar_Broad_Stellar_Only_085_096, Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096)
]

bagpipes_datasets_not_corrected = [
    ('025_035', 0.3, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_035_045, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045),
    ('045_055', 0.5, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_045_055, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055),
    ('055_065', 0.6, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_055_065, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065),
    ('065_075', 0.7, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_065_075, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075),
    ('075_085', 0.8, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_075_085, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085),
    ('085_096', 0.905, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_085_096, Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096)
]

bagpipes_datasets_agn_stellar = [
    ('025_035', 0.3, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_035_045, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045),
    ('045_055', 0.5, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_045_055, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055),
    ('055_065', 0.6, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_055_065, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065),
    ('065_075', 0.7, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_065_075, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075),
    ('075_085', 0.8, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_075_085, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085),
    ('085_096', 0.905, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_085_096, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096)
]

bagpipes_datasets_agn_only = [
    ('025_035', 0.3, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_035_045, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_035_045),
    ('045_055', 0.5, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_045_055, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_045_055),
    ('055_065', 0.6, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_055_065, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_055_065),
    ('065_075', 0.7, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_065_075, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_065_075),
    ('075_085', 0.8, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_075_085, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_075_085),
    ('085_096', 0.905, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_085_096, Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_085_096)
]

bagpipes_datasets_stellar_only = [
    ('025_035', 0.3, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035),
    ('035_045', 0.4, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_035_045, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045),
    ('045_055', 0.5, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_045_055, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055),
    ('055_065', 0.6, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_055_065, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065),
    ('065_075', 0.7, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_065_075, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075),
    ('075_085', 0.8, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_075_085, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085),
    ('085_096', 0.905, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_085_096, Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096)
]

x_fit = np.linspace(0.1, 1.0, 100)

def delta_Z_fit(x, delta):
    return delta * np.log10(1 + x)

def plot_data(ax, datasets_nc, datasets_as, datasets_ao, datasets_so, is_bagpipes=False):
    x_data = np.array([dataset[1] for dataset in datasets_ao])
    y_data = np.array([dataset[2] for dataset in datasets_ao])
    y_errors = np.array([dataset[3] for dataset in datasets_ao])
    
    prefix = "Bagpipes " if is_bagpipes else ""
    print(f"{prefix}X data:", x_data)
    print(f"{prefix}Y data:", y_data)
    print(f"{prefix}Y errors:", y_errors)
    
    try:
        initial_guess = [0.5]
        popt, pcov = curve_fit(delta_Z_fit, x_data, y_data, sigma=y_errors, absolute_sigma=True, p0=initial_guess, maxfev=5000)
        delta_fit = popt[0]
        delta_err = np.sqrt(np.diag(pcov))[0]
        fit_successful = True
        print(f"{prefix}Fit successful! Delta = {delta_fit:.3f} ± {delta_err:.3f}")
    except Exception as e:
        fit_successful = False
        print(f"{prefix}Fit failed with error: {e}")
    
    z_ref = np.array([0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.905, 1.0])
    merloni_line = 0.68 * np.log10(1 + z_ref)
    ax.plot(z_ref, merloni_line, linewidth=2, linestyle="--", color="#5E386A", label="Merloni et al. 2010", zorder=5) 
    ax.fill_between(z_ref, merloni_line - 0.12, merloni_line + 0.12, color="#F3C5FF", alpha=0.4, zorder=5)
    
    alpha0 = 8.69
    beta0 = 1.17
    alpha_z = 1.04
    alpha_z_err = 0.5
    
    def matt_relation(z, alpha_z_val):
        ref_term = alpha0 * (1.0)**alpha_z_val
        evol_term = alpha0 * (1 + z)**alpha_z_val
        return np.log10(evol_term) - np.log10(ref_term)
    
    z_matt = np.linspace(0.0, 1.0, 100)
    matt_line = matt_relation(z_matt, alpha_z)
    matt_upper = matt_relation(z_matt, alpha_z + alpha_z_err)
    matt_lower = matt_relation(z_matt, alpha_z - alpha_z_err)
    
    ax.plot(z_matt, matt_line, linewidth=2, color="#0047AB", label="Matt et al. 2025", linestyle='-.', zorder=-10)
    ax.fill_between(z_matt, matt_lower, matt_upper, color="#8ABBFF", alpha=0.4, zorder=-10)
    
    if fit_successful:
        y_fit = delta_Z_fit(x_fit, delta_fit)
        y_fit_upper = delta_Z_fit(x_fit, delta_fit + delta_err)
        y_fit_lower = delta_Z_fit(x_fit, delta_fit - delta_err)
        ax.plot(x_fit, y_fit, color='black', linewidth=2, label=f'Best Fit (δ = {delta_fit:.3f})', zorder=30)
        ax.fill_between(x_fit, y_fit_lower, y_fit_upper, color='gray', alpha=0.3, zorder=25)
    
    all_datasets = [
        (datasets_nc, "d", 'NOT Corrected'),
        (datasets_as, 's', 'AGN + Stellar'),
        (datasets_ao, '^', 'AGN Only'),
        (datasets_so, 'o', 'Stellar Only')
    ]
    
    for datasets, marker, label in all_datasets:
        for i, (key, x_data_point, y_data_point, y_err) in enumerate(datasets):
            ax.scatter(x_data_point, y_data_point, color=colors[key], edgecolor="black", linewidth=1, s=50, marker=marker, alpha=1.0, zorder=10000, label=label if i == 0 else "")
            ax.errorbar(x_data_point, y_data_point, yerr=y_err, linestyle='', ecolor='black', capsize=5, capthick=2, elinewidth=1.5, alpha=0.8, zorder=100)
    
    ax.set_xlabel(r"Z", fontsize=14, color="black")
    ax.set_ylabel(r"$\Delta$ log$_{10}$(M$_{BH}$/M$_{*}$)", fontsize=14, color="black")
    
    from matplotlib.colors import LinearSegmentedColormap
    color_list = list(colors.values())
    custom_cmap = LinearSegmentedColormap.from_list("custom", color_list, N=256)
    
    labels_2 = [r'$0.3$', r'$0.4$', r'$0.5$', r'$0.6$', r'$0.7$', r'$0.8$', r'$0.905$']
    
    sm = cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=len(labels_2)-1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02, shrink=1.0)
    cbar.set_ticks(np.arange(len(labels_2)))
    cbar.set_ticklabels(labels_2)
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label('Redshift', fontsize=14, labelpad=15)
    
    legend_elements = []
    markers_dict = {'NOT Corrected': "d", 'AGN + Stellar': 's', 'AGN Only': '^', 'Stellar Only': 'o'}
    for label, marker in markers_dict.items():
        legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w', markerfacecolor=None, markeredgecolor='black', markeredgewidth=1, markersize=8, label=label))
    
    legend_elements.extend([
        plt.Line2D([0], [0], linestyle="--", color="#5E386A", linewidth=2, label="Merloni et al. 2010"),
        plt.Line2D([0], [0], linestyle="-.", color="#0047AB", linewidth=2, label="Matt et al. 2025")
    ])
    
    if fit_successful:
        legend_elements.append(plt.Line2D([0], [0], color='black', linewidth=2, label=f'Best Fit (δ = {delta_fit:.3f})'))
    
    legend1 = ax.legend(handles=legend_elements, loc='upper left', fontsize=10, frameon=True, fancybox=True, edgecolor='black', facecolor='white', framealpha=1.0)
    
    ax.grid(visible=True, which='major', axis='both', linestyle='--', alpha=0.7, zorder=-10)
    ax.minorticks_on()
    
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
    
    ax.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
    ax.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')
    ax.tick_params(top=True, right=True)
    
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin/1.05, ymax * 1.5)
    ax.set_xlim(0.2, 0.95)
    ax.set_ylim(-0.43, 1.25)

plot_data(ax1, datasets_not_corrected, datasets_agn_stellar, datasets_agn_only, datasets_stellar_only, is_bagpipes=False)
plot_data(ax2, bagpipes_datasets_not_corrected, bagpipes_datasets_agn_stellar, bagpipes_datasets_agn_only, bagpipes_datasets_stellar_only, is_bagpipes=True)

# ============================================================================
# THIRD PLOT (from document 2) - Bar Chart
# ============================================================================

colors_bar = {
    'NOT_Corrected': "#a714ff",
    'AGN_Stellar': "#ff14f5",
    'AGN_Only': "#14D8FF",
    'Stellar_Only': "#00FF9C"
}

colors_dark = {
    'NOT_Corrected': "#FFD700",
    'AGN_Stellar': "#00CED1",
    'AGN_Only': "#FF6347",
    'Stellar_Only': "#FF1493"
}

data_groups = {}
error_groups = {}

def add_to_group(redshift, delta_value, dataset_type, error_value=None):
    redshift_key = redshift
    if redshift_key not in data_groups:
        data_groups[redshift_key] = {}
        error_groups[redshift_key] = {}
    data_groups[redshift_key][dataset_type] = delta_value
    if error_value is not None:
        error_groups[redshift_key][dataset_type] = error_value

redshift_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.905]

delta_vars_not_corrected = [
    Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
    Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
    Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
    Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
    Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
    Delta_BH_Stellar_Broad_NOT_Corrected_085_096
]

error_vars_not_corrected = [
    Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085,
    Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_not_corrected, error_vars_not_corrected):
    add_to_group(z, delta, 'NOT_Corrected', error)

delta_vars_agn_stellar = [
    Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
    Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
    Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
    Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
    Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
    Delta_BH_Stellar_Broad_AGN_Stellar_085_096
]

error_vars_agn_stellar = [
    Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085,
    Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_agn_stellar, error_vars_agn_stellar):
    add_to_group(z, delta, 'AGN_Stellar', error)

delta_vars_agn_only = [
    Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_AGN_Only_035_045,
    Delta_BH_Stellar_Broad_AGN_Only_045_055,
    Delta_BH_Stellar_Broad_AGN_Only_055_065,
    Delta_BH_Stellar_Broad_AGN_Only_065_075,
    Delta_BH_Stellar_Broad_AGN_Only_075_085,
    Delta_BH_Stellar_Broad_AGN_Only_085_096
]

error_vars_agn_only = [
    Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_AGN_Only_SD_035_045,
    Delta_BH_Stellar_Broad_AGN_Only_SD_045_055,
    Delta_BH_Stellar_Broad_AGN_Only_SD_055_065,
    Delta_BH_Stellar_Broad_AGN_Only_SD_065_075,
    Delta_BH_Stellar_Broad_AGN_Only_SD_075_085,
    Delta_BH_Stellar_Broad_AGN_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_agn_only, error_vars_agn_only):
    add_to_group(z, delta, 'AGN_Only', error)

delta_vars_stellar_only = [
    Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
    Delta_BH_Stellar_Broad_Stellar_Only_035_045,
    Delta_BH_Stellar_Broad_Stellar_Only_045_055,
    Delta_BH_Stellar_Broad_Stellar_Only_055_065,
    Delta_BH_Stellar_Broad_Stellar_Only_065_075,
    Delta_BH_Stellar_Broad_Stellar_Only_075_085,
    Delta_BH_Stellar_Broad_Stellar_Only_085_096
]

error_vars_stellar_only = [
    Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085,
    Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, delta_vars_stellar_only, error_vars_stellar_only):
    add_to_group(z, delta, 'Stellar_Only', error)

bagpipes_delta_vars_not_corrected = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_085_096
]

bagpipes_error_vars_not_corrected = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_NOT_Corrected_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_not_corrected, bagpipes_error_vars_not_corrected):
    add_to_group(z, delta, 'Bagpipes_NOT_Corrected', error)

bagpipes_delta_vars_agn_stellar = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_085_096
]

bagpipes_error_vars_agn_stellar = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Stellar_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_agn_stellar, bagpipes_error_vars_agn_stellar):
    add_to_group(z, delta, 'Bagpipes_AGN_Stellar', error)

bagpipes_delta_vars_agn_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_085_096
]

bagpipes_error_vars_agn_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_AGN_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_agn_only, bagpipes_error_vars_agn_only):
    add_to_group(z, delta, 'Bagpipes_AGN_Only', error)

bagpipes_delta_vars_stellar_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_085_096
]

bagpipes_error_vars_stellar_only = [
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_FeII_Removed_SD_025_035,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_035_045,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_045_055,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_055_065,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_065_075,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_075_085,
    Bagpipes_Corrected_Delta_BH_Stellar_Broad_Stellar_Only_SD_085_096
]

for z, delta, error in zip(redshift_values, bagpipes_delta_vars_stellar_only, bagpipes_error_vars_stellar_only):
    add_to_group(z, delta, 'Bagpipes_Stellar_Only', error)

bar_width = 0.15
group_spacing = 1.0

sorted_redshift_values = sorted(data_groups.keys())
x_positions = {z: i * group_spacing for i, z in enumerate(sorted_redshift_values)}

offset_map = {
    'NOT_Corrected': -1.5,
    'AGN_Stellar': -0.5,
    'AGN_Only': 0.5,
    'Stellar_Only': 1.5,
    'Bagpipes_NOT_Corrected': -1.5,
    'Bagpipes_AGN_Stellar': -0.5,
    'Bagpipes_AGN_Only': 0.5,
    'Bagpipes_Stellar_Only': 1.5
}

for z_val in sorted_redshift_values:
    datasets = data_groups[z_val]
    errors = error_groups[z_val]
    base_x = x_positions[z_val]
    
    for dataset_type in ['NOT_Corrected', 'AGN_Stellar', 'AGN_Only', 'Stellar_Only']:
        if dataset_type in datasets:
            delta_value = datasets[dataset_type]
            error_value = errors.get(dataset_type, 0)
            x_pos = base_x + offset_map[dataset_type] * bar_width
            ax3.bar(x_pos, delta_value, bar_width, color=colors_bar[dataset_type], alpha=0.9, edgecolor='black', linewidth=1.0, zorder=1)
            ax3.errorbar(x_pos, delta_value, yerr=error_value, fmt='none', ecolor='black', capsize=3, capthick=1.5, elinewidth=1.2, alpha=0.8, zorder=2)
    
    for dataset_type in ['Bagpipes_NOT_Corrected', 'Bagpipes_AGN_Stellar', 'Bagpipes_AGN_Only', 'Bagpipes_Stellar_Only']:
        if dataset_type in datasets:
            delta_value = datasets[dataset_type]
            error_value = errors.get(dataset_type, 0)
            x_pos = base_x + offset_map[dataset_type] * bar_width
            base_type = dataset_type.replace('Bagpipes_', '')
            color = colors_dark[base_type]
            ax3.bar(x_pos, delta_value, bar_width, color=color, alpha=0.3, edgecolor='black', linewidth=1.0, zorder=3)
            ax3.errorbar(x_pos, delta_value, yerr=error_value, fmt='none', ecolor='black', capsize=3, capthick=1.5, elinewidth=1.2, alpha=0.6, zorder=4)

ax3.set_xticks([x_positions[z] for z in sorted_redshift_values])
ax3.set_xticklabels([f'{z}' for z in sorted_redshift_values], rotation=0)

legend_elements_col1 = [
    plt.Rectangle((0,0),1,1, facecolor=colors_bar['NOT_Corrected'], edgecolor='black', alpha=0.9, label='NOT Corrected'),
    plt.Rectangle((0,0),1,1, facecolor=colors_bar['AGN_Stellar'], edgecolor='black', alpha=0.9, label='AGN Stellar'),
    plt.Rectangle((0,0),1,1, facecolor=colors_bar['AGN_Only'], edgecolor='black', alpha=0.9, label='AGN Only'),
    plt.Rectangle((0,0),1,1, facecolor=colors_bar['Stellar_Only'], edgecolor='black', alpha=0.9, label='Stellar Only')
]

legend_elements_col2 = [
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['NOT_Corrected'], edgecolor='black', alpha=0.3, label='NOT Corrected (Corrected M*)'),
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['AGN_Stellar'], edgecolor='black', alpha=0.3, label='AGN Stellar (Corrected M*)'),
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['AGN_Only'], edgecolor='black', alpha=0.3, label='AGN Only (Corrected M*)'),
    plt.Rectangle((0,0),1,1, facecolor=colors_dark['Stellar_Only'], edgecolor='black', alpha=0.3, label='Stellar Only (Corrected M*)')
]

legend_elements = legend_elements_col1 + legend_elements_col2
ax3.legend(handles=legend_elements, loc='upper left', fontsize=9, ncol=2)

ax3.set_xlabel(r"Redshift", fontsize=14, color="black")
ax3.set_ylabel(r"$\Delta$ log$_{10}$(M$_{BH}$/M$_{*}$)", fontsize=14, color="black")

ax3.grid(visible=True, which='both', axis='both', linestyle='-', alpha=0.3, zorder=0)
ax3.minorticks_on()

for spine in ax3.spines.values():
    spine.set_linewidth(2.5)

ax3.tick_params(axis='both', which='major', labelsize=12, length=8, width=2.0, direction='in')
ax3.tick_params(axis='both', which='minor', labelsize=10, length=4, width=1.5, direction='in')
ax3.tick_params(top=True, right=True)

ax1.set_title("No SED Correction")
ax2.set_title("With SED Correction")

fig.savefig("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/Plots_For_Paper/Test_Luminositys_With_Bagpipes.png", dpi=300, 
             bbox_inches='tight', facecolor='white', edgecolor='none')

plt.show()
